# 🗄️ SQL Business Analytics

## SuperStore Sales & Profitability Analysis

This notebook performs SQL-based business analytics using the
processed SuperStore dataset stored in a SQLite relational database.

The analysis moves from transaction-level data toward
business-level insights using SQL.

---

## 🎯 Objectives

The primary objectives of this notebook are:

- Connect Python with the SQLite database.
- Validate the database structure and data.
- Understand the relational table.
- Perform SQL filtering and aggregation.
- Analyze Sales and Profit performance.
- Analyze Customers and Products.
- Analyze Categories and Sub-Categories.
- Analyze Regions, States, and Cities.
- Analyze Shipping and Payment operations.
- Analyze Returns.
- Perform Time-Based Sales Analysis.
- Perform advanced SQL business analysis.
- Use Window Functions for rankings and comparisons.
- Use CTEs for complex business queries.
- Create reusable SQL business metrics.

---

## 🗃️ Data Source

The processed dataset is stored in:

```text
superstore_analysis.db


---

# 1️⃣ Database Connection

## 1. Database Connection

The first step is to establish a connection with the SQLite
database containing the processed SuperStore dataset.

The database will be queried directly using SQL.

### Import Librarie's

In [2]:
import sqlite3
import pandas as pd
import numpy as np

from pathlib import Path

## 2. Database Validation

Before performing business analysis, the database structure and
stored records are validated.

This ensures that the SQL analysis is working on the expected
dataset.

In [3]:
db_path = Path("superstore_analysis.db")

conn = sqlite3.connect(db_path)

print("Database connected successfully.")
print(f"Database: {db_path}")

Database connected successfully.
Database: superstore_analysis.db


In [4]:
tables = pd.read_sql_query(
    """
    SELECT name
    FROM sqlite_master
    WHERE type = 'table'
    ORDER BY name;
    """,
    conn
)

display(tables)

,name
0,superstore


In [5]:
row_count = pd.read_sql_query(
    """
    SELECT COUNT(*) AS total_rows
    FROM superstore;
    """,
    conn
)

display(row_count)

,total_rows
0,5901


In [6]:
schema = pd.read_sql_query(
    """
    PRAGMA table_info(superstore);
    """,
    conn
)

display(schema)

,cid,name,type,notnull,dflt_value,pk
0,0,Row_ID,INTEGER,0,None,0
1,1,Order_ID,TEXT,0,None,0
2,2,Order_Date,TEXT,0,None,0
3,3,Ship_Date,TEXT,0,None,0
4,4,Ship_Mode,TEXT,0,None,0
5,5,Customer_ID,TEXT,0,None,0
6,6,Customer_Name,TEXT,0,None,0
7,7,Segment,TEXT,0,None,0
8,8,Country,TEXT,0,None,0
9,9,City,TEXT,0,None,0


## 3. SQL Data Preview

A small sample of the transaction table is retrieved to verify
that the database contains the expected business fields.

In [7]:
preview = pd.read_sql_query(
    """
    SELECT *
    FROM superstore
    LIMIT 10;
    """,
    conn
)

display(preview)

,Row_ID,Order_ID,Order_Date,Ship_Date,Ship_Mode,Customer_ID,Customer_Name,Segment,Country,City,...,City_Total_Profit,Segment_Total_Sales,Segment_Total_Profit,Category_Total_Sales,Category_Total_Profit,SubCategory_Total_Sales,SubCategory_Total_Profit,ShipMode_Order_Count,PaymentMode_Order_Count,Return_Flag
0,4918,CA-2019-160304,2019-01-01,2019-07-01,Standard Class,BM-11575,Brendan Murry,Corporate,United States,Gaithersburg,...,133.8078,509743.1262,57805.7991,451508.6452,10006.6112,57577.6862,-342.8883,1773,1562,0
1,4919,CA-2019-160304,2019-02-01,2019-07-01,Standard Class,BM-11575,Brendan Murry,Corporate,United States,Gaithersburg,...,133.8078,509743.1262,57805.7991,451508.6452,10006.6112,57577.6862,-342.8883,1773,1562,0
2,4920,CA-2019-160304,2019-02-01,2019-07-01,Standard Class,BM-11575,Brendan Murry,Corporate,United States,Gaithersburg,...,133.8078,509743.1262,57805.7991,470587.9910,90458.2486,196563.5460,22308.9179,1773,985,0
3,3074,CA-2019-125206,2019-03-01,2019-05-01,First Class,LR-16915,Lena Radford,Consumer,United States,Los Angeles,...,19174.9301,753002.1291,81338.5875,643707.6870,74797.2461,150341.3180,13607.0875,499,1562,0
4,8604,US-2019-116365,2019-03-01,2019-08-01,Standard Class,CA-12310,Christine Abelman,Corporate,United States,San Antonio,...,-23.1982,509743.1262,57805.7991,470587.9910,90458.2486,122301.0860,25336.6455,1773,1562,0
5,8605,US-2019-116365,2019-03-01,2019-08-01,Standard Class,CA-12310,Christine Abelman,Corporate,United States,San Antonio,...,-23.1982,509743.1262,57805.7991,470587.9910,90458.2486,122301.0860,25336.6455,1773,1562,0
6,8606,US-2019-116365,2019-03-01,2019-08-01,Standard Class,CA-12310,Christine Abelman,Corporate,United States,San Antonio,...,-23.1982,509743.1262,57805.7991,470587.9910,90458.2486,196563.5460,22308.9179,1773,985,0
7,9494,CA-2019-105207,2019-03-01,2019-08-01,Standard Class,BO-11350,Bill Overfelt,Corporate,United States,Broken Arrow,...,361.8182,509743.1262,57805.7991,451508.6452,10006.6112,119293.7430,-11091.6365,1773,1717,0
8,9495,CA-2019-105207,2019-03-01,2019-08-01,Standard Class,BO-11350,Bill Overfelt,Corporate,United States,Broken Arrow,...,361.8182,509743.1262,57805.7991,643707.6870,74797.2461,174978.3900,17885.3759,1773,1717,0
9,2898,US-2019-164630,2019-04-01,2019-09-01,Standard Class,EB-13975,Erica Bern,Corporate,United States,Charlotte,...,322.6946,509743.1262,57805.7991,470587.9910,90458.2486,59735.7980,42774.5828,1773,1562,0


## 4. Database-Level Business Validation

The following SQL query validates the major business metrics
stored in the database.

The results will later be compared with the original Pandas
DataFrame if required.

In [9]:
validation = pd.read_sql_query(
    """
    SELECT
        COUNT(*) AS Total_Transactions,
        COUNT(DISTINCT Order_ID) AS Total_Orders,
        COUNT(DISTINCT Customer_ID) AS Total_Customers,
        COUNT(DISTINCT Product_ID) AS Total_Products,
        SUM(Sales) AS Total_Sales,
        SUM(Profit) AS Total_Profit,
        SUM(Quantity) AS Total_Quantity,
        AVG(Sales) AS Average_Sales,
        AVG(Profit) AS Average_Profit
    FROM superstore;
    """,
    conn
)

display(validation)

,Total_Transactions,Total_Orders,Total_Customers,Total_Products,Total_Sales,Total_Profit,Total_Quantity,Average_Sales,Average_Profit
0,5901,3003,773,1755,1.565804e+06,175262.1059,22317,265.345589,29.700408


## 5. SQL NULL Validation

The database is checked for NULL values in important analytical
columns before business analysis begins.

In [10]:
null_validation = pd.read_sql_query(
    """
    SELECT
        SUM(CASE WHEN Sales IS NULL THEN 1 ELSE 0 END) AS Sales_Null,
        SUM(CASE WHEN Profit IS NULL THEN 1 ELSE 0 END) AS Profit_Null,
        SUM(CASE WHEN Quantity IS NULL THEN 1 ELSE 0 END) AS Quantity_Null,
        SUM(CASE WHEN Order_Date IS NULL THEN 1 ELSE 0 END) AS Order_Date_Null,
        SUM(CASE WHEN Customer_ID IS NULL THEN 1 ELSE 0 END) AS Customer_ID_Null,
        SUM(CASE WHEN Product_ID IS NULL THEN 1 ELSE 0 END) AS Product_ID_Null
    FROM superstore;
    """,
    conn
)

display(null_validation)

,Sales_Null,Profit_Null,Quantity_Null,Order_Date_Null,Customer_ID_Null,Product_ID_Null
0,0,0,0,3491,0,0


## 6. Basic SQL Query Test

The basic SQL operations are validated before moving into
business-oriented analytical queries.

Operations covered:

- SELECT
- WHERE
- DISTINCT
- ORDER BY
- LIMIT

In [11]:
basic_query = pd.read_sql_query(
    """
    SELECT
        Order_ID,
        Order_Date,
        Customer_Name,
        Category,
        Sales,
        Profit
    FROM superstore
    ORDER BY Sales DESC
    LIMIT 10;
    """,
    conn
)

display(basic_query)

,Order_ID,Order_Date,Customer_Name,Category,Sales,Profit
0,US-2019-107440,NaN,Bill Shonely,Technology,9099.930,2365.9818
1,CA-2020-166709,NaN,Hunter Lopez,Technology,5517.970,5039.9856
2,CA-2020-138289,NaN,Andy Reiter,Office Supplies,5455.960,2504.2216
3,US-2019-140158,2019-04-10,Daniel Raglin,Technology,5399.910,2591.9568
4,CA-2020-143112,2020-05-10,Todd Sumrall,Technology,5234.960,1351.9896
5,CA-2020-135909,NaN,Jane Waco,Office Supplies,5107.960,1906.4850
6,CA-2019-136301,NaN,Edward Hooks,Office Supplies,4912.590,196.5036
7,US-2019-143819,2019-01-03,Karen Daniels,Technology,4899.930,2400.9657
8,CA-2020-149881,2020-01-04,Nick Crebassa,Technology,4823.984,359.9988
9,CA-2019-158841,2019-02-02,Sanjit Engle,Technology,4749.950,2799.9840


# 🔵 Stage 2 — Basic SQL Business Analysis

This stage uses fundamental SQL operations to explore transaction-level
business data and identify important Sales, Profit, Customer, Product,
Geography, and Operational patterns.

## SQL Concepts

- SELECT
- DISTINCT
- WHERE
- ORDER BY
- LIMIT

## Business Objectives

- Identify highest-value transactions.
- Identify lowest-profit transactions.
- Find high-quantity orders.
- Explore major business categories.
- Explore customer and product dimensions.
- Identify high-value customers and products.
- Understand geographic coverage.
- Identify important operational patterns.

## Business Dimensions

```text
Category
Sub-Category
Product
Customer
Segment
Region
State
City
Ship Mode
Payment Mode


---

## 1. Highest Sales Transactions


Identify the individual transactions with the highest Sales value.

Business Question:

> Which transactions generate the highest revenue?

In [14]:
top_sales_transactions = pd.read_sql_query(
    """
    SELECT
        Order_ID,
        Order_Date,
        Customer_Name,
        Category,
        Product_Name,
        Sales,
        Quantity,
        Profit
    FROM superstore
    ORDER BY Sales DESC
    LIMIT 10;
    """,
    conn
)

display(top_sales_transactions)

,Order_ID,Order_Date,Customer_Name,Category,Product_Name,Sales,Quantity,Profit
0,US-2019-107440,NaN,Bill Shonely,Technology,"3D Systems Cube Printer, 2nd Generation, Magenta",9099.930,7,2365.9818
1,CA-2020-166709,NaN,Hunter Lopez,Technology,Canon imageCLASS 2200 Advanced Copier,5517.970,3,5039.9856
2,CA-2020-138289,NaN,Andy Reiter,Office Supplies,GBC DocuBind P400 Electric Binding System,5455.960,4,2504.2216
3,US-2019-140158,2019-04-10,Daniel Raglin,Technology,Hewlett Packard LaserJet 3310 Copier,5399.910,9,2591.9568
4,CA-2020-143112,2020-05-10,Todd Sumrall,Technology,"3D Systems Cube Printer, 2nd Generation, Magenta",5234.960,4,1351.9896
5,CA-2020-135909,NaN,Jane Waco,Office Supplies,Fellowes PB500 Electric Punch Plastic Comb Bin...,5107.960,5,1906.4850
6,CA-2019-136301,NaN,Edward Hooks,Office Supplies,High Speed Automatic Electric Letter Opener,4912.590,3,196.5036
7,US-2019-143819,2019-01-03,Karen Daniels,Technology,Ativa V4110MDD Micro-Cut Shredder,4899.930,7,2400.9657
8,CA-2020-149881,2020-01-04,Nick Crebassa,Technology,Cubify CubeX 3D Printer Double Head Print,4823.984,2,359.9988
9,CA-2019-158841,2019-02-02,Sanjit Engle,Technology,HP Designjet T520 Inkjet Large Format Printer ...,4749.950,5,2799.9840


## 2. Lowest Sales Transactions

Identify transactions with the lowest Sales values.

Business Question:

> Which transactions contribute the least revenue?

In [15]:
lowest_sales_transactions = pd.read_sql_query(
    """
    SELECT
        Order_ID,
        Customer_Name,
        Product_Name,
        Category,
        Sales,
        Quantity,
        Profit
    FROM superstore
    ORDER BY Sales ASC
    LIMIT 10;
    """,
    conn
)

display(lowest_sales_transactions)

,Order_ID,Customer_Name,Product_Name,Category,Sales,Quantity,Profit
0,CA-2019-168361,Ken Brennan,Avery Durable Slant Ring Binders With Label Ho...,Office Supplies,0.836,1,-1.3376
1,US-2019-110156,Eric Hoffmann,Avery Hidden Tab Dividers for Binding Systems,Office Supplies,1.192,2,-2.0264
2,CA-2019-159373,Liz Thompson,Insertable Tab Indexes For Data Binders,Office Supplies,1.272,2,-2.1624
3,CA-2019-163153,Dario Medina,Prang Dustless Chalk Sticks,Office Supplies,1.344,1,0.5040
4,CA-2019-169922,Mary Zewe,Computer Printout Index Tabs,Office Supplies,1.344,4,-2.1504
5,CA-2019-114748,Maria Zettner,Newell 310,Office Supplies,1.408,1,0.1584
6,CA-2019-132409,Tiffany House,"OIC #2 Pencils, Medium Soft",Office Supplies,1.504,1,0.1692
7,CA-2019-120824,Arthur Wiediger,"Acco Pressboard Covers with Storage Hooks, 14 ...",Office Supplies,1.524,2,-2.6670
8,CA-2019-111682,Ted Butterfield,Prang Dustless Chalk Sticks,Office Supplies,1.680,1,0.8400
9,CA-2019-168032,David Flashing,Avery Durable Binders,Office Supplies,1.728,3,-2.6784


## 3. Highest Profit Transactions

Identify transactions generating the highest individual Profit.

Business Question:

> Which transactions are the most profitable?

In [16]:
top_profit_transactions = pd.read_sql_query(
    """
    SELECT
        Order_ID,
        Customer_Name,
        Product_Name,
        Category,
        Sales,
        Quantity,
        Profit,
        Profit_Margin
    FROM superstore
    ORDER BY Profit DESC
    LIMIT 10;
    """,
    conn
)

display(top_profit_transactions)

,Order_ID,Customer_Name,Product_Name,Category,Sales,Quantity,Profit,Profit_Margin
0,CA-2019-118689,Tamara Chand,Canon imageCLASS 2200 Advanced Copier,Technology,499.950,5,8399.9760,1680.163216
1,CA-2020-140151,Raymond Buch,Canon imageCLASS 2200 Advanced Copier,Technology,3445.960,4,6719.9808,195.010412
2,CA-2020-166709,Hunter Lopez,Canon imageCLASS 2200 Advanced Copier,Technology,5517.970,3,5039.9856,91.337677
3,CA-2019-117121,Adrian Barton,GBC Ibimaster 500 Manual ProClick Binding System,Office Supplies,2892.740,13,4946.3700,170.992554
4,CA-2020-127180,Tom Ashbrook,Canon imageCLASS 2200 Advanced Copier,Technology,2212.968,4,3919.9888,177.137166
5,CA-2019-158841,Sanjit Engle,HP Designjet T520 Inkjet Large Format Printer ...,Technology,4749.950,5,2799.9840,58.947652
6,US-2019-140158,Daniel Raglin,Hewlett Packard LaserJet 3310 Copier,Technology,5399.910,9,2591.9568,48.000000
7,CA-2020-138289,Andy Reiter,GBC DocuBind P400 Electric Binding System,Office Supplies,5455.960,4,2504.2216,45.898826
8,US-2019-143819,Karen Daniels,Ativa V4110MDD Micro-Cut Shredder,Technology,4899.930,7,2400.9657,49.000000
9,US-2019-107440,Bill Shonely,"3D Systems Cube Printer, 2nd Generation, Magenta",Technology,9099.930,7,2365.9818,26.000000


## 4. Loss-Making Transactions

Identify transactions where Profit is negative.

Business Question:

> Which individual transactions generate financial losses?

In [17]:
loss_transactions = pd.read_sql_query(
    """
    SELECT
        Order_ID,
        Order_Date,
        Customer_Name,
        Product_Name,
        Category,
        Sales,
        Quantity,
        Profit,
        Profit_Margin
    FROM superstore
    WHERE Profit < 0
    ORDER BY Profit ASC
    LIMIT 20;
    """,
    conn
)

display(loss_transactions)

,Order_ID,Order_Date,Customer_Name,Product_Name,Category,Sales,Quantity,Profit,Profit_Margin
0,CA-2019-108196,NaN,Cindy Stewart,Cubify CubeX 3D Printer Double Head Print,Technology,4499.985,5,-6599.9780,-146.666667
1,US-2020-168116,2020-04-11,Grant Thornton,Cubify CubeX 3D Printer Triple Head Print,Technology,3009.980,4,-3839.9904,-127.575280
2,CA-2020-134845,NaN,Sharelle Roach,Lexmark MX611dhe Monochrome Laser Printer,Technology,2579.985,5,-3399.9800,-131.782937
3,US-2020-122714,2020-07-12,Henry Goldwyn,Ibico EPK-21 Electric Binding System,Office Supplies,1901.990,5,-2929.4845,-154.022077
4,CA-2020-131254,NaN,Nathan Cano,Fellowes PB500 Electric Punch Plastic Comb Bin...,Office Supplies,1556.188,6,-2287.7820,-147.011929
5,CA-2019-130946,2019-08-04,Zuschuss Carroll,GBC DocuBind P400 Electric Binding System,Office Supplies,1088.792,4,-1850.9464,-170.000000
6,US-2020-120390,NaN,Tracy Hopkins,GBC DocuBind P400 Electric Binding System,Office Supplies,1677.188,4,-1306.5504,-77.901249
7,CA-2020-128363,NaN,Dan Campbell,GBC DocuBind TL300 Electric Binding System,Office Supplies,1641.582,6,-1237.8462,-75.405688
8,CA-2020-152093,2020-10-09,Skye Norling,Fellowes PB500 Electric Punch Plastic Comb Bin...,Office Supplies,804.594,3,-1143.8910,-142.169964
9,US-2020-148551,2020-12-01,David Bremer,GBC Ibimaster 500 Manual ProClick Binding System,Office Supplies,790.980,5,-1141.4700,-144.310855


## 5. High-Quantity Transactions

Identify transactions containing unusually high quantities.

Business Question:

> Which orders contain the largest quantities of products?

In [18]:
high_quantity_orders = pd.read_sql_query(
    """
    SELECT
        Order_ID,
        Customer_Name,
        Product_Name,
        Category,
        Quantity,
        Sales,
        Profit
    FROM superstore
    ORDER BY Quantity DESC
    LIMIT 20;
    """,
    conn
)

display(high_quantity_orders)

,Order_ID,Customer_Name,Product_Name,Category,Quantity,Sales,Profit
0,CA-2019-141887,Mark Packer,Ultra Door Push Plate,Furniture,14,54.992,8.9362
1,CA-2019-140571,Sanjit Jacobs,Xerox 1964,Office Supplies,14,319.760,147.0896
2,CA-2019-142405,Sanjit Engle,Wilson Jones Turn Tabs Binder Tool for Ring Bi...,Office Supplies,14,53.984,17.5448
3,CA-2019-105732,Alejandro Grove,"Electrix Architect's Clamp-On Swing Arm Lamp, ...",Furniture,14,1336.440,387.5676
4,CA-2019-145583,Lena Creighton,Longer-Life Soft White Bulbs,Furniture,14,43.120,20.6976
5,CA-2019-116596,Ben Wallace,"Metal Folding Chairs, Beige, 4/Carton",Furniture,14,427.644,80.7772
6,US-2019-103674,Anne Pryor,Ibico Recycled Linen-Style Covers,Office Supplies,14,437.472,153.1152
7,CA-2020-151750,Janet Martin,"Pressboard Covers with Storage Hooks, 9 1/2"" x...",Office Supplies,14,137.748,-22.6842
8,CA-2020-161410,Chris Cortes,Plantronics Voyager Pro HD - Bluetooth Headset,Technology,14,577.916,72.7888
9,CA-2020-130036,Ben Peterman,Logitech Wireless Headset h800,Technology,14,1131.888,209.9790


## 6. Highest Profit Margin Transactions

Identify transactions with the highest Profit Margin.

Business Question:

> Which individual transactions generate the strongest profitability
> relative to their Sales value?

In [19]:
top_margin_transactions = pd.read_sql_query(
    """
    SELECT
        Order_ID,
        Customer_Name,
        Product_Name,
        Category,
        Sales,
        Profit,
        Profit_Margin
    FROM superstore
    WHERE Sales > 0
    ORDER BY Profit_Margin DESC
    LIMIT 20;
    """,
    conn
)

display(top_margin_transactions)

,Order_ID,Customer_Name,Product_Name,Category,Sales,Profit,Profit_Margin
0,CA-2019-118689,Tamara Chand,Canon imageCLASS 2200 Advanced Copier,Technology,499.950,8399.9760,1680.163216
1,CA-2020-152912,Brian Moss,Adjustable Depth Letter/Legal Cart,Office Supplies,51.140,473.6106,926.105984
2,CA-2019-157791,Carol Adams,Hewlett Packard 610 Color Digital Copier / Pri...,Technology,99.960,899.9820,900.342137
3,CA-2019-163804,Deborah Brumfield,Deluxe Rollaway Locking File with Drawer,Office Supplies,79.400,582.2320,733.289673
4,CA-2020-152912,Brian Moss,Maxell iVDR EX 500GB Cartridge,Technology,55.620,355.4466,639.062567
5,CA-2020-151799,Ben Ferrer,Canon PC1080F Personal Copier,Technology,162.980,467.9922,287.147012
6,CA-2020-120376,Theone Pippenger,Hon GuestStacker Chair,Furniture,155.690,412.5394,264.974886
7,CA-2020-148922,Stephanie Ulpright,Razer Tiamat Over Ear 7.1 Surround Sound PC Ga...,Technology,106.970,257.9871,241.177059
8,CA-2020-140151,Raymond Buch,Canon imageCLASS 2200 Advanced Copier,Technology,3445.960,6719.9808,195.010412
9,CA-2020-145219,Robert Marley,Hewlett Packard LaserJet 3310 Copier,Technology,536.952,1007.9832,187.723148


## 7. Lowest Profit Margin Transactions

Identify transactions with the weakest profitability relative to
Sales.

Business Question:

> Which transactions have the poorest profit efficiency?

In [20]:
lowest_margin_transactions = pd.read_sql_query(
    """
    SELECT
        Order_ID,
        Customer_Name,
        Product_Name,
        Category,
        Sales,
        Profit,
        Profit_Margin
    FROM superstore
    WHERE Sales > 0
    ORDER BY Profit_Margin ASC
    LIMIT 20;
    """,
    conn
)

display(lowest_margin_transactions)

,Order_ID,Customer_Name,Product_Name,Category,Sales,Profit,Profit_Margin
0,US-2020-110576,Ross Baird,"Riverside Furniture Oval Coffee Table, Oval En...",Furniture,115.320,-619.5960,-537.284079
1,CA-2020-155075,Giulietta Weimer,GBC DocuBind 300 Electric Binding Machine,Office Supplies,155.176,-462.8624,-298.282209
2,CA-2019-110898,Lena Cacioppo,Commercial WindTunnel Clean Air Upright Vacuum...,Office Supplies,2.334,-6.3018,-270.000000
3,CA-2019-163398,Christy Brittain,Acco Smartsocket Color-Coded Six-Outlet AC Ada...,Office Supplies,26.406,-71.2962,-270.000000
4,US-2019-144057,Cynthia Voltz,Euro Pro Shark Stick Mini Vacuum,Office Supplies,48.784,-131.7168,-270.000000
5,CA-2019-143406,Lisa Ryan,Hoover Commercial Lightweight Upright Vacuum w...,Office Supplies,93.032,-251.1864,-270.000000
6,US-2019-131114,Rob Williams,Belkin 6 Outlet Metallic Surge Strip,Office Supplies,4.356,-11.7612,-270.000000
7,CA-2019-144540,Gary Hansen,Eureka The Boss Plus 12-Amp Hard Box Upright V...,Office Supplies,62.790,-166.3935,-265.000000
8,CA-2019-130393,John Murray,Acco 6 Outlet Guardian Premium Surge Suppressor,Office Supplies,11.648,-30.8672,-265.000000
9,US-2019-103646,Sibella Parks,Kensington 4 Outlet MasterPiece Compact Power ...,Office Supplies,48.792,-126.8592,-260.000000


## 8. Highest Shipping Duration

Identify transactions with the longest shipping duration.

Business Question:

> Which orders experienced the longest shipping times?

In [21]:
long_shipping_orders = pd.read_sql_query(
    """
    SELECT
        Order_ID,
        Order_Date,
        Ship_Date,
        Customer_Name,
        Ship_Mode,
        Shipping_Days,
        Sales,
        Profit
    FROM superstore
    WHERE Shipping_Days IS NOT NULL
    ORDER BY Shipping_Days DESC
    LIMIT 20;
    """,
    conn
)

display(long_shipping_orders)

,Order_ID,Order_Date,Ship_Date,Customer_Name,Ship_Mode,Shipping_Days,Sales,Profit
0,CA-2019-118514,2019-03-02,2019-10-02,Liz Carlisle,Standard Class,214.0,866.400,225.2640
1,CA-2019-163986,2019-03-09,2019-10-09,Jennifer Jackson,Standard Class,214.0,54.500,14.1700
2,CA-2019-128867,2019-03-11,2019-10-11,Clay Ludtke,Standard Class,214.0,75.960,22.7880
3,CA-2019-128867,2019-03-11,2019-10-11,Clay Ludtke,Standard Class,214.0,27.240,13.3476
4,CA-2020-111815,2020-03-03,2020-10-03,Emily Phan,Standard Class,214.0,125.980,42.9914
5,CA-2020-111815,2020-03-03,2020-10-03,Emily Phan,Standard Class,214.0,227.980,47.0548
6,CA-2020-140872,2020-03-06,2020-10-06,Nick Radford,Standard Class,214.0,153.400,-4.1400
7,CA-2020-140872,2020-03-06,2020-10-06,Nick Radford,Standard Class,214.0,222.992,-2.5749
8,CA-2020-140872,2020-03-06,2020-10-06,Nick Radford,Standard Class,214.0,276.568,0.0000
9,CA-2020-140872,2020-03-06,2020-10-06,Nick Radford,Standard Class,214.0,528.960,50.3960


## 9. Business Category Exploration

Identify all unique product categories available in the dataset.

Business Question:

> What major product categories does the business operate in?

In [22]:
categories = pd.read_sql_query(
    """
    SELECT DISTINCT
        Category
    FROM superstore
    WHERE Category IS NOT NULL
    ORDER BY Category;
    """,
    conn
)

display(categories)

,Category
0,Furniture
1,Office Supplies
2,Technology


## 10. Business Geography Exploration

Identify the major regions and geographic coverage of the business.

Business Question:

> Across which regions does the business operate?

In [23]:
geography = pd.read_sql_query(
    """
    SELECT DISTINCT
        Region
    FROM superstore
    WHERE Region IS NOT NULL
    ORDER BY Region;
    """,
    conn
)

display(geography)

,Region
0,Central
1,East
2,South
3,West


# 🔵 Stage 3 — GROUP BY & Business Aggregation

This stage transforms transaction-level records into business-level
performance summaries using SQL aggregation functions.

## SQL Concepts

- GROUP BY
- SUM()
- COUNT()
- COUNT(DISTINCT)
- AVG()
- MIN()
- MAX()
- HAVING
- ORDER BY

## Business Metrics

- Total Sales
- Total Profit
- Total Quantity
- Order Count
- Customer Count
- Product Count
- Average Sales
- Average Profit
- Minimum Sales
- Maximum Sales

## Business Dimensions

```text
Category
Sub-Category
Product
Customer
Segment
Region
State
City
Ship Mode
Payment Mode
Year
Quarter
Month


---

# 1️⃣ Category Sales Performance

Business Question:

> Which product category generates the highest total Sales?

The analysis calculates total Sales, Profit, Quantity, Orders,
Customers, and Sales contribution for every category.

In [24]:
category_performance = pd.read_sql_query(
    """
    SELECT
        Category,
        SUM(Sales) AS Total_Sales,
        SUM(Profit) AS Total_Profit,
        SUM(Quantity) AS Total_Quantity,
        COUNT(DISTINCT Order_ID) AS Total_Orders,
        COUNT(DISTINCT Customer_ID) AS Total_Customers
    FROM superstore
    GROUP BY Category
    ORDER BY Total_Sales DESC;
    """,
    conn
)

category_performance["Sales_Contribution_%"] = (
    category_performance["Total_Sales"]
    / category_performance["Total_Sales"].sum()
    * 100
)

display(category_performance.round(2))
display(category_performance.describe().T)

,Category,Total_Sales,Total_Profit,Total_Quantity,Total_Orders,Total_Customers,Sales_Contribution_%
0,Office Supplies,643707.69,74797.25,13625,2219,739,41.11
1,Technology,470587.99,90458.25,4061,915,538,30.05
2,Furniture,451508.65,10006.61,4631,1040,583,28.84


,count,mean,std,min,25%,50%,75%,max
Total_Sales,3.0,521934.774400,105889.031733,451508.645200,461048.318100,470587.991000,557147.839000,643707.687000
Total_Profit,3.0,58420.701967,42652.782892,10006.611200,42401.928650,74797.246100,82627.747350,90458.248600
Total_Quantity,3.0,7439.000000,5364.808664,4061.000000,4346.000000,4631.000000,9128.000000,13625.000000
Total_Orders,3.0,1391.333333,719.500058,915.000000,977.500000,1040.000000,1629.500000,2219.000000
Total_Customers,3.0,620.000000,105.484596,538.000000,560.500000,583.000000,661.000000,739.000000
Sales_Contribution_%,3.0,33.333333,6.762597,28.835573,29.444823,30.054074,35.582214,41.110353


## 2. Sub-Category Performance

Business Question:

> Which sub-categories are the strongest and weakest contributors
> to Sales and Profit?

This provides a more detailed view than Category-level analysis.

In [37]:
schema.columns,
schema.name

0                       Row_ID
1                     Order_ID
2                   Order_Date
3                    Ship_Date
4                    Ship_Mode
5                  Customer_ID
6                Customer_Name
7                      Segment
8                      Country
9                         City
10                       State
11                      Region
12                  Product_ID
13                    Category
14                Sub-Category
15                Product_Name
16                       Sales
17                    Quantity
18                      Profit
19                     Returns
20                Payment_Mode
21                  Order_Year
22               Order_Quarter
23                 Order_Month
24            Order_Month_Name
25                   Order_Day
26              Order_Day_Name
27               Shipping_Days
28              Sales_Per_Unit
29             Profit_Per_Unit
30               Profit_Margin
31        Customer_Order_Count
32      

In [40]:
subcategory_performance = pd.read_sql_query(
    """
    SELECT
        Category,
        "Sub-Category" AS Sub_Category,
        SUM(Sales) AS Total_Sales,
        SUM(Profit) AS Total_Profit,
        SUM(Quantity) AS Total_Quantity,
        COUNT(DISTINCT Order_ID) AS Total_Orders
    FROM superstore
    GROUP BY
        Category,
        "Sub-Category"
    ORDER BY Total_Sales DESC;
    """,
    conn
)

display(subcategory_performance)
display(subcategory_performance.describe().T)

,Category,Sub_Category,Total_Sales,Total_Profit,Total_Quantity,Total_Orders
0,Technology,Phones,196563.5460,22308.9179,1908,487
1,Furniture,Chairs,181945.9980,13406.7032,1288,331
2,Office Supplies,Binders,174978.3900,17885.3759,3670,785
3,Office Supplies,Storage,150341.3180,13607.0875,1830,459
4,Technology,Accessories,122301.0860,25336.6455,1761,422
5,Furniture,Tables,119293.7430,-11091.6365,736,186
6,Office Supplies,Paper,99453.6120,21112.3779,3074,719
7,Furniture,Furnishings,92691.2180,8034.4328,2133,527
8,Technology,Machines,91987.5610,38.1024,250,64
9,Office Supplies,Appliances,80305.2470,13166.6098,1050,271


,count,mean,std,min,25%,50%,75%,max
Total_Sales,17.0,92106.136659,58637.229584,15205.2380,50762.9760,91987.5610,122301.0860,196563.5460
Total_Profit,17.0,10309.535641,12969.304754,-11091.6365,598.4175,8034.4328,17885.3759,42774.5828
Total_Quantity,17.0,1312.764706,1011.405429,142.0000,474.0000,1050.0000,1830.0000,3670.0000
Total_Orders,17.0,318.529412,225.954674,38.0000,130.0000,271.0000,459.0000,785.0000


## 3. Regional Sales Performance

Business Question:

> Which region generates the highest Sales and Profit?

The query compares revenue, profitability, quantity, orders, and
customer reach across regions.

In [41]:
region_performance = pd.read_sql_query(
    """
    SELECT
        Region,
        SUM(Sales) AS Total_Sales,
        SUM(Profit) AS Total_Profit,
        SUM(Quantity) AS Total_Quantity,
        COUNT(DISTINCT Order_ID) AS Total_Orders,
        COUNT(DISTINCT Customer_ID) AS Total_Customers
    FROM superstore
    GROUP BY Region
    ORDER BY Total_Sales DESC;
    """,
    conn
)

region_performance["Sales_Contribution_%"] = (
    region_performance["Total_Sales"]
    / region_performance["Total_Sales"].sum()
    * 100
)

display(region_performance.round(2))
display(region_performance.describe().T)

,Region,Total_Sales,Total_Profit,Total_Quantity,Total_Orders,Total_Customers,Sales_Contribution_%
0,West,522441.05,67859.96,7298,961,551,33.37
1,East,450234.67,53400.42,6251,844,523,28.75
2,Central,341007.52,27450.01,5239,711,471,21.78
3,South,252121.08,26551.72,3529,487,357,16.10


,count,mean,std,min,25%,50%,75%,max
Total_Sales,4.0,391451.080800,119123.582455,252121.081000,318785.913400,395621.095100,468286.262500,522441.052000
Total_Profit,4.0,43815.526475,20296.751213,26551.716300,27225.434400,40425.215700,57015.307775,67859.958200
Total_Quantity,4.0,5579.250000,1604.643756,3529.000000,4811.500000,5745.000000,6512.750000,7298.000000
Total_Orders,4.0,750.750000,203.342691,487.000000,655.000000,777.500000,873.250000,961.000000
Total_Customers,4.0,475.500000,85.671855,357.000000,442.500000,497.000000,530.000000,551.000000
Sales_Contribution_%,4.0,25.000000,7.607821,16.101698,20.359243,25.266318,29.907074,33.365667


## 4. Customer Segment Performance

Business Question:

> Which customer segment contributes the most Sales and Profit?

The analysis compares Consumer, Corporate, and other available
segments.

In [42]:
segment_performance = pd.read_sql_query(
    """
    SELECT
        Segment,
        SUM(Sales) AS Total_Sales,
        SUM(Profit) AS Total_Profit,
        SUM(Quantity) AS Total_Quantity,
        COUNT(DISTINCT Order_ID) AS Total_Orders,
        COUNT(DISTINCT Customer_ID) AS Total_Customers
    FROM superstore
    GROUP BY Segment
    ORDER BY Total_Sales DESC;
    """,
    conn
)

segment_performance["Sales_Contribution_%"] = (
    segment_performance["Total_Sales"]
    / segment_performance["Total_Sales"].sum()
    * 100
)

display(segment_performance.round(2))
display(segment_performance.describe().T)

,Segment,Total_Sales,Total_Profit,Total_Quantity,Total_Orders,Total_Customers,Sales_Contribution_%
0,Consumer,753002.13,81338.59,11199,1528,400,48.09
1,Corporate,509743.13,57805.80,6865,915,230,32.55
2,Home Office,303059.07,36117.72,4253,560,143,19.35


,count,mean,std,min,25%,50%,75%,max
Total_Sales,3.0,521934.774400,225219.152815,303059.067900,406401.097050,509743.126200,631372.627650,753002.129100
Total_Profit,3.0,58420.701967,22616.704210,36117.719300,46961.759200,57805.799100,69572.193300,81338.587500
Total_Quantity,3.0,7439.000000,3508.395075,4253.000000,5559.000000,6865.000000,9032.000000,11199.000000
Total_Orders,3.0,1001.000000,489.696845,560.000000,737.500000,915.000000,1221.500000,1528.000000
Total_Customers,3.0,257.666667,130.714702,143.000000,186.500000,230.000000,315.000000,400.000000
Sales_Contribution_%,3.0,33.333333,14.383608,19.354849,25.954782,32.554714,40.322575,48.090436


## 5. Customer Revenue Analysis

Business Question:

> Which customers contribute the highest total Sales?

Customers are ranked using transaction-level Sales aggregation.

The analysis also measures order frequency and total Profit.

In [43]:
customer_performance = pd.read_sql_query(
    """
    SELECT
        Customer_ID,
        Customer_Name,
        Segment,
        Region,
        SUM(Sales) AS Total_Sales,
        SUM(Profit) AS Total_Profit,
        SUM(Quantity) AS Total_Quantity,
        COUNT(DISTINCT Order_ID) AS Total_Orders
    FROM superstore
    GROUP BY
        Customer_ID,
        Customer_Name,
        Segment,
        Region
    ORDER BY Total_Sales DESC
    LIMIT 20;
    """,
    conn
)

display(customer_performance)
display(customer_performance.describe().T)

,Customer_ID,Customer_Name,Segment,Region,Total_Sales,Total_Profit,Total_Quantity,Total_Orders
0,SV-20365,Seth Vernon,Consumer,East,9537.296,1545.4399,77,5
1,BS-11365,Bill Shonely,Corporate,East,9135.190,2381.1596,14,1
2,JW-15220,Jane Waco,Corporate,West,7645.530,2073.2828,35,3
3,EH-13765,Edward Hooks,Corporate,West,7408.090,573.4860,49,4
4,KF-16285,Karen Ferguson,Home Office,West,6864.774,795.9138,28,3
5,TS-21370,Todd Sumrall,Corporate,East,6718.314,1574.9714,17,2
6,HW-14935,Helen Wasserman,Corporate,Central,6613.426,1361.6776,46,3
7,KD-16270,Karen Daniels,Consumer,East,6241.282,2283.0463,22,2
8,NC-18535,Nick Crebassa,Corporate,West,6043.405,591.5469,21,3
9,AR-10540,Andy Reiter,Consumer,Central,5963.700,2602.0939,14,1


,count,mean,std,min,25%,50%,75%,max
Total_Sales,20.0,6296.37230,1282.984362,5123.2820,5379.793250,5845.45600,6754.929000,9537.2960
Total_Profit,20.0,1464.16332,1241.799911,-1007.7976,590.486475,1444.99815,2125.723675,5045.8564
Total_Quantity,20.0,30.00000,16.616415,5.0000,16.750000,30.00000,37.500000,77.0000
Total_Orders,20.0,2.70000,1.080935,1.0000,2.000000,3.00000,3.000000,5.0000


## 6. Product Revenue Analysis

Business Question:

> Which products generate the highest Sales and Profit?

Products are ranked using original transaction-level Sales and
Profit values.

In [45]:
product_performance = pd.read_sql_query(
    """
    SELECT
        Product_ID,
        Product_Name,
        Category,
        "sub-Category" as Sub_Category,
        SUM(Sales) AS Total_Sales,
        SUM(Profit) AS Total_Profit,
        SUM(Quantity) AS Total_Quantity,
        COUNT(DISTINCT Order_ID) AS Total_Orders
    FROM superstore
    GROUP BY
        Product_ID,
        Product_Name,
        Category,
        Sub_Category
    ORDER BY Total_Sales DESC
    LIMIT 20;
    """,
    conn
)

display(product_performance)
display(product_performance.describe().T)

,Product_ID,Product_Name,Category,Sub_Category,Total_Sales,Total_Profit,Total_Quantity,Total_Orders
0,TEC-MA-10001047,"3D Systems Cube Printer, 2nd Generation, Magenta",Technology,Machines,14334.890,3717.9714,11,2
1,TEC-CO-10004722,Canon imageCLASS 2200 Advanced Copier,Technology,Copiers,14076.824,25199.9280,20,5
2,TEC-CO-10001449,Hewlett Packard LaserJet 3310 Copier,Technology,Copiers,13837.732,6407.8932,31,6
3,OFF-BI-10001359,GBC DocuBind TL300 Electric Binding System,Office Supplies,Binders,12890.258,2753.7593,21,6
4,OFF-BI-10004995,GBC DocuBind P400 Electric Binding System,Office Supplies,Binders,12577.108,762.1544,16,4
5,TEC-PH-10001459,Samsung Galaxy Mega 6.3,Technology,Phones,12370.708,1696.7596,34,5
6,OFF-SU-10002881,Martin Yale Chadless Opener Electric Letter Op...,Office Supplies,Supplies,12268.902,-1232.5588,16,4
7,FUR-CH-10002024,HON 5400 Series Task Chairs for Big and Tall,Furniture,Chairs,11887.562,70.0980,21,4
8,FUR-CH-10001215,Global Troy Executive Leather Low-Back Tilter,Furniture,Chairs,10217.894,776.5190,25,7
9,OFF-BI-10003527,Fellowes PB500 Electric Punch Plastic Comb Bin...,Office Supplies,Binders,9756.524,-508.3960,16,5


,count,mean,std,min,25%,50%,75%,max
Total_Sales,20.0,10228.00810,2533.919262,7287.6400,7809.548500,9540.2465,12422.308000,14334.890
Total_Profit,20.0,2052.35281,5940.732107,-6239.9792,12.922525,828.9445,1814.740375,25199.928
Total_Quantity,20.0,22.40000,13.156547,6.0000,14.000000,20.5000,31.750000,59.000
Total_Orders,20.0,5.20000,2.627787,2.0000,3.750000,5.0000,7.000000,12.000


## 7. State-Level Performance

Business Question:

> Which states are the strongest revenue markets?

State-level Sales, Profit, Orders, Customers, and Quantity are
aggregated to identify major geographic markets.

In [46]:
state_performance = pd.read_sql_query(
    """
    SELECT
        State,
        Region,
        SUM(Sales) AS Total_Sales,
        SUM(Profit) AS Total_Profit,
        SUM(Quantity) AS Total_Quantity,
        COUNT(DISTINCT Order_ID) AS Total_Orders,
        COUNT(DISTINCT Customer_ID) AS Total_Customers
    FROM superstore
    GROUP BY State, Region
    ORDER BY Total_Sales DESC;
    """,
    conn
)

display(state_performance)
display(state_performance.describe().T)

,State,Region,Total_Sales,Total_Profit,Total_Quantity,Total_Orders,Total_Customers
0,California,West,335190.2560,49372.1750,4632,619,428
1,New York,East,186748.0970,41012.0212,2512,329,267
2,Texas,Central,116261.9302,-14078.1598,2158,286,246
3,Washington,West,92975.1800,21466.6555,1257,164,149
4,Pennsylvania,East,82354.9500,-9297.7975,1227,170,155
5,Ohio,East,68194.8600,-9339.4223,1100,151,136
6,Illinois,Central,64224.3300,-9554.6539,1069,167,151
7,Florida,South,49914.9990,-227.7465,757,117,110
8,Michigan,Central,48504.6840,17480.2806,533,64,61
9,North Carolina,South,39705.1250,-4827.6923,625,84,77


,count,mean,std,min,25%,50%,75%,max
Total_Sales,49.0,31955.190269,56174.348742,419.8300,4622.2800,13602.7100,32086.9420,335190.256
Total_Profit,49.0,3576.777671,10731.160969,-14078.1598,165.2938,1301.1721,3860.6173,49372.175
Total_Quantity,49.0,455.448980,799.759340,4.0000,74.0000,173.0000,416.0000,4632.000
Total_Orders,49.0,61.285714,107.140290,1.0000,10.0000,27.0000,57.0000,619.000
Total_Customers,49.0,53.183673,80.854209,1.0000,10.0000,26.0000,53.0000,428.000


## 8. City-Level Performance

Business Question:

> Which cities represent the most valuable markets?

City-level aggregation helps identify geographic concentration
and high-value local markets.

In [47]:
city_performance = pd.read_sql_query(
    """
    SELECT
        City,
        State,
        Region,
        SUM(Sales) AS Total_Sales,
        SUM(Profit) AS Total_Profit,
        COUNT(DISTINCT Order_ID) AS Total_Orders,
        COUNT(DISTINCT Customer_ID) AS Total_Customers
    FROM superstore
    GROUP BY
        City,
        State,
        Region
    ORDER BY Total_Sales DESC
    LIMIT 20;
    """,
    conn
)

display(city_performance)
display(city_performance.describe().T)

,City,State,Region,Total_Sales,Total_Profit,Total_Orders,Total_Customers
0,New York City,New York,East,154919.6990,34760.2974,273,231
1,Los Angeles,California,West,120017.4405,19174.9301,230,200
2,San Francisco,California,West,80891.4220,10475.5284,169,150
3,Seattle,Washington,West,77924.5420,18839.3111,133,122
4,Philadelphia,Pennsylvania,East,77305.4400,-8065.9684,157,145
5,Houston,Texas,Central,45159.5078,-7197.7160,100,95
6,Chicago,Illinois,Central,39993.1610,-6497.3566,109,101
7,San Diego,California,West,29269.5500,3002.3181,41,41
8,Detroit,Michigan,Central,26631.2060,10803.1896,28,26
9,Dallas,Texas,Central,20019.1136,-1983.7422,57,56


,count,mean,std,min,25%,50%,75%,max
Total_Sales,20.0,39569.553545,41516.036032,9418.0600,12260.632500,17435.88430,53195.99085,154919.6990
Total_Profit,20.0,4970.310350,10133.775401,-8065.9684,-304.235525,3017.96275,7366.00920,34760.2974
Total_Orders,20.0,73.250000,80.152404,6.0000,16.250000,29.50000,115.00000,273.0000
Total_Customers,20.0,66.550000,69.268718,6.0000,16.000000,28.00000,106.25000,231.0000


## 9. Shipping Mode Performance

Business Question:

> Which shipping modes handle the most orders and generate the
> highest Sales?

Operational performance is compared using Sales, Profit,
Orders, Quantity, and Shipping Days.

In [48]:
shipping_performance = pd.read_sql_query(
    """
    SELECT
        Ship_Mode,
        COUNT(DISTINCT Order_ID) AS Total_Orders,
        SUM(Sales) AS Total_Sales,
        SUM(Profit) AS Total_Profit,
        SUM(Quantity) AS Total_Quantity,
        AVG(Shipping_Days) AS Avg_Shipping_Days
    FROM superstore
    GROUP BY Ship_Mode
    ORDER BY Total_Orders DESC;
    """,
    conn
)

display(shipping_performance.round(2))
display(shipping_performance.describe().T)

,Ship_Mode,Total_Orders,Total_Sales,Total_Profit,Total_Quantity,Avg_Shipping_Days
0,Standard Class,1773,912401.04,99767.39,13150,145.36
1,Second Class,568,314508.06,36936.03,4392,92.33
2,First Class,499,242936.72,29749.87,3554,60.88
3,Same Day,163,95958.50,8808.82,1221,2.23


,count,mean,std,min,25%,50%,75%,max
Total_Orders,4.0,750.750000,704.088240,163.000000,415.000000,533.500000,869.250000,1773.000000
Total_Sales,4.0,391451.080800,359017.857881,95958.501000,206192.164800,278722.391700,463981.307700,912401.038800
Total_Profit,4.0,43815.526475,39163.122036,8808.824300,24514.605950,33342.946500,52643.867025,99767.388600
Total_Quantity,4.0,5579.250000,5222.445109,1221.000000,2970.750000,3973.000000,6581.500000,13150.000000
Avg_Shipping_Days,4.0,75.199950,59.850345,2.225434,46.217601,76.607495,105.589844,145.359375


## 10. Payment Mode Performance

Business Question:

> Which payment methods are most frequently used and what is their
> associated Sales performance?

The analysis measures transaction volume and financial contribution
by payment method.

In [49]:
payment_performance = pd.read_sql_query(
    """
    SELECT
        Payment_Mode,
        COUNT(*) AS Total_Transactions,
        COUNT(DISTINCT Order_ID) AS Total_Orders,
        SUM(Sales) AS Total_Sales,
        SUM(Profit) AS Total_Profit,
        SUM(Quantity) AS Total_Quantity
    FROM superstore
    GROUP BY Payment_Mode
    ORDER BY Total_Transactions DESC;
    """,
    conn
)

payment_performance["Sales_Contribution_%"] = (
    payment_performance["Total_Sales"]
    / payment_performance["Total_Sales"].sum()
    * 100
)

display(payment_performance.round(2))
display(payment_performance.describe().T)

,Payment_Mode,Total_Transactions,Total_Orders,Total_Sales,Total_Profit,Total_Quantity,Sales_Contribution_%
0,COD,2453,1717,667417.75,82092.42,9247,42.62
1,Online,2164,1562,553993.46,54047.52,8167,35.38
2,Cards,1284,985,344393.11,39122.16,4903,21.99


,count,mean,std,min,25%,50%,75%,max
Total_Transactions,3.0,1967.000000,608.889974,1284.000000,1724.000000,2164.000000,2308.500000,2453.000000
Total_Orders,3.0,1421.333333,385.741278,985.000000,1273.500000,1562.000000,1639.500000,1717.000000
Total_Sales,3.0,521934.774400,163881.204088,344393.111200,449193.285950,553993.460700,610705.606000,667417.751300
Total_Profit,3.0,58420.701967,21816.381238,39122.157500,46584.840450,54047.523400,68069.974200,82092.425000
Total_Quantity,3.0,7439.000000,2261.652493,4903.000000,6535.000000,8167.000000,8707.000000,9247.000000
Sales_Contribution_%,3.0,33.333333,10.466263,21.994646,28.687702,35.380759,39.002677,42.624595


# 🔵 Stage 4 — CASE WHEN & Conditional Aggregation

This stage introduces business logic directly inside SQL queries
using CASE WHEN and conditional aggregation.

The objective is to convert raw transaction-level metrics into
business classifications, performance flags, customer segments,
profitability categories, and operational indicators.

## SQL Concepts

- CASE WHEN
- ELSE
- SUM(CASE WHEN ...)
- COUNT(CASE WHEN ...)
- Conditional Aggregation
- Calculated Business Metrics
- Business Classification

## Business Analysis Areas

- Sales Classification
- Profitability Classification
- Customer Value
- Product Performance
- Return Analysis
- Shipping Performance
- Regional Performance
- Category Performance
- Transaction Quality
- Business Contribution

## Core Metrics

Sales
Profit
Quantity
Profit_Margin
Shipping_Days
Returns
Order_ID
Customer_ID
Product_ID

## Important

CASE WHEN creates analytical business logic without modifying the
original database table.

The result is generated dynamically during SQL analysis.

---

## 1. Sales Value Classification

Business Question:

> How are transactions distributed across different Sales-value
> segments?

Transactions are classified into Low, Medium, High, and Very High
Sales groups.

In [50]:
sales_classification = pd.read_sql_query(
    """
    SELECT
        Order_ID,
        Customer_Name,
        Product_Name,
        Sales,
        CASE
            WHEN Sales < 100 THEN 'Low Sales'
            WHEN Sales < 500 THEN 'Medium Sales'
            WHEN Sales < 1000 THEN 'High Sales'
            ELSE 'Very High Sales'
        END AS Sales_Category
    FROM superstore
    ORDER BY Sales DESC;
    """,
    conn
)

display(sales_classification.head(20))
display(sales_classification.describe().T)

,Order_ID,Customer_Name,Product_Name,Sales,Sales_Category
0,US-2019-107440,Bill Shonely,"3D Systems Cube Printer, 2nd Generation, Magenta",9099.930,Very High Sales
1,CA-2020-166709,Hunter Lopez,Canon imageCLASS 2200 Advanced Copier,5517.970,Very High Sales
2,CA-2020-138289,Andy Reiter,GBC DocuBind P400 Electric Binding System,5455.960,Very High Sales
3,US-2019-140158,Daniel Raglin,Hewlett Packard LaserJet 3310 Copier,5399.910,Very High Sales
4,CA-2020-143112,Todd Sumrall,"3D Systems Cube Printer, 2nd Generation, Magenta",5234.960,Very High Sales
5,CA-2020-135909,Jane Waco,Fellowes PB500 Electric Punch Plastic Comb Bin...,5107.960,Very High Sales
6,CA-2019-136301,Edward Hooks,High Speed Automatic Electric Letter Opener,4912.590,Very High Sales
7,US-2019-143819,Karen Daniels,Ativa V4110MDD Micro-Cut Shredder,4899.930,Very High Sales
8,CA-2020-149881,Nick Crebassa,Cubify CubeX 3D Printer Double Head Print,4823.984,Very High Sales
9,CA-2019-158841,Sanjit Engle,HP Designjet T520 Inkjet Large Format Printer ...,4749.950,Very High Sales


,count,mean,std,min,25%,50%,75%,max
Sales,5901.0,265.345589,474.260645,0.836,71.976,128.648,265.17,9099.93


## 2. Sales Segment Distribution

Business Question:

> How many transactions fall into each Sales-value segment?

This converts transaction-level Sales into a business distribution.

In [51]:
sales_segment_distribution = pd.read_sql_query(
    """
    SELECT
        CASE
            WHEN Sales < 100 THEN 'Low Sales'
            WHEN Sales < 500 THEN 'Medium Sales'
            WHEN Sales < 1000 THEN 'High Sales'
            ELSE 'Very High Sales'
        END AS Sales_Category,
        COUNT(*) AS Total_Transactions,
        SUM(Sales) AS Total_Sales,
        AVG(Sales) AS Average_Sales
    FROM superstore
    GROUP BY Sales_Category
    ORDER BY Total_Sales DESC;
    """,
    conn
)

display(sales_segment_distribution.round(2))
display(sales_segment_distribution.describe().T)

,Sales_Category,Total_Transactions,Total_Sales,Average_Sales
0,Medium Sales,3296,652210.06,197.88
1,Very High Sales,279,533238.39,1911.25
2,High Sales,434,304611.35,701.87
3,Low Sales,1892,75744.52,40.03


,count,mean,std,min,25%,50%,75%,max
Total_Transactions,4.0,1475.250000,1414.688040,279.000000,395.250000,1163.000000,2243.000000,3296.000000
Total_Sales,4.0,391451.080800,255153.977961,75744.522000,247394.642175,418924.870250,562981.308875,652210.060700
Average_Sales,4.0,712.757888,847.379511,40.034103,158.417975,449.874367,1004.214279,1911.248715


## 3. Profitability Classification

Business Question:

> How many transactions are profitable, break-even, or loss-making?

This provides a direct view of transaction-level profitability.

In [52]:
profitability_classification = pd.read_sql_query(
    """
    SELECT
        CASE
            WHEN Profit < 0 THEN 'Loss'
            WHEN Profit = 0 THEN 'Break-Even'
            ELSE 'Profit'
        END AS Profit_Status,
        COUNT(*) AS Total_Transactions,
        SUM(Sales) AS Total_Sales,
        SUM(Profit) AS Total_Profit
    FROM superstore
    GROUP BY Profit_Status
    ORDER BY Total_Profit DESC;
    """,
    conn
)

display(profitability_classification.round(2))
display(profitability_classification.describe().T)

,Profit_Status,Total_Transactions,Total_Sales,Total_Profit
0,Profit,4764,1250612.74,266971.83
1,Break-Even,39,16780.30,0.00
2,Loss,1098,298411.29,-91709.73


,count,mean,std,min,25%,50%,75%,max
Total_Transactions,3.0,1967.000000,2479.471113,39.0000,568.50000,1098.0000,2931.0000,4.764000e+03
Total_Sales,3.0,521934.774400,646573.806796,16780.2970,157595.79110,298411.2852,774512.0131,1.250613e+06
Total_Profit,3.0,58420.701967,186340.681252,-91709.7279,-45854.86395,0.0000,133485.9169,2.669718e+05


## 4. Profit Margin Classification

Business Question:

> Which transactions have weak, healthy, or highly profitable
> margins?

Profit Margin is converted into meaningful business categories.

In [53]:
margin_classification = pd.read_sql_query(
    """
    SELECT
        CASE
            WHEN Profit_Margin < 0 THEN 'Negative Margin'
            WHEN Profit_Margin < 10 THEN 'Low Margin'
            WHEN Profit_Margin < 25 THEN 'Healthy Margin'
            ELSE 'High Margin'
        END AS Margin_Category,
        COUNT(*) AS Total_Transactions,
        SUM(Sales) AS Total_Sales,
        SUM(Profit) AS Total_Profit,
        AVG(Profit_Margin) AS Average_Margin
    FROM superstore
    GROUP BY Margin_Category
    ORDER BY Average_Margin DESC;
    """,
    conn
)

display(margin_classification.round(2))
display(margin_classification.describe().T)

,Margin_Category,Total_Transactions,Total_Sales,Total_Profit,Average_Margin
0,High Margin,1925,462260.01,191071.02,39.76
1,Healthy Margin,990,320600.24,53259.87,16.16
2,Low Margin,1888,484532.79,22640.95,4.66
3,Negative Margin,1098,298411.29,-91709.73,-38.22


,count,mean,std,min,25%,50%,75%,max
Total_Transactions,4.0,1475.250000,500.140897,990.000000,1071.000000,1493.000000,1897.250000,1925.000000
Total_Sales,4.0,391451.080800,95488.836829,298411.285200,315053.001300,391430.125000,467828.204500,484532.788000
Total_Profit,4.0,43815.526475,116316.855380,-91709.727900,-5946.720825,37950.408200,87712.655500,191071.017400
Average_Margin,4.0,5.589242,32.659811,-38.223673,-6.060217,10.408404,22.057864,39.763834


## 5. Conditional Sales & Profit

Business Question:

> How much Sales and Profit come from profitable versus loss-making
> transactions?

Conditional aggregation allows multiple business metrics to be
calculated in a single query.

In [54]:
conditional_profit_analysis = pd.read_sql_query(
    """
    SELECT
        COUNT(*) AS Total_Transactions,

        SUM(
            CASE
                WHEN Profit > 0 THEN Sales
                ELSE 0
            END
        ) AS Sales_From_Profitable_Transactions,

        SUM(
            CASE
                WHEN Profit < 0 THEN Sales
                ELSE 0
            END
        ) AS Sales_From_Loss_Transactions,

        SUM(
            CASE
                WHEN Profit > 0 THEN Profit
                ELSE 0
            END
        ) AS Total_Positive_Profit,

        SUM(
            CASE
                WHEN Profit < 0 THEN Profit
                ELSE 0
            END
        ) AS Total_Loss
    FROM superstore;
    """,
    conn
)

display(conditional_profit_analysis.round(2))

,Total_Transactions,Sales_From_Profitable_Transactions,Sales_From_Loss_Transactions,Total_Positive_Profit,Total_Loss
0,5901,1250612.74,298411.29,266971.83,-91709.73


## 6. Category Profitability Classification

Business Question:

> Which categories generate positive or negative overall profit?

Categories are classified based on their aggregated Profit.

In [55]:
category_profit_status = pd.read_sql_query(
    """
    SELECT
        Category,
        SUM(Sales) AS Total_Sales,
        SUM(Profit) AS Total_Profit,

        CASE
            WHEN SUM(Profit) < 0 THEN 'Loss-Making'
            WHEN SUM(Profit) = 0 THEN 'Break-Even'
            ELSE 'Profitable'
        END AS Profitability_Status

    FROM superstore

    GROUP BY Category

    ORDER BY Total_Profit DESC;
    """,
    conn
)

display(category_profit_status.round(2))
display(category_profit_status.describe().T)

,Category,Total_Sales,Total_Profit,Profitability_Status
0,Technology,470587.99,90458.25,Profitable
1,Office Supplies,643707.69,74797.25,Profitable
2,Furniture,451508.65,10006.61,Profitable


,count,mean,std,min,25%,50%,75%,max
Total_Sales,3.0,521934.774400,105889.031733,451508.6452,461048.31810,470587.9910,557147.83900,643707.6870
Total_Profit,3.0,58420.701967,42652.782892,10006.6112,42401.92865,74797.2461,82627.74735,90458.2486


## 7. Customer Value Classification

Business Question:

> Which customers are low-value, medium-value, or high-value
> revenue contributors?

Customers are classified according to their total Sales.

In [56]:
customer_value_segments = pd.read_sql_query(
    """
    SELECT
        Customer_ID,
        Customer_Name,
        SUM(Sales) AS Total_Sales,
        SUM(Profit) AS Total_Profit,
        COUNT(DISTINCT Order_ID) AS Total_Orders,

        CASE
            WHEN SUM(Sales) < 500 THEN 'Low Value'
            WHEN SUM(Sales) < 1500 THEN 'Medium Value'
            WHEN SUM(Sales) < 3000 THEN 'High Value'
            ELSE 'Very High Value'
        END AS Customer_Value

    FROM superstore

    GROUP BY
        Customer_ID,
        Customer_Name

    ORDER BY Total_Sales DESC;
    """,
    conn
)

display(customer_value_segments.head(20))
display(customer_value_segments.describe().T)

,Customer_ID,Customer_Name,Total_Sales,Total_Profit,Total_Orders,Customer_Value
0,CJ-12010,Caroline Jumper,11596.9740,858.7414,8,Very High Value
1,KF-16285,Karen Ferguson,10941.2740,1577.6134,6,Very High Value
2,SV-20365,Seth Vernon,10751.1480,1610.0020,7,Very High Value
3,HW-14935,Helen Wasserman,10074.9340,2146.9635,7,Very High Value
4,EH-13765,Edward Hooks,9542.9880,1164.6312,9,Very High Value
5,BS-11365,Bill Shonely,9199.7800,2405.3645,2,Very High Value
6,PK-19075,Pete Kriz,8812.0540,2024.9500,11,Very High Value
7,JL-15835,John Lee,8765.3320,839.9550,7,Very High Value
8,AB-10060,Adam Bellavance,8167.0800,2045.8747,7,Very High Value
9,JW-15220,Jane Waco,7933.5540,2084.5100,5,Very High Value


,count,mean,std,min,25%,50%,75%,max
Total_Sales,773.0,2025.620082,1778.099150,5.3040,748.7420,1584.7840,2717.4900,11596.9740
Total_Profit,773.0,226.729762,752.374279,-6652.7235,15.4452,115.7168,317.5336,8764.9483
Total_Orders,773.0,3.884864,2.013154,1.0000,2.0000,4.0000,5.0000,14.0000


## 8. Product Performance Classification

Business Question:

> Which products are strong sellers, average performers, or weak
> performers based on Sales?

Product performance is classified after aggregation.

In [57]:
product_performance_status = pd.read_sql_query(
    """
    SELECT
        Product_ID,
        Product_Name,
        Category,
        SUM(Sales) AS Total_Sales,
        SUM(Profit) AS Total_Profit,

        CASE
            WHEN SUM(Sales) < 500 THEN 'Low Performer'
            WHEN SUM(Sales) < 2000 THEN 'Average Performer'
            WHEN SUM(Sales) < 5000 THEN 'High Performer'
            ELSE 'Top Performer'
        END AS Performance_Level

    FROM superstore

    GROUP BY
        Product_ID,
        Product_Name,
        Category

    ORDER BY Total_Sales DESC;
    """,
    conn
)

display(product_performance_status.head(20))
display(product_performance_status.describe().T)

,Product_ID,Product_Name,Category,Total_Sales,Total_Profit,Performance_Level
0,TEC-MA-10001047,"3D Systems Cube Printer, 2nd Generation, Magenta",Technology,14334.890,3717.9714,Top Performer
1,TEC-CO-10004722,Canon imageCLASS 2200 Advanced Copier,Technology,14076.824,25199.9280,Top Performer
2,TEC-CO-10001449,Hewlett Packard LaserJet 3310 Copier,Technology,13837.732,6407.8932,Top Performer
3,OFF-BI-10001359,GBC DocuBind TL300 Electric Binding System,Office Supplies,12890.258,2753.7593,Top Performer
4,OFF-BI-10004995,GBC DocuBind P400 Electric Binding System,Office Supplies,12577.108,762.1544,Top Performer
5,TEC-PH-10001459,Samsung Galaxy Mega 6.3,Technology,12370.708,1696.7596,Top Performer
6,OFF-SU-10002881,Martin Yale Chadless Opener Electric Letter Op...,Office Supplies,12268.902,-1232.5588,Top Performer
7,FUR-CH-10002024,HON 5400 Series Task Chairs for Big and Tall,Furniture,11887.562,70.0980,Top Performer
8,FUR-CH-10001215,Global Troy Executive Leather Low-Back Tilter,Furniture,10217.894,776.5190,Top Performer
9,OFF-BI-10003527,Fellowes PB500 Electric Punch Plastic Comb Bin...,Office Supplies,9756.524,-508.3960,Top Performer


,count,mean,std,min,25%,50%,75%,max
Total_Sales,1783.0,878.185263,1430.452031,2.6100,211.5200,428.440,873.7210,14334.890
Total_Profit,1783.0,98.296190,709.745443,-6239.9792,5.0886,26.235,82.7292,25199.928


## 9. Shipping Performance Classification

Business Question:

> How are transactions distributed according to shipping duration?

Shipping duration is classified into operational performance
categories.

In [58]:
shipping_classification = pd.read_sql_query(
    """
    SELECT
        CASE
            WHEN Shipping_Days <= 30 THEN 'Fast'
            WHEN Shipping_Days <= 90 THEN 'Standard'
            WHEN Shipping_Days <= 180 THEN 'Slow'
            ELSE 'Very Slow'
        END AS Shipping_Performance,

        COUNT(*) AS Total_Transactions,
        SUM(Sales) AS Total_Sales,
        SUM(Profit) AS Total_Profit,
        AVG(Shipping_Days) AS Average_Shipping_Days

    FROM superstore

    WHERE Shipping_Days IS NOT NULL

    GROUP BY Shipping_Performance

    ORDER BY Average_Shipping_Days;
    """,
    conn
)

display(shipping_classification.round(2))
display(shipping_classification.describe().T)

,Shipping_Performance,Total_Transactions,Total_Sales,Total_Profit,Average_Shipping_Days
0,Fast,224,58531.91,2753.33,8.52
1,Standard,380,113583.94,12916.48,59.99
2,Slow,895,235116.91,30906.72,126.76
3,Very Slow,186,38196.55,12876.85,189.94


,count,mean,std,min,25%,50%,75%,max
Total_Transactions,4.0,421.250000,326.797975,186.000000,214.500000,302.000000,508.750000,895.00000
Total_Sales,4.0,111357.327425,88439.085912,38196.552000,53448.069750,86057.924300,143967.181975,235116.90910
Total_Profit,4.0,14863.347225,11715.782488,2753.332400,10345.972775,12896.668650,17414.043100,30906.71920
Average_Shipping_Days,4.0,96.300775,78.994984,8.517857,47.119596,93.372192,142.553371,189.94086


## 10. Return Analysis Classification

Business Question:

> How do Sales and Profit differ between returned and non-returned
> transactions?

Conditional aggregation is used to calculate both groups in one
business query.

In [59]:
return_analysis = pd.read_sql_query(
    """
    SELECT
        COUNT(*) AS Total_Transactions,

        SUM(
            CASE
                WHEN Return_Flag = 1 THEN 1
                ELSE 0
            END
        ) AS Returned_Transactions,

        SUM(
            CASE
                WHEN Return_Flag = 0 THEN 1
                ELSE 0
            END
        ) AS Non_Returned_Transactions,

        SUM(
            CASE
                WHEN Return_Flag = 1 THEN Sales
                ELSE 0
            END
        ) AS Returned_Sales,

        SUM(
            CASE
                WHEN Return_Flag = 0 THEN Sales
                ELSE 0
            END
        ) AS Non_Returned_Sales,

        SUM(
            CASE
                WHEN Return_Flag = 1 THEN Profit
                ELSE 0
            END
        ) AS Returned_Profit,

        SUM(
            CASE
                WHEN Return_Flag = 0 THEN Profit
                ELSE 0
            END
        ) AS Non_Returned_Profit

    FROM superstore;
    """,
    conn
)

display(return_analysis.round(2))

,Total_Transactions,Returned_Transactions,Non_Returned_Transactions,Returned_Sales,Non_Returned_Sales,Returned_Profit,Non_Returned_Profit
0,5901,287,5614,88366.51,1477437.81,15438.95,159823.16


## 11. Category × Profitability

Business Question:

> Which categories contain the highest proportion of profitable
> and loss-making transactions?

This combines category grouping with conditional aggregation.

In [60]:
category_profit_mix = pd.read_sql_query(
    """
    SELECT
        Category,

        COUNT(*) AS Total_Transactions,

        SUM(
            CASE
                WHEN Profit > 0 THEN 1
                ELSE 0
            END
        ) AS Profitable_Transactions,

        SUM(
            CASE
                WHEN Profit < 0 THEN 1
                ELSE 0
            END
        ) AS Loss_Transactions,

        SUM(Sales) AS Total_Sales,
        SUM(Profit) AS Total_Profit

    FROM superstore

    GROUP BY Category

    ORDER BY Total_Sales DESC;
    """,
    conn
)

category_profit_mix["Profit_Transaction_%"] = (
    category_profit_mix["Profitable_Transactions"]
    / category_profit_mix["Total_Transactions"]
    * 100
)

category_profit_mix["Loss_Transaction_%"] = (
    category_profit_mix["Loss_Transactions"]
    / category_profit_mix["Total_Transactions"]
    * 100
)

display(category_profit_mix.round(2))
display(category_profit_mix.describe().T)

,Category,Total_Transactions,Profitable_Transactions,Loss_Transactions,Total_Sales,Total_Profit,Profit_Transaction_%,Loss_Transaction_%
0,Office Supplies,3569,3042,511,643707.69,74797.25,85.23,14.32
1,Technology,1083,920,163,470587.99,90458.25,84.95,15.05
2,Furniture,1249,802,424,451508.65,10006.61,64.21,33.95


,count,mean,std,min,25%,50%,75%,max
Total_Transactions,3.0,1967.000000,1389.853230,1083.000000,1166.000000,1249.000000,2409.000000,3569.000000
Profitable_Transactions,3.0,1588.000000,1260.582405,802.000000,861.000000,920.000000,1981.000000,3042.000000
Loss_Transactions,3.0,366.000000,181.104942,163.000000,293.500000,424.000000,467.500000,511.000000
Total_Sales,3.0,521934.774400,105889.031733,451508.645200,461048.318100,470587.991000,557147.839000,643707.687000
Total_Profit,3.0,58420.701967,42652.782892,10006.611200,42401.928650,74797.246100,82627.747350,90458.248600
Profit_Transaction_%,3.0,78.131514,12.056040,64.211369,74.580292,84.949215,85.091587,85.233959
Loss_Transaction_%,3.0,21.105226,11.127477,14.317736,14.684260,15.050785,24.498971,33.947158


## 12. Regional Business Health

Business Question:

> Which regions have strong Sales and Profit, and which regions
> require attention?

A business health classification is created using aggregated
Sales and Profit.

In [61]:
regional_business_health = pd.read_sql_query(
    """
    SELECT
        Region,
        SUM(Sales) AS Total_Sales,
        SUM(Profit) AS Total_Profit,

        CASE
            WHEN SUM(Sales) > 0
                 AND SUM(Profit) > 0
                THEN 'Healthy'

            WHEN SUM(Sales) > 0
                 AND SUM(Profit) <= 0
                THEN 'Revenue but Weak Profit'

            ELSE 'Critical'
        END AS Business_Health

    FROM superstore

    GROUP BY Region

    ORDER BY Total_Sales DESC;
    """,
    conn
)

display(regional_business_health.round(2))
display(regional_business_health.describe().T)

,Region,Total_Sales,Total_Profit,Business_Health
0,West,522441.05,67859.96,Healthy
1,East,450234.67,53400.42,Healthy
2,Central,341007.52,27450.01,Healthy
3,South,252121.08,26551.72,Healthy


,count,mean,std,min,25%,50%,75%,max
Total_Sales,4.0,391451.080800,119123.582455,252121.0810,318785.9134,395621.0951,468286.262500,522441.0520
Total_Profit,4.0,43815.526475,20296.751213,26551.7163,27225.4344,40425.2157,57015.307775,67859.9582


## 13. Segment-wise Profitable Sales

Business Question:

> How much Sales comes from profitable transactions within each
> customer segment?

This query combines GROUP BY with conditional SUM().

In [62]:
segment_conditional_sales = pd.read_sql_query(
    """
    SELECT
        Segment,

        SUM(Sales) AS Total_Sales,

        SUM(
            CASE
                WHEN Profit > 0 THEN Sales
                ELSE 0
            END
        ) AS Profitable_Sales,

        SUM(
            CASE
                WHEN Profit < 0 THEN Sales
                ELSE 0
            END
        ) AS Loss_Sales,

        SUM(Profit) AS Total_Profit

    FROM superstore

    GROUP BY Segment

    ORDER BY Total_Sales DESC;
    """,
    conn
)

segment_conditional_sales["Profitable_Sales_%"] = (
    segment_conditional_sales["Profitable_Sales"]
    / segment_conditional_sales["Total_Sales"]
    * 100
)

display(segment_conditional_sales.round(2))
display(segment_conditional_sales.describe().T)

,Segment,Total_Sales,Profitable_Sales,Loss_Sales,Total_Profit,Profitable_Sales_%
0,Consumer,753002.13,586462.68,154902.46,81338.59,77.88
1,Corporate,509743.13,419126.90,87359.79,57805.80,82.22
2,Home Office,303059.07,245023.17,56149.03,36117.72,80.85


,count,mean,std,min,25%,50%,75%,max
Total_Sales,3.0,521934.774400,225219.152815,303059.067900,406401.09705,509743.12620,631372.627650,753002.129100
Profitable_Sales,3.0,416870.913667,170730.933829,245023.166000,332075.03275,419126.89950,502794.787500,586462.675500
Loss_Sales,3.0,99470.428400,50478.319508,56149.029900,71754.41130,87359.79270,121131.127650,154902.462600
Total_Profit,3.0,58420.701967,22616.704210,36117.719300,46961.75920,57805.79910,69572.193300,81338.587500
Profitable_Sales_%,3.0,80.318799,2.218167,77.883269,79.36662,80.84997,81.536564,82.223159


# 🔵 Stage 5 — SQL Subqueries & Benchmark Analysis

This stage introduces SQL subqueries for solving business questions
that require comparing individual records or groups against
calculated business benchmarks.

## SQL Concepts

- Scalar Subqueries
- Subqueries with WHERE
- Subqueries with HAVING
- Subqueries with FROM
- Correlated Subqueries
- Aggregate Subqueries
- Nested Business Logic

## Business Objectives

- Find transactions above average Sales.
- Find transactions below average Profit.
- Find customers above average customer revenue.
- Find products above average product Sales.
- Find categories above overall average Sales.
- Identify high-value customers.
- Identify products generating above-average Profit.
- Compare regional performance against overall business performance.
- Identify transactions exceeding category benchmarks.
- Identify customers with above-average order frequency.

## Business Flow

```text
Raw Transactions
       ↓
Subquery Benchmark
       ↓
Compare Individual / Group Result
       ↓
Filter
       ↓
Business Insight


---

# 1️⃣ Transactions Above Average Sales

Business Question:

> Which transactions generate Sales greater than the overall
> average transaction Sales?

The average Sales is calculated inside a subquery and then used
as a benchmark for filtering transactions.

In [63]:
above_average_sales = pd.read_sql_query(
    """
    SELECT
        Order_ID,
        Order_Date,
        Customer_Name,
        Product_Name,
        Category,
        Sales,
        Profit
    FROM superstore
    WHERE Sales > (
        SELECT AVG(Sales)
        FROM superstore
    )
    ORDER BY Sales DESC
    LIMIT 20;
    """,
    conn
)

display(above_average_sales)
display(above_average_sales.describe().T)

,Order_ID,Order_Date,Customer_Name,Product_Name,Category,Sales,Profit
0,US-2019-107440,NaN,Bill Shonely,"3D Systems Cube Printer, 2nd Generation, Magenta",Technology,9099.930,2365.9818
1,CA-2020-166709,NaN,Hunter Lopez,Canon imageCLASS 2200 Advanced Copier,Technology,5517.970,5039.9856
2,CA-2020-138289,NaN,Andy Reiter,GBC DocuBind P400 Electric Binding System,Office Supplies,5455.960,2504.2216
3,US-2019-140158,2019-04-10,Daniel Raglin,Hewlett Packard LaserJet 3310 Copier,Technology,5399.910,2591.9568
4,CA-2020-143112,2020-05-10,Todd Sumrall,"3D Systems Cube Printer, 2nd Generation, Magenta",Technology,5234.960,1351.9896
5,CA-2020-135909,NaN,Jane Waco,Fellowes PB500 Electric Punch Plastic Comb Bin...,Office Supplies,5107.960,1906.4850
6,CA-2019-136301,NaN,Edward Hooks,High Speed Automatic Electric Letter Opener,Office Supplies,4912.590,196.5036
7,US-2019-143819,2019-01-03,Karen Daniels,Ativa V4110MDD Micro-Cut Shredder,Technology,4899.930,2400.9657
8,CA-2020-149881,2020-01-04,Nick Crebassa,Cubify CubeX 3D Printer Double Head Print,Technology,4823.984,359.9988
9,CA-2019-158841,2019-02-02,Sanjit Engle,HP Designjet T520 Inkjet Large Format Printer ...,Technology,4749.950,2799.9840


,count,mean,std,min,25%,50%,75%,max
Sales,20.0,4964.41965,1059.357254,4158.912,4428.35450,4721.3430,5139.710000,9099.9300
Profit,20.0,951.45673,2265.258045,-6599.978,294.82005,1383.7096,2374.727775,5039.9856


## 2. Transactions Below Average Profit

Business Question:

> Which transactions generate Profit below the overall average
> transaction Profit?

This helps identify transactions performing below the business
profit benchmark.

In [64]:
below_average_profit = pd.read_sql_query(
    """
    SELECT
        Order_ID,
        Customer_Name,
        Product_Name,
        Category,
        Sales,
        Profit
    FROM superstore
    WHERE Profit < (
        SELECT AVG(Profit)
        FROM superstore
    )
    ORDER BY Profit ASC
    LIMIT 20;
    """,
    conn
)

display(below_average_profit)
display(below_average_profit.describe().T)

,Order_ID,Customer_Name,Product_Name,Category,Sales,Profit
0,CA-2019-108196,Cindy Stewart,Cubify CubeX 3D Printer Double Head Print,Technology,4499.985,-6599.9780
1,US-2020-168116,Grant Thornton,Cubify CubeX 3D Printer Triple Head Print,Technology,3009.980,-3839.9904
2,CA-2020-134845,Sharelle Roach,Lexmark MX611dhe Monochrome Laser Printer,Technology,2579.985,-3399.9800
3,US-2020-122714,Henry Goldwyn,Ibico EPK-21 Electric Binding System,Office Supplies,1901.990,-2929.4845
4,CA-2020-131254,Nathan Cano,Fellowes PB500 Electric Punch Plastic Comb Bin...,Office Supplies,1556.188,-2287.7820
5,CA-2019-130946,Zuschuss Carroll,GBC DocuBind P400 Electric Binding System,Office Supplies,1088.792,-1850.9464
6,US-2020-120390,Tracy Hopkins,GBC DocuBind P400 Electric Binding System,Office Supplies,1677.188,-1306.5504
7,CA-2020-128363,Dan Campbell,GBC DocuBind TL300 Electric Binding System,Office Supplies,1641.582,-1237.8462
8,CA-2020-152093,Skye Norling,Fellowes PB500 Electric Punch Plastic Comb Bin...,Office Supplies,804.594,-1143.8910
9,US-2020-148551,David Bremer,GBC Ibimaster 500 Manual ProClick Binding System,Office Supplies,790.980,-1141.4700


,count,mean,std,min,25%,50%,75%,max
Sales,20.0,1897.431750,1266.485743,334.620,1017.7425,1598.8850,2396.333250,4692.736
Profit,20.0,-1733.691575,1467.664602,-6599.978,-1960.1553,-1095.4053,-930.263625,-760.980


## 3. Customers Above Average Revenue

Business Question:

> Which customers generate more Sales than the average customer?

First, customer-level Sales are aggregated. The outer query then
compares each customer against the average customer revenue.

In [65]:
high_value_customers = pd.read_sql_query(
    """
    SELECT
        Customer_ID,
        Customer_Name,
        Total_Sales,
        Total_Profit,
        Total_Orders
    FROM (
        SELECT
            Customer_ID,
            Customer_Name,
            SUM(Sales) AS Total_Sales,
            SUM(Profit) AS Total_Profit,
            COUNT(DISTINCT Order_ID) AS Total_Orders
        FROM superstore
        GROUP BY
            Customer_ID,
            Customer_Name
    )
    WHERE Total_Sales > (
        SELECT AVG(Customer_Sales)
        FROM (
            SELECT
                Customer_ID,
                SUM(Sales) AS Customer_Sales
            FROM superstore
            GROUP BY Customer_ID
        )
    )
    ORDER BY Total_Sales DESC;
    """,
    conn
)

display(high_value_customers)
display(high_value_customers.describe().T)

,Customer_ID,Customer_Name,Total_Sales,Total_Profit,Total_Orders
0,CJ-12010,Caroline Jumper,11596.974,858.7414,8
1,KF-16285,Karen Ferguson,10941.274,1577.6134,6
2,SV-20365,Seth Vernon,10751.148,1610.0020,7
3,HW-14935,Helen Wasserman,10074.934,2146.9635,7
4,EH-13765,Edward Hooks,9542.988,1164.6312,9
...,...,...,...,...,...
298,SA-20830,Sue Ann Reed,2051.450,160.7448,6
299,CK-12595,Clytie Kelty,2046.986,533.1557,8
300,EB-13750,Edward Becker,2046.700,-257.7020,8
301,RB-19705,Roger Barcio,2030.872,325.8525,2


,count,mean,std,min,25%,50%,75%,max
Total_Sales,303.0,3690.965243,1727.062705,2028.1120,2485.59200,3158.7220,4186.74800,11596.9740
Total_Profit,303.0,423.464051,1037.698374,-6652.7235,96.32415,328.0549,695.36565,6976.0959
Total_Orders,303.0,5.118812,1.942657,1.0000,4.00000,5.0000,6.00000,14.0000


## 4. Products Above Average Sales

Business Question:

> Which products generate more Sales than the average product?

The inner query calculates product-level Sales while the benchmark
subquery calculates average product Sales.

In [71]:
# Identify products whose total Sales are above the average product Sales

above_average_products = pd.read_sql_query(
    """
    SELECT
        Product_ID,
        Product_Name,
        Category,
        "Sub-Category" AS Sub_Category,
        Total_Sales,
        Total_Profit
    FROM (
        SELECT
            Product_ID,
            Product_Name,
            Category,
            "Sub-Category",
            SUM(Sales) AS Total_Sales,
            SUM(Profit) AS Total_Profit
        FROM superstore
        GROUP BY
            Product_ID,
            Product_Name,
            Category,
            "Sub-Category"
    ) AS product_summary
    WHERE Total_Sales > (
        SELECT AVG(Product_Sales)
        FROM (
            SELECT
                Product_ID,
                SUM(Sales) AS Product_Sales
            FROM superstore
            GROUP BY Product_ID
        ) AS product_average
    )
    ORDER BY Total_Sales DESC;
    """,
    conn
)

display(above_average_products)

display(
    above_average_products.describe().T
)

,Product_ID,Product_Name,Category,Sub_Category,Total_Sales,Total_Profit
0,TEC-MA-10001047,"3D Systems Cube Printer, 2nd Generation, Magenta",Technology,Machines,14334.890,3717.9714
1,TEC-CO-10004722,Canon imageCLASS 2200 Advanced Copier,Technology,Copiers,14076.824,25199.9280
2,TEC-CO-10001449,Hewlett Packard LaserJet 3310 Copier,Technology,Copiers,13837.732,6407.8932
3,OFF-BI-10001359,GBC DocuBind TL300 Electric Binding System,Office Supplies,Binders,12890.258,2753.7593
4,OFF-BI-10004995,GBC DocuBind P400 Electric Binding System,Office Supplies,Binders,12577.108,762.1544
...,...,...,...,...,...,...
430,OFF-PA-10004071,"Eaton Premium Continuous-Feed Paper, 25% Cotto...",Office Supplies,Paper,903.816,355.0720
431,TEC-AC-10000358,Imation Secure Drive + Hardware Encrypted USB ...,Technology,Accessories,901.860,127.6464
432,OFF-PA-10001289,White Computer Printout Paper by Universal,Office Supplies,Paper,901.224,374.0340
433,OFF-PA-10003673,Strathmore Photo Mount Cards,Office Supplies,Paper,900.436,106.3104


,count,mean,std,min,25%,50%,75%,max
Total_Sales,435.0,2515.486971,2166.106219,898.9000,1225.8950,1707.25,3016.37000,14334.890
Total_Profit,435.0,307.375189,1410.331877,-6239.9792,-0.0984,146.82,437.44985,25199.928


## 6. Products With Above-Average Profit

Business Question:

> Which products generate more Profit than the average product?

This identifies products with stronger-than-average profitability.

In [72]:
high_profit_products = pd.read_sql_query(
    """
    SELECT
        Product_ID,
        Product_Name,
        Category,
        Total_Sales,
        Total_Profit
    FROM (
        SELECT
            Product_ID,
            Product_Name,
            Category,
            SUM(Sales) AS Total_Sales,
            SUM(Profit) AS Total_Profit
        FROM superstore
        GROUP BY
            Product_ID,
            Product_Name,
            Category
    )
    WHERE Total_Profit > (
        SELECT AVG(Product_Profit)
        FROM (
            SELECT
                Product_ID,
                SUM(Profit) AS Product_Profit
            FROM superstore
            GROUP BY Product_ID
        )
    )
    ORDER BY Total_Profit DESC;
    """,
    conn
)

display(high_profit_products)
display(high_profit_products.describe().T)

,Product_ID,Product_Name,Category,Total_Sales,Total_Profit
0,TEC-CO-10004722,Canon imageCLASS 2200 Advanced Copier,Technology,14076.824,25199.9280
1,TEC-CO-10001449,Hewlett Packard LaserJet 3310 Copier,Technology,13837.732,6407.8932
2,TEC-MA-10001047,"3D Systems Cube Printer, 2nd Generation, Magenta",Technology,14334.890,3717.9714
3,TEC-MA-10001127,HP Designjet T520 Inkjet Large Format Printer ...,Technology,4749.950,2799.9840
4,OFF-BI-10001359,GBC DocuBind TL300 Electric Binding System,Office Supplies,12890.258,2753.7593
...,...,...,...,...,...
395,OFF-BI-10002133,"Wilson Jones Elliptical Ring 3 1/2"" Capacity B...",Office Supplies,805.480,101.4360
396,OFF-AP-10000159,Belkin F9M820V08 8 Outlet Surge,Office Supplies,678.740,101.0030
397,OFF-AR-10004022,Panasonic KP-380BK Classic Electric Pencil Sha...,Office Supplies,591.504,100.7440
398,OFF-PA-10003039,Xerox 1960,Office Supplies,269.036,100.6850


,count,mean,std,min,25%,50%,75%,max
Total_Sales,400.0,1950.789015,2120.340560,269.0360,738.425000,1169.791,2465.06550,14334.890
Total_Profit,400.0,491.235320,1356.569737,100.0728,147.229875,244.557,452.50515,25199.928


## 7. Regions Above Average Sales

Business Question:

> Which regions generate Sales above the average regional
> performance?

This identifies geographically strong revenue markets.

In [73]:
high_sales_regions = pd.read_sql_query(
    """
    SELECT
        Region,
        Total_Sales,
        Total_Profit
    FROM (
        SELECT
            Region,
            SUM(Sales) AS Total_Sales,
            SUM(Profit) AS Total_Profit
        FROM superstore
        GROUP BY Region
    )
    WHERE Total_Sales > (
        SELECT AVG(Region_Sales)
        FROM (
            SELECT
                Region,
                SUM(Sales) AS Region_Sales
            FROM superstore
            GROUP BY Region
        )
    )
    ORDER BY Total_Sales DESC;
    """,
    conn
)

display(high_sales_regions)
display(high_sales_regions.describe().T)

,Region,Total_Sales,Total_Profit
0,West,522441.052,67859.9582
1,East,450234.666,53400.4243


,count,mean,std,min,25%,50%,75%,max
Total_Sales,2.0,486337.85900,51057.625186,450234.6660,468286.262500,486337.85900,504389.455500,522441.0520
Total_Profit,2.0,60630.19125,10224.434473,53400.4243,57015.307775,60630.19125,64245.074725,67859.9582


## 8. Customers With Above-Average Order Frequency

Business Question:

> Which customers place more orders than the average customer?

This helps identify highly active customers independent of their
total revenue.

In [74]:
frequent_customers = pd.read_sql_query(
    """
    SELECT
        Customer_ID,
        Customer_Name,
        Total_Orders,
        Total_Sales,
        Total_Profit
    FROM (
        SELECT
            Customer_ID,
            Customer_Name,
            COUNT(DISTINCT Order_ID) AS Total_Orders,
            SUM(Sales) AS Total_Sales,
            SUM(Profit) AS Total_Profit
        FROM superstore
        GROUP BY
            Customer_ID,
            Customer_Name
    )
    WHERE Total_Orders > (
        SELECT AVG(Customer_Orders)
        FROM (
            SELECT
                Customer_ID,
                COUNT(DISTINCT Order_ID) AS Customer_Orders
            FROM superstore
            GROUP BY Customer_ID
        )
    )
    ORDER BY Total_Orders DESC;
    """,
    conn
)

display(frequent_customers)
display(frequent_customers.describe().T)

,Customer_ID,Customer_Name,Total_Orders,Total_Sales,Total_Profit
0,EP-13915,Emily Phan,14,7230.6328,-391.2423
1,PK-19075,Pete Kriz,11,8812.0540,2024.9500
2,SH-19975,Sally Hughsby,11,3381.4915,447.9062
3,SJ-20125,Sanjit Jacobs,11,4931.1480,209.9379
4,BD-11320,Bill Donatelli,10,3740.9310,295.0142
...,...,...,...,...,...
394,VB-21745,Victoria Brennan,4,1809.4120,286.6274
395,VG-21790,Vivek Gonzalez,4,846.8340,124.0900
396,VG-21805,Vivek Grady,4,177.8360,30.5510
397,VM-21835,Vivian Mathis,4,491.4100,110.1253


,count,mean,std,min,25%,50%,75%,max
Total_Orders,399.0,5.463659,1.457284,4.0000,4.0000,5.000,6.0000,14.0000
Total_Sales,399.0,2813.955743,1878.871183,177.8360,1490.7305,2374.928,3524.7748,11596.9740
Total_Profit,399.0,292.440584,821.902148,-6652.7235,51.2418,189.237,421.4193,6976.0959


## 9. Transactions Above Category Average Sales

Business Question:

> Which transactions generate more Sales than the average Sales
> of their own category?

This is a deeper benchmark analysis because each transaction is
compared against its category-specific average rather than the
overall dataset average.

In [75]:
category_benchmark_transactions = pd.read_sql_query(
    """
    SELECT
        s.Order_ID,
        s.Customer_Name,
        s.Product_Name,
        s.Category,
        s.Sales,
        s.Profit
    FROM superstore AS s
    WHERE s.Sales > (
        SELECT AVG(c.Sales)
        FROM superstore AS c
        WHERE c.Category = s.Category
    )
    ORDER BY s.Sales DESC
    LIMIT 20;
    """,
    conn
)

display(category_benchmark_transactions)
display(category_benchmark_transactions.describe().T)

,Order_ID,Customer_Name,Product_Name,Category,Sales,Profit
0,US-2019-107440,Bill Shonely,"3D Systems Cube Printer, 2nd Generation, Magenta",Technology,9099.930,2365.9818
1,CA-2020-166709,Hunter Lopez,Canon imageCLASS 2200 Advanced Copier,Technology,5517.970,5039.9856
2,CA-2020-138289,Andy Reiter,GBC DocuBind P400 Electric Binding System,Office Supplies,5455.960,2504.2216
3,US-2019-140158,Daniel Raglin,Hewlett Packard LaserJet 3310 Copier,Technology,5399.910,2591.9568
4,CA-2020-143112,Todd Sumrall,"3D Systems Cube Printer, 2nd Generation, Magenta",Technology,5234.960,1351.9896
5,CA-2020-135909,Jane Waco,Fellowes PB500 Electric Punch Plastic Comb Bin...,Office Supplies,5107.960,1906.4850
6,CA-2019-136301,Edward Hooks,High Speed Automatic Electric Letter Opener,Office Supplies,4912.590,196.5036
7,US-2019-143819,Karen Daniels,Ativa V4110MDD Micro-Cut Shredder,Technology,4899.930,2400.9657
8,CA-2020-149881,Nick Crebassa,Cubify CubeX 3D Printer Double Head Print,Technology,4823.984,359.9988
9,CA-2019-158841,Sanjit Engle,HP Designjet T520 Inkjet Large Format Printer ...,Technology,4749.950,2799.9840


,count,mean,std,min,25%,50%,75%,max
Sales,20.0,4964.41965,1059.357254,4158.912,4428.35450,4721.3430,5139.710000,9099.9300
Profit,20.0,951.45673,2265.258045,-6599.978,294.82005,1383.7096,2374.727775,5039.9856


## 10. Products Above Their Category Average

Business Question:

> Which products outperform the average product Sales within
> their own category?

This provides a category-adjusted product performance benchmark.

In [76]:
products_above_category_average = pd.read_sql_query(
    """
    SELECT
        p.Product_ID,
        p.Product_Name,
        p.Category,
        p.Total_Sales,
        p.Total_Profit
    FROM (
        SELECT
            Product_ID,
            Product_Name,
            Category,
            SUM(Sales) AS Total_Sales,
            SUM(Profit) AS Total_Profit
        FROM superstore
        GROUP BY
            Product_ID,
            Product_Name,
            Category
    ) AS p
    WHERE p.Total_Sales > (
        SELECT AVG(category_product_sales)
        FROM (
            SELECT
                Product_ID,
                Category,
                SUM(Sales) AS category_product_sales
            FROM superstore
            WHERE Category = p.Category
            GROUP BY
                Product_ID,
                Category
        )
    )
    ORDER BY p.Total_Sales DESC;
    """,
    conn
)

display(products_above_category_average)
display(products_above_category_average.describe().T)

,Product_ID,Product_Name,Category,Total_Sales,Total_Profit
0,TEC-MA-10001047,"3D Systems Cube Printer, 2nd Generation, Magenta",Technology,14334.890,3717.9714
1,TEC-CO-10004722,Canon imageCLASS 2200 Advanced Copier,Technology,14076.824,25199.9280
2,TEC-CO-10001449,Hewlett Packard LaserJet 3310 Copier,Technology,13837.732,6407.8932
3,OFF-BI-10001359,GBC DocuBind TL300 Electric Binding System,Office Supplies,12890.258,2753.7593
4,OFF-BI-10004995,GBC DocuBind P400 Electric Binding System,Office Supplies,12577.108,762.1544
...,...,...,...,...,...
455,OFF-AP-10001492,"Acco Six-Outlet Power Strip, 4' Cord Length",Office Supplies,626.260,51.5476
456,OFF-ST-10000777,"Companion Letter/Legal File, Black",Office Supplies,625.880,137.4464
457,OFF-PA-10002245,Xerox 1895,Office Supplies,624.700,40.3650
458,OFF-BI-10003718,GBC Therma-A-Bind 250T Electric Binding System,Office Supplies,622.352,206.6232


,count,mean,std,min,25%,50%,75%,max
Total_Sales,460.0,2360.151953,2184.334521,621.9800,953.1520,1621.8505,2902.98200,14334.890
Total_Profit,460.0,301.512762,1369.306806,-6239.9792,10.3685,130.0633,414.35835,25199.928


## 11. Business Benchmark Summary

The subquery analysis can be summarized into benchmark-based
business groups.

The objective is to identify:

- Above-average transactions
- Above-average customers
- Above-average products
- Above-average categories
- Above-average regions
- High-frequency customers
- Category-adjusted high performers

In [77]:
# Identify products whose total Sales are above the average product Sales

above_average_products = pd.read_sql_query(
    """
    SELECT
        Product_ID,
        Product_Name,
        Category,
        "Sub-Category" AS Sub_Category,
        Total_Sales,
        Total_Profit
    FROM (
        SELECT
            Product_ID,
            Product_Name,
            Category,
            "Sub-Category",
            SUM(Sales) AS Total_Sales,
            SUM(Profit) AS Total_Profit
        FROM superstore
        GROUP BY
            Product_ID,
            Product_Name,
            Category,
            "Sub-Category"
    ) AS product_summary
    WHERE Total_Sales > (
        SELECT AVG(Product_Sales)
        FROM (
            SELECT
                Product_ID,
                SUM(Sales) AS Product_Sales
            FROM superstore
            GROUP BY Product_ID
        ) AS product_average
    )
    ORDER BY Total_Sales DESC;
    """,
    conn
)

display(above_average_products)

display(
    above_average_products.describe().T
)

,Product_ID,Product_Name,Category,Sub_Category,Total_Sales,Total_Profit
0,TEC-MA-10001047,"3D Systems Cube Printer, 2nd Generation, Magenta",Technology,Machines,14334.890,3717.9714
1,TEC-CO-10004722,Canon imageCLASS 2200 Advanced Copier,Technology,Copiers,14076.824,25199.9280
2,TEC-CO-10001449,Hewlett Packard LaserJet 3310 Copier,Technology,Copiers,13837.732,6407.8932
3,OFF-BI-10001359,GBC DocuBind TL300 Electric Binding System,Office Supplies,Binders,12890.258,2753.7593
4,OFF-BI-10004995,GBC DocuBind P400 Electric Binding System,Office Supplies,Binders,12577.108,762.1544
...,...,...,...,...,...,...
430,OFF-PA-10004071,"Eaton Premium Continuous-Feed Paper, 25% Cotto...",Office Supplies,Paper,903.816,355.0720
431,TEC-AC-10000358,Imation Secure Drive + Hardware Encrypted USB ...,Technology,Accessories,901.860,127.6464
432,OFF-PA-10001289,White Computer Printout Paper by Universal,Office Supplies,Paper,901.224,374.0340
433,OFF-PA-10003673,Strathmore Photo Mount Cards,Office Supplies,Paper,900.436,106.3104


,count,mean,std,min,25%,50%,75%,max
Total_Sales,435.0,2515.486971,2166.106219,898.9000,1225.8950,1707.25,3016.37000,14334.890
Total_Profit,435.0,307.375189,1410.331877,-6239.9792,-0.0984,146.82,437.44985,25199.928


# 🔵 Stage 6 — CTE & Multi-Step Business Analysis

Common Table Expressions (CTEs) are used to divide complex SQL
business problems into multiple logical steps.

A CTE creates a temporary named result set using the WITH clause.

## SQL Concept

```sql
WITH cte_name AS (
    SELECT ...
)
SELECT ...
FROM cte_name;


---

# 1️⃣ Customer Sales CTE


Business Question:

> Which customers generate the highest Sales and Profit?

The first CTE aggregates transaction-level data to customer level.
The final query ranks customers according to Total Sales.

In [78]:
# Aggregate transaction data to customer level using a CTE

customer_sales_cte = pd.read_sql_query(
    """
    WITH customer_summary AS (
        SELECT
            Customer_ID,
            Customer_Name,
            Segment,
            SUM(Sales) AS Total_Sales,
            SUM(Profit) AS Total_Profit,
            SUM(Quantity) AS Total_Quantity,
            COUNT(DISTINCT Order_ID) AS Total_Orders
        FROM superstore
        GROUP BY
            Customer_ID,
            Customer_Name,
            Segment
    )

    SELECT
        Customer_ID,
        Customer_Name,
        Segment,
        Total_Sales,
        Total_Profit,
        Total_Quantity,
        Total_Orders
    FROM customer_summary
    ORDER BY Total_Sales DESC
    LIMIT 20;
    """,
    conn
)

display(customer_sales_cte)
display(customer_sales_cte.describe().T)

,Customer_ID,Customer_Name,Segment,Total_Sales,Total_Profit,Total_Quantity,Total_Orders
0,CJ-12010,Caroline Jumper,Consumer,11596.9740,858.7414,83,8
1,KF-16285,Karen Ferguson,Home Office,10941.2740,1577.6134,61,6
2,SV-20365,Seth Vernon,Consumer,10751.1480,1610.0020,88,7
3,HW-14935,Helen Wasserman,Corporate,10074.9340,2146.9635,76,7
4,EH-13765,Edward Hooks,Corporate,9542.9880,1164.6312,89,9
5,BS-11365,Bill Shonely,Corporate,9199.7800,2405.3645,18,2
6,PK-19075,Pete Kriz,Consumer,8812.0540,2024.9500,101,11
7,JL-15835,John Lee,Consumer,8765.3320,839.9550,94,7
8,AB-10060,Adam Bellavance,Home Office,8167.0800,2045.8747,54,7
9,JW-15220,Jane Waco,Corporate,7933.5540,2084.5100,44,5


,count,mean,std,min,25%,50%,75%,max
Total_Sales,20.0,8515.82688,1396.494327,6952.2930,7470.9537,7877.76600,9285.582000,11596.9740
Total_Profit,20.0,1185.26413,970.176052,-1151.9078,765.9427,1244.54625,2030.181175,2436.9525
Total_Quantity,20.0,66.75000,25.618404,18.0000,49.5000,73.50000,84.250000,113.0000
Total_Orders,20.0,6.85000,2.700390,2.0000,5.7500,7.00000,7.250000,14.0000


## 2. Customer Profit Margin

Business Question:

> Which customers generate high Sales while maintaining strong
> profitability?

Customer-level Sales and Profit are aggregated first, then
Profit Margin is calculated from those aggregated values.

In [79]:
# Calculate customer-level profitability using a CTE

customer_profitability_cte = pd.read_sql_query(
    """
    WITH customer_summary AS (
        SELECT
            Customer_ID,
            Customer_Name,
            SUM(Sales) AS Total_Sales,
            SUM(Profit) AS Total_Profit
        FROM superstore
        GROUP BY
            Customer_ID,
            Customer_Name
    )

    SELECT
        Customer_ID,
        Customer_Name,
        Total_Sales,
        Total_Profit,
        CASE
            WHEN Total_Sales = 0 THEN 0
            ELSE (Total_Profit * 100.0 / Total_Sales)
        END AS Profit_Margin
    FROM customer_summary
    ORDER BY Total_Profit DESC
    LIMIT 20;
    """,
    conn
)

display(customer_profitability_cte.round(2))
display(customer_profitability_cte.describe().T)

,Customer_ID,Customer_Name,Total_Sales,Total_Profit,Profit_Margin
0,TC-20980,Tamara Chand,1344.05,8764.95,652.13
1,RB-19360,Raymond Buch,5041.34,6976.10,138.38
2,HL-15040,Hunter Lopez,6091.11,5174.40,84.95
3,AB-10105,Adrian Barton,6276.91,5088.94,81.07
4,TA-21385,Tom Ashbrook,3715.77,4523.36,121.73
5,AR-10540,Andy Reiter,6353.82,2617.22,41.19
6,DR-12940,Daniel Raglin,6408.62,2549.73,39.79
7,SE-20110,Sanjit Engle,7821.98,2436.95,31.16
8,BS-11365,Bill Shonely,9199.78,2405.36,26.15
9,KD-16270,Karen Daniels,6317.16,2318.71,36.70


,count,mean,std,min,25%,50%,75%,max
Total_Sales,20.0,6176.169750,2117.621800,1344.052000,4923.907000,6297.035500,7849.872000,10074.93400
Total_Profit,20.0,3121.683840,1956.786938,1711.889300,2002.750600,2247.879600,3093.752175,8764.94830
Profit_Margin,20.0,79.049177,138.770658,21.309951,29.935048,36.238526,56.972538,652.12866


## 3. Category Sales Contribution

Business Question:

> What percentage of total business Sales comes from each
> category?

The first CTE calculates category-level Sales.

The second CTE calculates total business Sales.

The final query calculates each category's contribution.

In [80]:
# Calculate category Sales contribution using multiple CTEs

category_contribution_cte = pd.read_sql_query(
    """
    WITH category_sales AS (
        SELECT
            Category,
            SUM(Sales) AS Total_Sales
        FROM superstore
        GROUP BY Category
    ),

    overall_sales AS (
        SELECT
            SUM(Sales) AS Business_Sales
        FROM superstore
    )

    SELECT
        c.Category,
        c.Total_Sales,
        o.Business_Sales,
        (c.Total_Sales * 100.0 / o.Business_Sales)
            AS Sales_Contribution_Percent
    FROM category_sales AS c
    CROSS JOIN overall_sales AS o
    ORDER BY c.Total_Sales DESC;
    """,
    conn
)

display(category_contribution_cte.round(2))
display(category_contribution_cte.describe().T)

,Category,Total_Sales,Business_Sales,Sales_Contribution_Percent
0,Office Supplies,643707.69,1565804.32,41.11
1,Technology,470587.99,1565804.32,30.05
2,Furniture,451508.65,1565804.32,28.84


,count,mean,std,min,25%,50%,75%,max
Total_Sales,3.0,5.219348e+05,1.058890e+05,4.515086e+05,4.610483e+05,4.705880e+05,5.571478e+05,6.437077e+05
Business_Sales,3.0,1.565804e+06,2.851581e-10,1.565804e+06,1.565804e+06,1.565804e+06,1.565804e+06,1.565804e+06
Sales_Contribution_Percent,3.0,3.333333e+01,6.762597e+00,2.883557e+01,2.944482e+01,3.005407e+01,3.558221e+01,4.111035e+01


## 4. Regional Sales Benchmark

Business Question:

> Which regions generate above-average Sales compared with the
> average regional Sales?

The first CTE calculates regional Sales.

The second CTE calculates the average regional Sales.

The final query compares every region against that benchmark.

In [81]:
# Compare regional Sales against average regional performance

region_benchmark_cte = pd.read_sql_query(
    """
    WITH region_sales AS (
        SELECT
            Region,
            SUM(Sales) AS Total_Sales
        FROM superstore
        GROUP BY Region
    ),

    regional_average AS (
        SELECT
            AVG(Total_Sales) AS Average_Region_Sales
        FROM region_sales
    )

    SELECT
        r.Region,
        r.Total_Sales,
        a.Average_Region_Sales,

        CASE
            WHEN r.Total_Sales > a.Average_Region_Sales
                THEN 'Above Average'
            ELSE 'Below Average'
        END AS Performance_Status

    FROM region_sales AS r
    CROSS JOIN regional_average AS a
    ORDER BY r.Total_Sales DESC;
    """,
    conn
)

display(region_benchmark_cte.round(2))
display(region_benchmark_cte.describe().T)

,Region,Total_Sales,Average_Region_Sales,Performance_Status
0,West,522441.05,391451.08,Above Average
1,East,450234.67,391451.08,Above Average
2,Central,341007.52,391451.08,Below Average
3,South,252121.08,391451.08,Below Average


,count,mean,std,min,25%,50%,75%,max
Total_Sales,4.0,391451.0808,119123.582455,252121.0810,318785.9134,395621.0951,468286.2625,522441.0520
Average_Region_Sales,4.0,391451.0808,0.000000,391451.0808,391451.0808,391451.0808,391451.0808,391451.0808


## 5. Product Sales & Profitability

Business Question:

> Which products combine strong Sales with strong Profit?

Products are first aggregated and then classified according to
their Sales and Profit performance.

In [82]:
# Create product-level performance classification

product_health_cte = pd.read_sql_query(
    """
    WITH product_summary AS (
        SELECT
            Product_ID,
            Product_Name,
            Category,
            "Sub-Category",
            SUM(Sales) AS Total_Sales,
            SUM(Profit) AS Total_Profit
        FROM superstore
        GROUP BY
            Product_ID,
            Product_Name,
            Category,
            "Sub-Category"
    )

    SELECT
        Product_ID,
        Product_Name,
        Category,
        "Sub-Category" AS Sub_Category,
        Total_Sales,
        Total_Profit,

        CASE
            WHEN Total_Sales > 1000
                 AND Total_Profit > 0
                THEN 'Strong Performer'

            WHEN Total_Sales > 1000
                 AND Total_Profit <= 0
                THEN 'High Sales but Loss'

            WHEN Total_Sales <= 1000
                 AND Total_Profit > 0
                THEN 'Low Sales but Profitable'

            ELSE 'Weak Performer'
        END AS Business_Status

    FROM product_summary

    ORDER BY Total_Sales DESC
    LIMIT 30;
    """,
    conn
)

display(product_health_cte.round(2))
display(product_health_cte.describe().T)

,Product_ID,Product_Name,Category,Sub_Category,Total_Sales,Total_Profit,Business_Status
0,TEC-MA-10001047,"3D Systems Cube Printer, 2nd Generation, Magenta",Technology,Machines,14334.89,3717.97,Strong Performer
1,TEC-CO-10004722,Canon imageCLASS 2200 Advanced Copier,Technology,Copiers,14076.82,25199.93,Strong Performer
2,TEC-CO-10001449,Hewlett Packard LaserJet 3310 Copier,Technology,Copiers,13837.73,6407.89,Strong Performer
3,OFF-BI-10001359,GBC DocuBind TL300 Electric Binding System,Office Supplies,Binders,12890.26,2753.76,Strong Performer
4,OFF-BI-10004995,GBC DocuBind P400 Electric Binding System,Office Supplies,Binders,12577.11,762.15,Strong Performer
5,TEC-PH-10001459,Samsung Galaxy Mega 6.3,Technology,Phones,12370.71,1696.76,Strong Performer
6,OFF-SU-10002881,Martin Yale Chadless Opener Electric Letter Op...,Office Supplies,Supplies,12268.90,-1232.56,High Sales but Loss
7,FUR-CH-10002024,HON 5400 Series Task Chairs for Big and Tall,Furniture,Chairs,11887.56,70.10,Strong Performer
8,FUR-CH-10001215,Global Troy Executive Leather Low-Back Tilter,Furniture,Chairs,10217.89,776.52,Strong Performer
9,OFF-BI-10003527,Fellowes PB500 Electric Punch Plastic Comb Bin...,Office Supplies,Binders,9756.52,-508.40,High Sales but Loss


,count,mean,std,min,25%,50%,75%,max
Total_Sales,30.0,8904.964217,2813.445906,5694.9200,6615.535625,7784.6970,11470.14500,14334.890
Total_Profit,30.0,1628.632950,4887.127446,-6239.9792,-32.595900,949.3245,1745.54575,25199.928


## 6. Customer Value Segmentation

Business Question:

> How many customers belong to Low, Medium, High, and Very High
> value segments?

Customers are first aggregated by Sales and then classified.

In [83]:
# Segment customers according to their aggregated Sales

customer_segments_cte = pd.read_sql_query(
    """
    WITH customer_sales AS (
        SELECT
            Customer_ID,
            Customer_Name,
            SUM(Sales) AS Total_Sales,
            SUM(Profit) AS Total_Profit
        FROM superstore
        GROUP BY
            Customer_ID,
            Customer_Name
    )

    SELECT
        CASE
            WHEN Total_Sales < 500 THEN 'Low Value'
            WHEN Total_Sales < 1500 THEN 'Medium Value'
            WHEN Total_Sales < 3000 THEN 'High Value'
            ELSE 'Very High Value'
        END AS Customer_Segment,

        COUNT(*) AS Total_Customers,
        SUM(Total_Sales) AS Segment_Sales,
        SUM(Total_Profit) AS Segment_Profit,
        AVG(Total_Sales) AS Average_Customer_Sales

    FROM customer_sales

    GROUP BY Customer_Segment

    ORDER BY Segment_Sales DESC;
    """,
    conn
)

display(customer_segments_cte.round(2))
display(customer_segments_cte.describe().T)

,Customer_Segment,Total_Customers,Segment_Sales,Segment_Profit,Average_Customer_Sales
0,Very High Value,170,793630.31,98797.74,4668.41
1,High Value,233,501047.77,43574.02,2150.42
2,Medium Value,230,234963.89,28657.36,1021.58
3,Low Value,140,36162.35,4232.98,258.30


,count,mean,std,min,25%,50%,75%,max
Total_Customers,4.0,193.250000,45.85030,140.000000,162.500000,200.000000,230.750000,233.000000
Segment_Sales,4.0,391451.080800,328875.81082,36162.348200,185263.501625,368005.830300,574193.409475,793630.314400
Segment_Profit,4.0,43815.526475,40081.78784,4232.984600,22551.268475,36115.693000,57379.951000,98797.735300
Average_Customer_Sales,4.0,2024.679462,1926.26000,258.302487,830.762207,1586.000873,2779.918128,4668.413614


## 7. Loss-Making Products

Business Question:

> Which products generate negative total Profit?

The CTE first calculates product-level Profit and then filters
products with negative aggregated Profit.

In [85]:
# Identify products with negative aggregated Profit

loss_products_cte = pd.read_sql_query(
    """
    WITH product_profit AS (
        SELECT
            Product_ID,
            Product_Name,
            Category,
            "Sub-Category",
            SUM(Sales) AS Total_Sales,
            SUM(Profit) AS Total_Profit
        FROM superstore
        GROUP BY
            Product_ID,
            Product_Name,
            Category,
            "Sub-Category"
    )

    SELECT
        Product_ID,
        Product_Name,
        Category,
        "Sub-Category" AS Sub_Category,
        Total_Sales,
        Total_Profit
    FROM product_profit
    WHERE Total_Profit < 0
    ORDER BY Total_Profit ASC
    LIMIT 20;
    """,
    conn
)

display(loss_products_cte.round(2))
display(loss_products_cte.describe().T)

,Product_ID,Product_Name,Category,Sub_Category,Total_Sales,Total_Profit
0,TEC-MA-10000418,Cubify CubeX 3D Printer Double Head Print,Technology,Machines,9323.97,-6239.98
1,TEC-MA-10004125,Cubify CubeX 3D Printer Triple Head Print,Technology,Machines,3009.98,-3839.99
2,TEC-MA-10000822,Lexmark MX611dhe Monochrome Laser Printer,Technology,Machines,5676.97,-2719.98
3,FUR-TA-10001889,Bush Advantage Collection Racetrack Conference...,Furniture,Tables,4334.10,-2019.24
4,OFF-BI-10001120,Ibico EPK-21 Electric Binding System,Office Supplies,Binders,6437.97,-1285.19
5,OFF-SU-10002881,Martin Yale Chadless Opener Electric Letter Op...,Office Supplies,Supplies,12268.90,-1232.56
6,FUR-TA-10004289,BoxOffice By Design Rectangular and Half-Moon ...,Furniture,Tables,1052.50,-986.56
7,FUR-TA-10004256,Bretford “Just In Time” Height-Adjustable Mult...,Furniture,Tables,3869.90,-964.19
8,TEC-MA-10001695,Zebra GK420t Direct Thermal/Thermal Transfer P...,Technology,Machines,703.71,-938.28
9,TEC-MA-10002210,Epson TM-T88V Direct Thermal Printer - Monochr...,Technology,Machines,693.00,-935.96


,count,mean,std,min,25%,50%,75%,max
Total_Sales,20.0,4490.898965,3384.208922,416.1700,2048.813875,3439.94000,5867.216750,12268.902
Total_Profit,20.0,-1384.846340,1415.683766,-6239.9792,-1245.717400,-916.87575,-612.912075,-508.396


## 8. Regional Profitability

Business Question:

> Which regions generate strong Sales but weak Profit?

This analysis combines Sales, Profit, and Profit Margin at the
regional level.

In [86]:
# Calculate regional Sales, Profit and Profit Margin

regional_profitability_cte = pd.read_sql_query(
    """
    WITH region_summary AS (
        SELECT
            Region,
            SUM(Sales) AS Total_Sales,
            SUM(Profit) AS Total_Profit
        FROM superstore
        GROUP BY Region
    )

    SELECT
        Region,
        Total_Sales,
        Total_Profit,

        CASE
            WHEN Total_Sales = 0 THEN 0
            ELSE Total_Profit * 100.0 / Total_Sales
        END AS Profit_Margin

    FROM region_summary

    ORDER BY Total_Profit DESC;
    """,
    conn
)

display(regional_profitability_cte.round(2))
display(regional_profitability_cte.describe().T)

,Region,Total_Sales,Total_Profit,Profit_Margin
0,West,522441.05,67859.96,12.99
1,East,450234.67,53400.42,11.86
2,Central,341007.52,27450.01,8.05
3,South,252121.08,26551.72,10.53


,count,mean,std,min,25%,50%,75%,max
Total_Sales,4.0,391451.080800,119123.582455,252121.081000,318785.913400,395621.095100,468286.262500,522441.052000
Total_Profit,4.0,43815.526475,20296.751213,26551.716300,27225.434400,40425.215700,57015.307775,67859.958200
Profit_Margin,4.0,10.857652,2.124443,8.049678,9.910921,11.195956,12.142686,12.989017


## 9. Category × Region Performance

Business Question:

> Which Category-Region combinations generate the strongest
> Sales and Profit?

This is a multi-dimensional business analysis using a CTE.

In [87]:
# Analyze Sales and Profit across Category and Region

category_region_cte = pd.read_sql_query(
    """
    WITH category_region AS (
        SELECT
            Category,
            Region,
            SUM(Sales) AS Total_Sales,
            SUM(Profit) AS Total_Profit,
            SUM(Quantity) AS Total_Quantity
        FROM superstore
        GROUP BY
            Category,
            Region
    )

    SELECT
        Category,
        Region,
        Total_Sales,
        Total_Profit,
        Total_Quantity,

        CASE
            WHEN Total_Sales = 0 THEN 0
            ELSE Total_Profit * 100.0 / Total_Sales
        END AS Profit_Margin

    FROM category_region

    ORDER BY Total_Sales DESC;
    """,
    conn
)

display(category_region_cte.round(2))
display(category_region_cte.describe().T)

,Category,Region,Total_Sales,Total_Profit,Total_Quantity,Profit_Margin
0,Office Supplies,West,204834.37,33966.03,4384,16.58
1,Office Supplies,East,189066.82,23248.61,3774,12.30
2,Technology,West,161660.33,28536.05,1356,17.65
3,Furniture,West,155946.35,5357.88,1558,3.44
4,Office Supplies,Central,150154.35,6477.63,3277,4.31
5,Technology,East,144095.87,26530.31,1187,18.41
6,Furniture,East,117071.97,3621.50,1290,3.09
7,Furniture,Central,105505.45,-1534.79,1057,-1.45
8,Office Supplies,South,99652.14,11104.98,2190,11.14
9,Technology,Central,85347.72,22507.17,905,26.37


,count,mean,std,min,25%,50%,75%,max
Total_Sales,12.0,130483.693600,43538.140042,72984.8670,96076.034500,130583.922500,157374.845750,204834.373000
Total_Profit,12.0,14605.175492,11836.880992,-1534.7880,4923.782225,11994.848000,24069.035300,33966.030100
Total_Quantity,12.0,1859.750000,1267.481693,613.0000,1019.000000,1323.000000,2461.750000,4384.000000
Profit_Margin,12.0,10.963849,8.372553,-1.4547,3.491688,11.720123,16.849609,26.371139


## 10. Monthly Sales Performance

Business Question:

> Which months generate the highest Sales and Profit?

The CTE aggregates transaction data by year and month and calculates
monthly business performance.

In [88]:
# Analyze monthly Sales and Profit using a CTE

monthly_performance_cte = pd.read_sql_query(
    """
    WITH monthly_summary AS (
        SELECT
            Order_Year,
            Order_Month,
            Order_Month_Name,
            SUM(Sales) AS Total_Sales,
            SUM(Profit) AS Total_Profit,
            SUM(Quantity) AS Total_Quantity,
            COUNT(DISTINCT Order_ID) AS Total_Orders
        FROM superstore
        GROUP BY
            Order_Year,
            Order_Month,
            Order_Month_Name
    )

    SELECT
        Order_Year,
        Order_Month,
        Order_Month_Name,
        Total_Sales,
        Total_Profit,
        Total_Quantity,
        Total_Orders,

        CASE
            WHEN Total_Sales = 0 THEN 0
            ELSE Total_Profit * 100.0 / Total_Sales
        END AS Profit_Margin

    FROM monthly_summary

    ORDER BY
        Order_Year,
        Order_Month;
    """,
    conn
)

display(monthly_performance_cte.round(2))
display(monthly_performance_cte.describe().T)

,Order_Year,Order_Month,Order_Month_Name,Total_Sales,Total_Profit,Total_Quantity,Total_Orders,Profit_Margin
0,NaN,NaN,NaN,944552.25,105137.52,13198,1791,11.13
1,2019.0,1.0,January,26416.48,7085.72,336,47,26.82
2,2019.0,2.0,February,19598.74,13891.17,324,37,70.88
3,2019.0,3.0,March,26053.38,1519.30,407,47,5.83
4,2019.0,4.0,April,22296.39,5031.83,324,46,22.57
5,2019.0,5.0,May,21777.46,3226.08,446,57,14.81
6,2019.0,6.0,June,15617.40,1431.60,287,32,9.17
7,2019.0,7.0,July,16571.28,3784.66,227,38,22.84
8,2019.0,8.0,August,23156.32,-328.34,402,53,-1.42
9,2019.0,9.0,September,9789.66,1522.42,231,38,15.55


,count,mean,std,min,25%,50%,75%,max
Order_Year,24.0,2019.500000,0.510754,2019.000000,2019.000000,2019.500000,2020.00000,2020.000000
Order_Month,24.0,6.500000,3.526299,1.000000,3.750000,6.500000,9.25000,12.000000
Total_Sales,25.0,62632.172928,183926.268903,9789.662000,21472.218500,23892.002000,32039.47950,944552.246000
Total_Profit,25.0,7010.484236,20676.185719,-3331.957800,1519.304300,2436.668400,3784.65640,105137.524300
Total_Quantity,25.0,892.680000,2565.217595,227.000000,327.000000,374.000000,421.00000,13198.000000
Total_Orders,25.0,120.160000,348.262195,32.000000,45.000000,51.000000,57.00000,1791.000000
Profit_Margin,25.0,12.342685,14.782047,-14.007022,6.358696,10.319549,14.81386,70.877913


# 🔵 Stage 7 — Window Functions & Advanced Business Analytics

Window Functions allow calculations across related rows without
collapsing the original result into a single row.

Unlike GROUP BY, Window Functions preserve row-level or
group-level detail while calculating rankings, totals, averages,
comparisons, and sequential metrics.

---

## 🎯 Business Objectives

This stage answers advanced business questions such as:

- Which are the top customers by Sales?
- Which are the top products within each category?
- Which categories rank highest by Profit?
- What percentage of total Sales does each category contribute?
- What is the cumulative Sales over time?
- How does current Sales compare with previous-period Sales?
- Which customers have the highest order value?
- Which products rank within their category?
- Which regions outperform other regions?
- How does each transaction compare with its category average?

---

## 🧠 Window Functions Covered

### Ranking

```text
ROW_NUMBER()
RANK()
DENSE_RANK()


---

# 1️⃣ Customer Sales Ranking

Business Question:

> Which customers generate the highest Sales?

Customers are aggregated first and then ranked using
DENSE_RANK().

DENSE_RANK() allows customers with equal Sales to receive the
same rank.

In [89]:
# Rank customers according to their total Sales

customer_sales_rank = pd.read_sql_query(
    """
    WITH customer_sales AS (
        SELECT
            Customer_ID,
            Customer_Name,
            Segment,
            SUM(Sales) AS Total_Sales,
            SUM(Profit) AS Total_Profit
        FROM superstore
        GROUP BY
            Customer_ID,
            Customer_Name,
            Segment
    )

    SELECT
        Customer_ID,
        Customer_Name,
        Segment,
        Total_Sales,
        Total_Profit,

        DENSE_RANK() OVER (
            ORDER BY Total_Sales DESC
        ) AS Sales_Rank

    FROM customer_sales

    ORDER BY Sales_Rank
    LIMIT 20;
    """,
    conn
)

display(customer_sales_rank.round(2))
display(customer_sales_rank.describe().T)

,Customer_ID,Customer_Name,Segment,Total_Sales,Total_Profit,Sales_Rank
0,CJ-12010,Caroline Jumper,Consumer,11596.97,858.74,1
1,KF-16285,Karen Ferguson,Home Office,10941.27,1577.61,2
2,SV-20365,Seth Vernon,Consumer,10751.15,1610.00,3
3,HW-14935,Helen Wasserman,Corporate,10074.93,2146.96,4
4,EH-13765,Edward Hooks,Corporate,9542.99,1164.63,5
5,BS-11365,Bill Shonely,Corporate,9199.78,2405.36,6
6,PK-19075,Pete Kriz,Consumer,8812.05,2024.95,7
7,JL-15835,John Lee,Consumer,8765.33,839.96,8
8,AB-10060,Adam Bellavance,Home Office,8167.08,2045.87,9
9,JW-15220,Jane Waco,Corporate,7933.55,2084.51,10


,count,mean,std,min,25%,50%,75%,max
Total_Sales,20.0,8515.82688,1396.494327,6952.2930,7470.9537,7877.76600,9285.582000,11596.9740
Total_Profit,20.0,1185.26413,970.176052,-1151.9078,765.9427,1244.54625,2030.181175,2436.9525
Sales_Rank,20.0,10.50000,5.916080,1.0000,5.7500,10.50000,15.250000,20.0000


## 2. Top Products Within Each Category

Business Question:

> Which products are the top performers inside their respective
> categories?

The ranking restarts for every Category using:

PARTITION BY Category

In [90]:
# Rank products independently within each Category

product_category_rank = pd.read_sql_query(
    """
    WITH product_sales AS (
        SELECT
            Product_ID,
            Product_Name,
            Category,
            "Sub-Category",
            SUM(Sales) AS Total_Sales,
            SUM(Profit) AS Total_Profit
        FROM superstore
        GROUP BY
            Product_ID,
            Product_Name,
            Category,
            "Sub-Category"
    )

    SELECT
        Product_ID,
        Product_Name,
        Category,
        "Sub-Category" AS Sub_Category,
        Total_Sales,
        Total_Profit,

        DENSE_RANK() OVER (
            PARTITION BY Category
            ORDER BY Total_Sales DESC
        ) AS Category_Product_Rank

    FROM product_sales

    ORDER BY
        Category,
        Category_Product_Rank;
    """,
    conn
)

display(product_category_rank.head(30).round(2))
display(product_category_rank.describe().T)

,Product_ID,Product_Name,Category,Sub_Category,Total_Sales,Total_Profit,Category_Product_Rank
0,FUR-CH-10002024,HON 5400 Series Task Chairs for Big and Tall,Furniture,Chairs,11887.56,70.10,1
1,FUR-CH-10001215,Global Troy Executive Leather Low-Back Tilter,Furniture,Chairs,10217.89,776.52,2
2,FUR-TA-10001095,Chromcraft Round Conference Tables,Furniture,Tables,8080.05,-158.60,3
3,FUR-CH-10000454,"Hon Deluxe Fabric Upholstered Stacking Chairs,...",Furniture,Chairs,7304.13,1732.26,4
4,FUR-TA-10003473,Bretford Rectangular Conference Table Tops,Furniture,Tables,6603.53,90.27,5
5,FUR-CH-10004086,Hon 4070 Series Pagoda Armless Upholstered Sta...,Furniture,Chairs,5799.08,1190.26,6
6,FUR-TA-10004915,"Office Impressions End Table, 20-1/2""H x 24""W ...",Furniture,Tables,5705.82,-66.83,7
7,FUR-CH-10002331,Hon 4700 Series Mobuis Mid-Back Task Chairs wi...,Furniture,Chairs,5575.43,608.73,8
8,FUR-TA-10001950,Balt Solid Wood Round Tables,Furniture,Tables,5534.18,-647.41,9
9,FUR-TA-10000198,Chromcraft Bull-Nose Wood Oval Conference Tabl...,Furniture,Tables,5322.41,-870.55,10


,count,mean,std,min,25%,50%,75%,max
Total_Sales,1783.0,878.185263,1430.452031,2.6100,211.5200,428.440,873.7210,14334.890
Total_Profit,1783.0,98.296190,709.745443,-6239.9792,5.0886,26.235,82.7292,25199.928
Category_Product_Rank,1783.0,384.309591,293.738274,1.0000,149.0000,298.000,604.5000,1045.000


## 3. Top 3 Products Per Category

Business Question:

> What are the three highest-selling products in each category?

This is a classic business ranking problem solved using a window
function and CTE.

In [91]:
# Identify the top 3 products within every Category

top_3_products_category = pd.read_sql_query(
    """
    WITH product_sales AS (
        SELECT
            Product_ID,
            Product_Name,
            Category,
            SUM(Sales) AS Total_Sales,
            SUM(Profit) AS Total_Profit
        FROM superstore
        GROUP BY
            Product_ID,
            Product_Name,
            Category
    ),

    ranked_products AS (
        SELECT
            Product_ID,
            Product_Name,
            Category,
            Total_Sales,
            Total_Profit,

            DENSE_RANK() OVER (
                PARTITION BY Category
                ORDER BY Total_Sales DESC
            ) AS Product_Rank

        FROM product_sales
    )

    SELECT *
    FROM ranked_products
    WHERE Product_Rank <= 3
    ORDER BY
        Category,
        Product_Rank;
    """,
    conn
)

display(top_3_products_category.round(2))
display(top_3_products_category.describe().T)

,Product_ID,Product_Name,Category,Total_Sales,Total_Profit,Product_Rank
0,FUR-CH-10002024,HON 5400 Series Task Chairs for Big and Tall,Furniture,11887.56,70.10,1
1,FUR-CH-10001215,Global Troy Executive Leather Low-Back Tilter,Furniture,10217.89,776.52,2
2,FUR-TA-10001095,Chromcraft Round Conference Tables,Furniture,8080.05,-158.60,3
3,OFF-BI-10001359,GBC DocuBind TL300 Electric Binding System,Office Supplies,12890.26,2753.76,1
4,OFF-BI-10004995,GBC DocuBind P400 Electric Binding System,Office Supplies,12577.11,762.15,2
5,OFF-SU-10002881,Martin Yale Chadless Opener Electric Letter Op...,Office Supplies,12268.90,-1232.56,3
6,TEC-MA-10001047,"3D Systems Cube Printer, 2nd Generation, Magenta",Technology,14334.89,3717.97,1
7,TEC-CO-10004722,Canon imageCLASS 2200 Advanced Copier,Technology,14076.82,25199.93,2
8,TEC-CO-10001449,Hewlett Packard LaserJet 3310 Copier,Technology,13837.73,6407.89,3


,count,mean,std,min,25%,50%,75%,max
Total_Sales,9.0,12241.247000,2011.554651,8080.0530,11887.562,12577.108,13837.7320,14334.890
Total_Profit,9.0,4255.240067,8195.070828,-1232.5588,70.098,776.519,3717.9714,25199.928
Product_Rank,9.0,2.000000,0.866025,1.0000,1.000,2.000,3.0000,3.000


## 4. Category Profit Ranking

Business Question:

> Which categories generate the highest total Profit?

Categories are ranked by aggregated Profit.

In [92]:
# Rank Categories according to Total Profit

category_profit_rank = pd.read_sql_query(
    """
    WITH category_profit AS (
        SELECT
            Category,
            SUM(Sales) AS Total_Sales,
            SUM(Profit) AS Total_Profit
        FROM superstore
        GROUP BY Category
    )

    SELECT
        Category,
        Total_Sales,
        Total_Profit,

        RANK() OVER (
            ORDER BY Total_Profit DESC
        ) AS Profit_Rank

    FROM category_profit

    ORDER BY Profit_Rank;
    """,
    conn
)

display(category_profit_rank.round(2))
display(category_profit_rank.describe().T)

,Category,Total_Sales,Total_Profit,Profit_Rank
0,Technology,470587.99,90458.25,1
1,Office Supplies,643707.69,74797.25,2
2,Furniture,451508.65,10006.61,3


,count,mean,std,min,25%,50%,75%,max
Total_Sales,3.0,521934.774400,105889.031733,451508.6452,461048.31810,470587.9910,557147.83900,643707.6870
Total_Profit,3.0,58420.701967,42652.782892,10006.6112,42401.92865,74797.2461,82627.74735,90458.2486
Profit_Rank,3.0,2.000000,1.000000,1.0000,1.50000,2.0000,2.50000,3.0000


## 5. Sales Contribution Percentage

Business Question:

> What percentage of total business Sales is generated by each
> category?

SUM() OVER() calculates the overall Sales total while preserving
each category row.

In [93]:
# Calculate Category contribution to total business Sales

category_sales_contribution = pd.read_sql_query(
    """
    WITH category_sales AS (
        SELECT
            Category,
            SUM(Sales) AS Total_Sales
        FROM superstore
        GROUP BY Category
    )

    SELECT
        Category,
        Total_Sales,

        SUM(Total_Sales) OVER () AS Overall_Sales,

        (
            Total_Sales * 100.0
            / SUM(Total_Sales) OVER ()
        ) AS Sales_Contribution_Percent

    FROM category_sales

    ORDER BY Total_Sales DESC;
    """,
    conn
)

display(category_sales_contribution.round(2))
display(category_sales_contribution.describe().T)

,Category,Total_Sales,Overall_Sales,Sales_Contribution_Percent
0,Office Supplies,643707.69,1565804.32,41.11
1,Technology,470587.99,1565804.32,30.05
2,Furniture,451508.65,1565804.32,28.84


,count,mean,std,min,25%,50%,75%,max
Total_Sales,3.0,5.219348e+05,1.058890e+05,4.515086e+05,4.610483e+05,4.705880e+05,5.571478e+05,6.437077e+05
Overall_Sales,3.0,1.565804e+06,2.851581e-10,1.565804e+06,1.565804e+06,1.565804e+06,1.565804e+06,1.565804e+06
Sales_Contribution_Percent,3.0,3.333333e+01,6.762597e+00,2.883557e+01,2.944482e+01,3.005407e+01,3.558221e+01,4.111035e+01


## 6. Running Sales Over Time

Business Question:

> How does cumulative Sales develop over time?

The running total helps understand how Sales accumulate across
the business timeline.

In [94]:
# Calculate cumulative Sales over time

running_sales = pd.read_sql_query(
    """
    WITH monthly_sales AS (
        SELECT
            Order_Year,
            Order_Month,
            Order_Month_Name,
            SUM(Sales) AS Total_Sales
        FROM superstore
        GROUP BY
            Order_Year,
            Order_Month,
            Order_Month_Name
    )

    SELECT
        Order_Year,
        Order_Month,
        Order_Month_Name,
        Total_Sales,

        SUM(Total_Sales) OVER (
            ORDER BY
                Order_Year,
                Order_Month
            ROWS BETWEEN UNBOUNDED PRECEDING
            AND CURRENT ROW
        ) AS Running_Sales

    FROM monthly_sales

    ORDER BY
        Order_Year,
        Order_Month;
    """,
    conn
)

display(running_sales.round(2))
display(running_sales.describe().T)

,Order_Year,Order_Month,Order_Month_Name,Total_Sales,Running_Sales
0,NaN,NaN,NaN,944552.25,944552.25
1,2019.0,1.0,January,26416.48,970968.73
2,2019.0,2.0,February,19598.74,990567.46
3,2019.0,3.0,March,26053.38,1016620.84
4,2019.0,4.0,April,22296.39,1038917.23
5,2019.0,5.0,May,21777.46,1060694.69
6,2019.0,6.0,June,15617.40,1076312.09
7,2019.0,7.0,July,16571.28,1092883.37
8,2019.0,8.0,August,23156.32,1116039.70
9,2019.0,9.0,September,9789.66,1125829.36


,count,mean,std,min,25%,50%,75%,max
Order_Year,24.0,2.019500e+03,0.510754,2019.000,2.019000e+03,2.019500e+03,2.020000e+03,2.020000e+03
Order_Month,24.0,6.500000e+00,3.526299,1.000,3.750000e+00,6.500000e+00,9.250000e+00,1.200000e+01
Total_Sales,25.0,6.263217e+04,183926.268903,9789.662,2.147222e+04,2.389200e+04,3.203948e+04,9.445522e+05
Running_Sales,25.0,1.232928e+06,192920.964077,944552.246,1.076312e+06,1.190416e+06,1.384786e+06,1.565804e+06


## 7. Previous Month Sales

Business Question:

> How does each month's Sales compare with the previous month?

LAG() retrieves the previous row's Sales value.

In [95]:
# Compare monthly Sales with the previous month

monthly_sales_lag = pd.read_sql_query(
    """
    WITH monthly_sales AS (
        SELECT
            Order_Year,
            Order_Month,
            Order_Month_Name,
            SUM(Sales) AS Total_Sales
        FROM superstore
        GROUP BY
            Order_Year,
            Order_Month,
            Order_Month_Name
    )

    SELECT
        Order_Year,
        Order_Month,
        Order_Month_Name,
        Total_Sales,

        LAG(Total_Sales) OVER (
            ORDER BY
                Order_Year,
                Order_Month
        ) AS Previous_Month_Sales

    FROM monthly_sales

    ORDER BY
        Order_Year,
        Order_Month;
    """,
    conn
)

monthly_sales_lag["Sales_Change"] = (
    monthly_sales_lag["Total_Sales"]
    - monthly_sales_lag["Previous_Month_Sales"]
)

monthly_sales_lag["Sales_Growth_%"] = (
    monthly_sales_lag["Sales_Change"]
    / monthly_sales_lag["Previous_Month_Sales"]
    * 100
)

display(monthly_sales_lag.round(2))
display(monthly_sales_lag.describe().T)

,Order_Year,Order_Month,Order_Month_Name,Total_Sales,Previous_Month_Sales,Sales_Change,Sales_Growth_%
0,NaN,NaN,NaN,944552.25,NaN,NaN,NaN
1,2019.0,1.0,January,26416.48,944552.25,-918135.76,-97.20
2,2019.0,2.0,February,19598.74,26416.48,-6817.75,-25.81
3,2019.0,3.0,March,26053.38,19598.74,6454.64,32.93
4,2019.0,4.0,April,22296.39,26053.38,-3756.98,-14.42
5,2019.0,5.0,May,21777.46,22296.39,-518.94,-2.33
6,2019.0,6.0,June,15617.40,21777.46,-6160.05,-28.29
7,2019.0,7.0,July,16571.28,15617.40,953.87,6.11
8,2019.0,8.0,August,23156.32,16571.28,6585.04,39.74
9,2019.0,9.0,September,9789.66,23156.32,-13366.66,-57.72


,count,mean,std,min,25%,50%,75%,max
Order_Year,24.0,2019.500000,0.510754,2019.00000,2019.000000,2019.500000,2020.000000,2020.00000
Order_Month,24.0,6.500000,3.526299,1.00000,3.750000,6.500000,9.250000,12.00000
Total_Sales,25.0,62632.172928,183926.268903,9789.66200,21472.218500,23892.002000,32039.479500,944552.24600
Previous_Month_Sales,24.0,64246.346717,187701.150446,9789.66200,21223.615875,23844.962000,32170.409625,944552.24600
Sales_Change,24.0,-38360.843500,187574.533637,-918135.76500,-6324.476850,-620.862350,4648.068500,18014.69350
Sales_Growth_%,24.0,2.288032,46.824931,-97.20328,-22.675188,-2.291687,16.011691,131.23019


## 8. Next Month Sales

Business Question:

> What was the Sales value of the following period?

LEAD() provides access to the next row without requiring a
self-join.

In [96]:
# Retrieve the next month's Sales

monthly_sales_lead = pd.read_sql_query(
    """
    WITH monthly_sales AS (
        SELECT
            Order_Year,
            Order_Month,
            Order_Month_Name,
            SUM(Sales) AS Total_Sales
        FROM superstore
        GROUP BY
            Order_Year,
            Order_Month,
            Order_Month_Name
    )

    SELECT
        Order_Year,
        Order_Month,
        Order_Month_Name,
        Total_Sales,

        LEAD(Total_Sales) OVER (
            ORDER BY
                Order_Year,
                Order_Month
        ) AS Next_Month_Sales

    FROM monthly_sales

    ORDER BY
        Order_Year,
        Order_Month;
    """,
    conn
)

display(monthly_sales_lead.round(2))
display(monthly_sales_lead.describe().T)

,Order_Year,Order_Month,Order_Month_Name,Total_Sales,Next_Month_Sales
0,NaN,NaN,NaN,944552.25,26416.48
1,2019.0,1.0,January,26416.48,19598.74
2,2019.0,2.0,February,19598.74,26053.38
3,2019.0,3.0,March,26053.38,22296.39
4,2019.0,4.0,April,22296.39,21777.46
5,2019.0,5.0,May,21777.46,15617.40
6,2019.0,6.0,June,15617.40,16571.28
7,2019.0,7.0,July,16571.28,23156.32
8,2019.0,8.0,August,23156.32,9789.66
9,2019.0,9.0,September,9789.66,22636.65


,count,mean,std,min,25%,50%,75%,max
Order_Year,24.0,2019.500000,0.510754,2019.000,2019.000000,2019.5000,2020.000000,2020.000
Order_Month,24.0,6.500000,3.526299,1.000,3.750000,6.5000,9.250000,12.000
Total_Sales,25.0,62632.172928,183926.268903,9789.662,21472.218500,23892.0020,32039.479500,944552.246
Next_Month_Sales,24.0,25885.503217,8603.074522,9789.662,21223.615875,23839.8845,31497.387375,47970.456


## 9. Category Average Sales Benchmark

Business Question:

> How does each category's Sales compare with the average category
> Sales?

AVG() OVER() calculates the average across all category rows.

In [97]:
# Compare each Category against average Category Sales

category_average_benchmark = pd.read_sql_query(
    """
    WITH category_sales AS (
        SELECT
            Category,
            SUM(Sales) AS Total_Sales
        FROM superstore
        GROUP BY Category
    )

    SELECT
        Category,
        Total_Sales,

        AVG(Total_Sales) OVER ()
            AS Average_Category_Sales,

        Total_Sales
        - AVG(Total_Sales) OVER ()
            AS Difference_From_Average,

        CASE
            WHEN Total_Sales >
                 AVG(Total_Sales) OVER ()
                THEN 'Above Average'
            ELSE 'Below Average'
        END AS Performance_Status

    FROM category_sales

    ORDER BY Total_Sales DESC;
    """,
    conn
)

display(category_average_benchmark.round(2))
display(category_average_benchmark.describe().T)

,Category,Total_Sales,Average_Category_Sales,Difference_From_Average,Performance_Status
0,Office Supplies,643707.69,521934.77,121772.91,Above Average
1,Technology,470587.99,521934.77,-51346.78,Below Average
2,Furniture,451508.65,521934.77,-70426.13,Below Average


,count,mean,std,min,25%,50%,75%,max
Total_Sales,3.0,5.219348e+05,105889.031733,451508.6452,461048.3181,470587.9910,557147.8390,643707.6870
Average_Category_Sales,3.0,5.219348e+05,0.000000,521934.7744,521934.7744,521934.7744,521934.7744,521934.7744
Difference_From_Average,3.0,1.940255e-11,105889.031733,-70426.1292,-60886.4563,-51346.7834,35213.0646,121772.9126


## 10. Customer Ranking Within Segment

Business Question:

> Who are the highest-value customers within each customer segment?

The ranking restarts separately for every Segment.

This allows fair comparison between customers belonging to the
same business segment.

In [98]:
# Rank customers independently within each Segment

customer_segment_rank = pd.read_sql_query(
    """
    WITH customer_sales AS (
        SELECT
            Customer_ID,
            Customer_Name,
            Segment,
            SUM(Sales) AS Total_Sales,
            SUM(Profit) AS Total_Profit,
            COUNT(DISTINCT Order_ID) AS Total_Orders
        FROM superstore
        GROUP BY
            Customer_ID,
            Customer_Name,
            Segment
    ),

    ranked_customers AS (
        SELECT
            Customer_ID,
            Customer_Name,
            Segment,
            Total_Sales,
            Total_Profit,
            Total_Orders,

            DENSE_RANK() OVER (
                PARTITION BY Segment
                ORDER BY Total_Sales DESC
            ) AS Segment_Rank

        FROM customer_sales
    )

    SELECT *
    FROM ranked_customers
    WHERE Segment_Rank <= 5
    ORDER BY
        Segment,
        Segment_Rank;
    """,
    conn
)

display(customer_segment_rank.round(2))
display(customer_segment_rank.describe().T)

,Customer_ID,Customer_Name,Segment,Total_Sales,Total_Profit,Total_Orders,Segment_Rank
0,CJ-12010,Caroline Jumper,Consumer,11596.97,858.74,8,1
1,SV-20365,Seth Vernon,Consumer,10751.15,1610.00,7,2
2,PK-19075,Pete Kriz,Consumer,8812.05,2024.95,11,3
3,JL-15835,John Lee,Consumer,8765.33,839.96,7,4
4,SE-20110,Sanjit Engle,Consumer,7821.98,2436.95,7,5
5,HW-14935,Helen Wasserman,Corporate,10074.93,2146.96,7,1
6,EH-13765,Edward Hooks,Corporate,9542.99,1164.63,9,2
7,BS-11365,Bill Shonely,Corporate,9199.78,2405.36,2,3
8,JW-15220,Jane Waco,Corporate,7933.55,2084.51,5,4
9,DK-13225,Dean Katz,Corporate,7581.80,543.91,6,5


,count,mean,std,min,25%,50%,75%,max
Total_Sales,15.0,8805.043933,1544.481470,6408.6160,7817.5355,8765.332,9808.96100,11596.9740
Total_Profit,15.0,1532.381193,789.439237,170.9746,849.3482,1610.002,2115.73675,2549.7344
Total_Orders,15.0,6.866667,2.166850,2.0000,6.0000,7.000,8.00000,11.0000
Segment_Rank,15.0,3.000000,1.463850,1.0000,2.0000,3.000,4.00000,5.0000


## 11. Product Profit Ranking Within Category

Business Question:

> Which products generate the highest Profit within each category?

This identifies category-specific profit leaders rather than
only overall leaders.

In [99]:
# Rank products by Profit within each Category

product_profit_category_rank = pd.read_sql_query(
    """
    WITH product_profit AS (
        SELECT
            Product_ID,
            Product_Name,
            Category,
            "Sub-Category",
            SUM(Sales) AS Total_Sales,
            SUM(Profit) AS Total_Profit
        FROM superstore
        GROUP BY
            Product_ID,
            Product_Name,
            Category,
            "Sub-Category"
    ),

    ranked_products AS (
        SELECT
            Product_ID,
            Product_Name,
            Category,
            "Sub-Category" AS Sub_Category,
            Total_Sales,
            Total_Profit,

            DENSE_RANK() OVER (
                PARTITION BY Category
                ORDER BY Total_Profit DESC
            ) AS Profit_Rank

        FROM product_profit
    )

    SELECT *
    FROM ranked_products
    WHERE Profit_Rank <= 5
    ORDER BY
        Category,
        Profit_Rank;
    """,
    conn
)

display(product_profit_category_rank.round(2))
display(product_profit_category_rank.describe().T)

,Product_ID,Product_Name,Category,Sub_Category,Total_Sales,Total_Profit,Profit_Rank
0,FUR-CH-10000454,"Hon Deluxe Fabric Upholstered Stacking Chairs,...",Furniture,Chairs,7304.13,1732.26,1
1,FUR-CH-10004086,Hon 4070 Series Pagoda Armless Upholstered Sta...,Furniture,Chairs,5799.08,1190.26,2
2,FUR-CH-10003746,Hon 4070 Series Pagoda Round Back Stacking Chairs,Furniture,Chairs,4685.21,956.52,3
3,FUR-BO-10002545,"Atlantic Metals Mobile 3-Shelf Bookcases, Cust...",Furniture,Bookcases,3411.74,882.11,4
4,FUR-FU-10002937,"GE 48"" Fluorescent Tube, Cool White Energy Sav...",Furniture,Furnishings,1969.52,873.22,5
5,OFF-BI-10001359,GBC DocuBind TL300 Electric Binding System,Office Supplies,Binders,12890.26,2753.76,1
6,OFF-AP-10002945,Honeywell Enviracaire Portable HEPA Air Cleane...,Office Supplies,Appliances,7275.81,2396.18,2
7,OFF-BI-10000545,GBC Ibimaster 500 Manual ProClick Binding System,Office Supplies,Binders,6651.54,1826.35,3
8,OFF-BI-10003925,Fellowes PB300 Plastic Comb Binding Machine,Office Supplies,Binders,5154.87,1753.71,4
9,OFF-BI-10003650,GBC DocuBind 300 Electric Binding Machine,Office Supplies,Binders,5047.19,1641.06,5


,count,mean,std,min,25%,50%,75%,max
Total_Sales,15.0,7472.57860,4174.279551,1969.524,4824.940,5799.081,10097.19200,14334.890
Total_Profit,15.0,3768.81198,6093.135444,873.224,1415.658,1826.352,2776.87165,25199.928
Profit_Rank,15.0,3.00000,1.463850,1.000,2.000,3.000,4.00000,5.000


## 12. Regional Sales Contribution

Business Question:

> What percentage of total Sales does each region contribute?

The overall Sales total is calculated using a window function.

In [100]:
# Calculate Regional contribution to total Sales

region_contribution = pd.read_sql_query(
    """
    WITH region_sales AS (
        SELECT
            Region,
            SUM(Sales) AS Total_Sales
        FROM superstore
        GROUP BY Region
    )

    SELECT
        Region,
        Total_Sales,

        SUM(Total_Sales) OVER ()
            AS Overall_Sales,

        (
            Total_Sales * 100.0
            / SUM(Total_Sales) OVER ()
        ) AS Sales_Contribution_Percent

    FROM region_sales

    ORDER BY Total_Sales DESC;
    """,
    conn
)

display(region_contribution.round(2))
display(region_contribution.describe().T)

,Region,Total_Sales,Overall_Sales,Sales_Contribution_Percent
0,West,522441.05,1565804.32,33.37
1,East,450234.67,1565804.32,28.75
2,Central,341007.52,1565804.32,21.78
3,South,252121.08,1565804.32,16.10


,count,mean,std,min,25%,50%,75%,max
Total_Sales,4.0,3.914511e+05,119123.582455,2.521211e+05,3.187859e+05,3.956211e+05,4.682863e+05,5.224411e+05
Overall_Sales,4.0,1.565804e+06,0.000000,1.565804e+06,1.565804e+06,1.565804e+06,1.565804e+06,1.565804e+06
Sales_Contribution_Percent,4.0,2.500000e+01,7.607821,1.610170e+01,2.035924e+01,2.526632e+01,2.990707e+01,3.336567e+01


# 🔵 Stage 8 — Advanced Ranking & Contribution Analysis

This stage extends SQL Window Functions into advanced business
ranking, concentration, and contribution analysis.

The objective is to identify not only the highest-performing
customers, products, and categories, but also understand how much
of the overall business is concentrated among the top performers.

## Business Questions

- Who are the Top 10 customers by Sales?
- Who are the Top 10 customers by Profit?
- Which products dominate Sales?
- Which products dominate Profit?
- Which categories contribute most of the business?
- How much Sales comes from the Top 20% of customers?
- How much Sales comes from the Top 20% of products?
- Which customers contribute to 80% of Sales?
- Which products contribute to 80% of Sales?
- What is the cumulative Sales contribution?
- How concentrated is business revenue?
- Which products are high Sales but low Profit?
- Which customers are high Sales but low Profit?

## Ranking Functions

```text
ROW_NUMBER()
RANK()
DENSE_RANK()


---

# 1️⃣ ROW_NUMBER vs RANK vs DENSE_RANK

Business Question:

> How do different SQL ranking functions treat tied Sales values?

The three ranking functions are compared on customer-level Sales.

### ROW_NUMBER

Every row receives a unique number.

### RANK

Tied values receive the same rank, with gaps after the tie.

### DENSE_RANK

Tied values receive the same rank without gaps.

In [101]:
# Compare the three major SQL ranking functions

ranking_comparison = pd.read_sql_query(
    """
    WITH customer_sales AS (
        SELECT
            Customer_ID,
            Customer_Name,
            SUM(Sales) AS Total_Sales
        FROM superstore
        GROUP BY
            Customer_ID,
            Customer_Name
    )

    SELECT
        Customer_ID,
        Customer_Name,
        Total_Sales,

        ROW_NUMBER() OVER (
            ORDER BY Total_Sales DESC
        ) AS Row_Number_Rank,

        RANK() OVER (
            ORDER BY Total_Sales DESC
        ) AS Rank_Value,

        DENSE_RANK() OVER (
            ORDER BY Total_Sales DESC
        ) AS Dense_Rank_Value

    FROM customer_sales

    ORDER BY Total_Sales DESC
    LIMIT 20;
    """,
    conn
)

display(ranking_comparison.round(2))
display(ranking_comparison.describe().T)

,Customer_ID,Customer_Name,Total_Sales,Row_Number_Rank,Rank_Value,Dense_Rank_Value
0,CJ-12010,Caroline Jumper,11596.97,1,1,1
1,KF-16285,Karen Ferguson,10941.27,2,2,2
2,SV-20365,Seth Vernon,10751.15,3,3,3
3,HW-14935,Helen Wasserman,10074.93,4,4,4
4,EH-13765,Edward Hooks,9542.99,5,5,5
5,BS-11365,Bill Shonely,9199.78,6,6,6
6,PK-19075,Pete Kriz,8812.05,7,7,7
7,JL-15835,John Lee,8765.33,8,8,8
8,AB-10060,Adam Bellavance,8167.08,9,9,9
9,JW-15220,Jane Waco,7933.55,10,10,10


,count,mean,std,min,25%,50%,75%,max
Total_Sales,20.0,8515.82688,1396.494327,6952.293,7470.9537,7877.766,9285.582,11596.974
Row_Number_Rank,20.0,10.50000,5.916080,1.000,5.7500,10.500,15.250,20.000
Rank_Value,20.0,10.50000,5.916080,1.000,5.7500,10.500,15.250,20.000
Dense_Rank_Value,20.0,10.50000,5.916080,1.000,5.7500,10.500,15.250,20.000


## 2. Top 10 Customers by Sales

Business Question:

> Which 10 customers generate the highest total Sales?

This identifies the most valuable revenue-generating customers.

In [102]:
# Identify the Top 10 customers by total Sales

top_10_customers = pd.read_sql_query(
    """
    WITH customer_sales AS (
        SELECT
            Customer_ID,
            Customer_Name,
            Segment,
            SUM(Sales) AS Total_Sales,
            SUM(Profit) AS Total_Profit,
            COUNT(DISTINCT Order_ID) AS Total_Orders
        FROM superstore
        GROUP BY
            Customer_ID,
            Customer_Name,
            Segment
    ),

    ranked_customers AS (
        SELECT
            *,
            ROW_NUMBER() OVER (
                ORDER BY Total_Sales DESC
            ) AS Sales_Rank
        FROM customer_sales
    )

    SELECT *
    FROM ranked_customers
    WHERE Sales_Rank <= 10
    ORDER BY Sales_Rank;
    """,
    conn
)

display(top_10_customers.round(2))
display(top_10_customers.describe().T)

,Customer_ID,Customer_Name,Segment,Total_Sales,Total_Profit,Total_Orders,Sales_Rank
0,CJ-12010,Caroline Jumper,Consumer,11596.97,858.74,8,1
1,KF-16285,Karen Ferguson,Home Office,10941.27,1577.61,6,2
2,SV-20365,Seth Vernon,Consumer,10751.15,1610.00,7,3
3,HW-14935,Helen Wasserman,Corporate,10074.93,2146.96,7,4
4,EH-13765,Edward Hooks,Corporate,9542.99,1164.63,9,5
5,BS-11365,Bill Shonely,Corporate,9199.78,2405.36,2,6
6,PK-19075,Pete Kriz,Consumer,8812.05,2024.95,11,7
7,JL-15835,John Lee,Consumer,8765.33,839.96,7,8
8,AB-10060,Adam Bellavance,Home Office,8167.08,2045.87,7,9
9,JW-15220,Jane Waco,Corporate,7933.55,2084.51,5,10


,count,mean,std,min,25%,50%,75%,max
Total_Sales,10.0,9578.51180,1231.271505,7933.554,8777.01250,9371.384,10582.094500,11596.9740
Total_Profit,10.0,1675.86057,559.925529,839.955,1267.87675,1817.476,2074.851175,2405.3645
Total_Orders,10.0,6.90000,2.378141,2.000,6.25000,7.000,7.750000,11.0000
Sales_Rank,10.0,5.50000,3.027650,1.000,3.25000,5.500,7.750000,10.0000


## 3. Top 10 Customers by Profit

Business Question:

> Which customers generate the highest total Profit?

High revenue customers are not always the most profitable, so
Sales and Profit rankings should be analyzed separately.

In [103]:
# Identify the Top 10 customers by total Profit

top_10_profit_customers = pd.read_sql_query(
    """
    WITH customer_profit AS (
        SELECT
            Customer_ID,
            Customer_Name,
            Segment,
            SUM(Sales) AS Total_Sales,
            SUM(Profit) AS Total_Profit
        FROM superstore
        GROUP BY
            Customer_ID,
            Customer_Name,
            Segment
    ),

    ranked_customers AS (
        SELECT
            *,
            ROW_NUMBER() OVER (
                ORDER BY Total_Profit DESC
            ) AS Profit_Rank
        FROM customer_profit
    )

    SELECT *
    FROM ranked_customers
    WHERE Profit_Rank <= 10
    ORDER BY Profit_Rank;
    """,
    conn
)

display(top_10_profit_customers.round(2))
display(top_10_profit_customers.describe().T)

,Customer_ID,Customer_Name,Segment,Total_Sales,Total_Profit,Profit_Rank
0,TC-20980,Tamara Chand,Corporate,1344.05,8764.95,1
1,RB-19360,Raymond Buch,Consumer,5041.34,6976.10,2
2,HL-15040,Hunter Lopez,Consumer,6091.11,5174.40,3
3,AB-10105,Adrian Barton,Consumer,6276.91,5088.94,4
4,TA-21385,Tom Ashbrook,Home Office,3715.77,4523.36,5
5,AR-10540,Andy Reiter,Consumer,6353.82,2617.22,6
6,DR-12940,Daniel Raglin,Home Office,6408.62,2549.73,7
7,SE-20110,Sanjit Engle,Consumer,7821.98,2436.95,8
8,BS-11365,Bill Shonely,Corporate,9199.78,2405.36,9
9,KD-16270,Karen Daniels,Consumer,6317.16,2318.71,10


,count,mean,std,min,25%,50%,75%,max
Total_Sales,10.0,5857.0534,2151.128246,1344.0520,5303.781250,6297.03550,6394.9170,9199.7800
Total_Profit,10.0,4285.5725,2246.314496,2318.7099,2465.147975,3570.28895,5153.0338,8764.9483
Profit_Rank,10.0,5.5000,3.027650,1.0000,3.250000,5.50000,7.7500,10.0000


## 4. Top 10 Products by Sales

Business Question:

> Which products generate the highest total Sales?

This identifies the strongest revenue-generating products.

In [104]:
# Identify the Top 10 products by Sales

top_10_products = pd.read_sql_query(
    """
    WITH product_sales AS (
        SELECT
            Product_ID,
            Product_Name,
            Category,
            "Sub-Category",
            SUM(Sales) AS Total_Sales,
            SUM(Profit) AS Total_Profit
        FROM superstore
        GROUP BY
            Product_ID,
            Product_Name,
            Category,
            "Sub-Category"
    ),

    ranked_products AS (
        SELECT
            *,
            ROW_NUMBER() OVER (
                ORDER BY Total_Sales DESC
            ) AS Sales_Rank
        FROM product_sales
    )

    SELECT *
    FROM ranked_products
    WHERE Sales_Rank <= 10
    ORDER BY Sales_Rank;
    """,
    conn
)

display(top_10_products.round(2))
display(top_10_products.describe().T)

,Product_ID,Product_Name,Category,Sub-Category,Total_Sales,Total_Profit,Sales_Rank
0,TEC-MA-10001047,"3D Systems Cube Printer, 2nd Generation, Magenta",Technology,Machines,14334.89,3717.97,1
1,TEC-CO-10004722,Canon imageCLASS 2200 Advanced Copier,Technology,Copiers,14076.82,25199.93,2
2,TEC-CO-10001449,Hewlett Packard LaserJet 3310 Copier,Technology,Copiers,13837.73,6407.89,3
3,OFF-BI-10001359,GBC DocuBind TL300 Electric Binding System,Office Supplies,Binders,12890.26,2753.76,4
4,OFF-BI-10004995,GBC DocuBind P400 Electric Binding System,Office Supplies,Binders,12577.11,762.15,5
5,TEC-PH-10001459,Samsung Galaxy Mega 6.3,Technology,Phones,12370.71,1696.76,6
6,OFF-SU-10002881,Martin Yale Chadless Opener Electric Letter Op...,Office Supplies,Supplies,12268.90,-1232.56,7
7,FUR-CH-10002024,HON 5400 Series Task Chairs for Big and Tall,Furniture,Chairs,11887.56,70.10,8
8,FUR-CH-10001215,Global Troy Executive Leather Low-Back Tilter,Furniture,Chairs,10217.89,776.52,9
9,OFF-BI-10003527,Fellowes PB500 Electric Punch Plastic Comb Bin...,Office Supplies,Binders,9756.52,-508.40,10


,count,mean,std,min,25%,50%,75%,max
Total_Sales,10.0,12421.84020,1524.603807,9756.5240,11982.8970,12473.9080,13600.863500,14334.890
Total_Profit,10.0,3964.41281,7790.191229,-1232.5588,243.1121,1236.6393,3476.918375,25199.928
Sales_Rank,10.0,5.50000,3.027650,1.0000,3.2500,5.5000,7.750000,10.000


## 5. Top 10 Products by Profit

Business Question:

> Which products generate the highest total Profit?

This provides a profitability-focused product ranking.

In [105]:
# Identify the Top 10 products by Profit

top_10_profit_products = pd.read_sql_query(
    """
    WITH product_profit AS (
        SELECT
            Product_ID,
            Product_Name,
            Category,
            "Sub-Category",
            SUM(Sales) AS Total_Sales,
            SUM(Profit) AS Total_Profit
        FROM superstore
        GROUP BY
            Product_ID,
            Product_Name,
            Category,
            "Sub-Category"
    ),

    ranked_products AS (
        SELECT
            *,
            ROW_NUMBER() OVER (
                ORDER BY Total_Profit DESC
            ) AS Profit_Rank
        FROM product_profit
    )

    SELECT *
    FROM ranked_products
    WHERE Profit_Rank <= 10
    ORDER BY Profit_Rank;
    """,
    conn
)

display(top_10_profit_products.round(2))
display(top_10_profit_products.describe().T)

,Product_ID,Product_Name,Category,Sub-Category,Total_Sales,Total_Profit,Profit_Rank
0,TEC-CO-10004722,Canon imageCLASS 2200 Advanced Copier,Technology,Copiers,14076.82,25199.93,1
1,TEC-CO-10001449,Hewlett Packard LaserJet 3310 Copier,Technology,Copiers,13837.73,6407.89,2
2,TEC-MA-10001047,"3D Systems Cube Printer, 2nd Generation, Magenta",Technology,Machines,14334.89,3717.97,3
3,TEC-MA-10001127,HP Designjet T520 Inkjet Large Format Printer ...,Technology,Machines,4749.95,2799.98,4
4,OFF-BI-10001359,GBC DocuBind TL300 Electric Binding System,Office Supplies,Binders,12890.26,2753.76,5
5,TEC-MA-10003979,Ativa V4110MDD Micro-Cut Shredder,Technology,Machines,4899.93,2400.97,6
6,OFF-AP-10002945,Honeywell Enviracaire Portable HEPA Air Cleane...,Office Supplies,Appliances,7275.81,2396.18,7
7,TEC-AC-10003033,Plantronics CS510 - Over-the-Head monaural Wir...,Technology,Accessories,7664.85,2062.19,8
8,TEC-CO-10003763,Canon PC1060 Personal Laser Copier,Technology,Copiers,5599.92,1889.97,9
9,TEC-AC-10003870,Logitech Z-906 Speaker sys - home theater - 5....,Technology,Accessories,4657.86,1847.94,10


,count,mean,std,min,25%,50%,75%,max
Total_Sales,10.0,8998.80240,4253.320488,4657.860,5074.92750,7470.3300,13600.86350,14334.890
Total_Profit,10.0,5147.67866,7173.281847,1847.944,2145.68575,2577.3625,3488.47455,25199.928
Profit_Rank,10.0,5.50000,3.027650,1.000,3.25000,5.5000,7.75000,10.000


## 6. Customer Sales Contribution

Business Question:

> What percentage of total business Sales does each customer
> contribute?

This identifies the revenue contribution of individual customers.

In [106]:
# Calculate individual customer contribution to total Sales

customer_contribution = pd.read_sql_query(
    """
    WITH customer_sales AS (
        SELECT
            Customer_ID,
            Customer_Name,
            SUM(Sales) AS Total_Sales
        FROM superstore
        GROUP BY
            Customer_ID,
            Customer_Name
    )

    SELECT
        Customer_ID,
        Customer_Name,
        Total_Sales,

        SUM(Total_Sales) OVER ()
            AS Overall_Sales,

        Total_Sales * 100.0
        / SUM(Total_Sales) OVER ()
            AS Sales_Contribution_Percent

    FROM customer_sales

    ORDER BY Total_Sales DESC;
    """,
    conn
)

display(customer_contribution.head(20).round(4))
display(customer_contribution.describe().T)

,Customer_ID,Customer_Name,Total_Sales,Overall_Sales,Sales_Contribution_Percent
0,CJ-12010,Caroline Jumper,11596.9740,1.565804e+06,0.7406
1,KF-16285,Karen Ferguson,10941.2740,1.565804e+06,0.6988
2,SV-20365,Seth Vernon,10751.1480,1.565804e+06,0.6866
3,HW-14935,Helen Wasserman,10074.9340,1.565804e+06,0.6434
4,EH-13765,Edward Hooks,9542.9880,1.565804e+06,0.6095
5,BS-11365,Bill Shonely,9199.7800,1.565804e+06,0.5875
6,PK-19075,Pete Kriz,8812.0540,1.565804e+06,0.5628
7,JL-15835,John Lee,8765.3320,1.565804e+06,0.5598
8,AB-10060,Adam Bellavance,8167.0800,1.565804e+06,0.5216
9,JW-15220,Jane Waco,7933.5540,1.565804e+06,0.5067


,count,mean,std,min,25%,50%,75%,max
Total_Sales,773.0,2.025620e+03,1778.099150,5.304000e+00,7.487420e+02,1.584784e+03,2.717490e+03,1.159697e+04
Overall_Sales,773.0,1.565804e+06,0.000000,1.565804e+06,1.565804e+06,1.565804e+06,1.565804e+06,1.565804e+06
Sales_Contribution_Percent,773.0,1.293661e-01,0.113558,3.387396e-04,4.781836e-02,1.012121e-01,1.735523e-01,7.406401e-01


## 7. Cumulative Customer Sales Contribution

Business Question:

> How quickly do the highest-value customers accumulate the
> company's total Sales?

Cumulative contribution is required for Pareto analysis.

Customers are ordered from highest to lowest Sales and their
contribution is accumulated progressively.

In [107]:
# Calculate cumulative Sales contribution by customer

customer_cumulative = pd.read_sql_query(
    """
    WITH customer_sales AS (
        SELECT
            Customer_ID,
            Customer_Name,
            SUM(Sales) AS Total_Sales
        FROM superstore
        GROUP BY
            Customer_ID,
            Customer_Name
    ),

    customer_contribution AS (
        SELECT
            Customer_ID,
            Customer_Name,
            Total_Sales,

            Total_Sales * 100.0
            / SUM(Total_Sales) OVER ()
                AS Sales_Contribution_Percent

        FROM customer_sales
    )

    SELECT
        Customer_ID,
        Customer_Name,
        Total_Sales,
        Sales_Contribution_Percent,

        SUM(Sales_Contribution_Percent) OVER (
            ORDER BY Total_Sales DESC
            ROWS BETWEEN UNBOUNDED PRECEDING
            AND CURRENT ROW
        ) AS Cumulative_Sales_Contribution

    FROM customer_contribution

    ORDER BY Total_Sales DESC;
    """,
    conn
)

display(customer_cumulative.head(20).round(4))
display(customer_cumulative.describe().T)

,Customer_ID,Customer_Name,Total_Sales,Sales_Contribution_Percent,Cumulative_Sales_Contribution
0,CJ-12010,Caroline Jumper,11596.9740,0.7406,0.7406
1,KF-16285,Karen Ferguson,10941.2740,0.6988,1.4394
2,SV-20365,Seth Vernon,10751.1480,0.6866,2.1260
3,HW-14935,Helen Wasserman,10074.9340,0.6434,2.7695
4,EH-13765,Edward Hooks,9542.9880,0.6095,3.3789
5,BS-11365,Bill Shonely,9199.7800,0.5875,3.9665
6,PK-19075,Pete Kriz,8812.0540,0.5628,4.5292
7,JL-15835,John Lee,8765.3320,0.5598,5.0890
8,AB-10060,Adam Bellavance,8167.0800,0.5216,5.6106
9,JW-15220,Jane Waco,7933.5540,0.5067,6.1173


,count,mean,std,min,25%,50%,75%,max
Total_Sales,773.0,2025.620082,1778.099150,5.304000,748.742000,1584.784000,2717.490000,11596.97400
Sales_Contribution_Percent,773.0,0.129366,0.113558,0.000339,0.047818,0.101212,0.173552,0.74064
Cumulative_Sales_Contribution,773.0,72.472289,26.569476,0.740640,55.044695,81.112642,95.566937,100.00000


## 8. Customers Required to Reach 80% of Sales

Business Question:

> How many customers are responsible for approximately 80% of
> total Sales?

This is a Pareto-style concentration analysis.

The cumulative Sales contribution is calculated and customers
contributing up to the 80% threshold are identified.

In [108]:
# Identify customers contributing to the first 80% of Sales

customer_80_percent = pd.read_sql_query(
    """
    WITH customer_sales AS (
        SELECT
            Customer_ID,
            Customer_Name,
            SUM(Sales) AS Total_Sales
        FROM superstore
        GROUP BY
            Customer_ID,
            Customer_Name
    ),

    ranked_customers AS (
        SELECT
            *,
            SUM(Total_Sales) OVER ()
                AS Overall_Sales,

            SUM(Total_Sales) OVER (
                ORDER BY Total_Sales DESC
                ROWS BETWEEN UNBOUNDED PRECEDING
                AND CURRENT ROW
            ) AS Cumulative_Sales

        FROM customer_sales
    )

    SELECT
        Customer_ID,
        Customer_Name,
        Total_Sales,
        Overall_Sales,
        Cumulative_Sales,
        Cumulative_Sales * 100.0
            / Overall_Sales
            AS Cumulative_Sales_Percent

    FROM ranked_customers

    WHERE
        Cumulative_Sales * 100.0
        / Overall_Sales <= 80

    ORDER BY Total_Sales DESC;
    """,
    conn
)

display(customer_80_percent.round(4))

,Customer_ID,Customer_Name,Total_Sales,Overall_Sales,Cumulative_Sales,Cumulative_Sales_Percent
0,CJ-12010,Caroline Jumper,11596.9740,1.565804e+06,1.159697e+04,0.7406
1,KF-16285,Karen Ferguson,10941.2740,1.565804e+06,2.253825e+04,1.4394
2,SV-20365,Seth Vernon,10751.1480,1.565804e+06,3.328940e+04,2.1260
3,HW-14935,Helen Wasserman,10074.9340,1.565804e+06,4.336433e+04,2.7695
4,EH-13765,Edward Hooks,9542.9880,1.565804e+06,5.290732e+04,3.3789
...,...,...,...,...,...,...
371,DW-13480,Dianna Wilson,1653.9980,1.565804e+06,1.245864e+06,79.5670
372,MD-17350,Maribeth Dona,1653.1580,1.565804e+06,1.247517e+06,79.6726
373,DV-13045,Darrin Van Huff,1648.4830,1.565804e+06,1.249166e+06,79.7779
374,NL-18310,Nancy Lomonaco,1639.7620,1.565804e+06,1.250806e+06,79.8826


## 9. Product Sales Concentration

Business Question:

> What percentage of total Sales is generated by the highest-
> selling products?

This measures how concentrated business revenue is across the
product portfolio.

In [109]:
# Calculate cumulative Product Sales contribution

product_concentration = pd.read_sql_query(
    """
    WITH product_sales AS (
        SELECT
            Product_ID,
            Product_Name,
            Category,
            SUM(Sales) AS Total_Sales
        FROM superstore
        GROUP BY
            Product_ID,
            Product_Name,
            Category
    )

    SELECT
        Product_ID,
        Product_Name,
        Category,
        Total_Sales,

        SUM(Total_Sales) OVER ()
            AS Overall_Sales,

        SUM(Total_Sales) OVER (
            ORDER BY Total_Sales DESC
            ROWS BETWEEN UNBOUNDED PRECEDING
            AND CURRENT ROW
        ) AS Cumulative_Sales,

        SUM(Total_Sales) OVER (
            ORDER BY Total_Sales DESC
            ROWS BETWEEN UNBOUNDED PRECEDING
            AND CURRENT ROW
        ) * 100.0
        / SUM(Total_Sales) OVER ()
            AS Cumulative_Sales_Percent

    FROM product_sales

    ORDER BY Total_Sales DESC;
    """,
    conn
)

display(product_concentration.head(20).round(4))
display(product_concentration.describe().T)

,Product_ID,Product_Name,Category,Total_Sales,Overall_Sales,Cumulative_Sales,Cumulative_Sales_Percent
0,TEC-MA-10001047,"3D Systems Cube Printer, 2nd Generation, Magenta",Technology,14334.890,1.565804e+06,14334.890,0.9155
1,TEC-CO-10004722,Canon imageCLASS 2200 Advanced Copier,Technology,14076.824,1.565804e+06,28411.714,1.8145
2,TEC-CO-10001449,Hewlett Packard LaserJet 3310 Copier,Technology,13837.732,1.565804e+06,42249.446,2.6983
3,OFF-BI-10001359,GBC DocuBind TL300 Electric Binding System,Office Supplies,12890.258,1.565804e+06,55139.704,3.5215
4,OFF-BI-10004995,GBC DocuBind P400 Electric Binding System,Office Supplies,12577.108,1.565804e+06,67716.812,4.3247
5,TEC-PH-10001459,Samsung Galaxy Mega 6.3,Technology,12370.708,1.565804e+06,80087.520,5.1148
6,OFF-SU-10002881,Martin Yale Chadless Opener Electric Letter Op...,Office Supplies,12268.902,1.565804e+06,92356.422,5.8983
7,FUR-CH-10002024,HON 5400 Series Task Chairs for Big and Tall,Furniture,11887.562,1.565804e+06,104243.984,6.6575
8,FUR-CH-10001215,Global Troy Executive Leather Low-Back Tilter,Furniture,10217.894,1.565804e+06,114461.878,7.3101
9,OFF-BI-10003527,Fellowes PB500 Electric Punch Plastic Comb Bin...,Office Supplies,9756.524,1.565804e+06,124218.402,7.9332


,count,mean,std,min,25%,50%,75%,max
Total_Sales,1783.0,8.781853e+02,1.430452e+03,2.610000e+00,2.115200e+02,4.284400e+02,8.737210e+02,1.433489e+04
Overall_Sales,1783.0,1.565804e+06,2.328960e-10,1.565804e+06,1.565804e+06,1.565804e+06,1.565804e+06,1.565804e+06
Cumulative_Sales,1783.0,1.254618e+06,3.342686e+05,1.433489e+04,1.104401e+06,1.374918e+06,1.513318e+06,1.565804e+06
Cumulative_Sales_Percent,1783.0,8.012610e+01,2.134805e+01,9.154969e-01,7.053249e+01,8.780904e+01,9.664796e+01,1.000000e+02


## 10. Product Sales Percentile

Business Question:

> Where does each product stand within the overall product
> Sales distribution?

PERCENT_RANK() assigns a relative position to every product based
on Total Sales.

In [110]:
# Calculate relative Sales position of every product

product_percentile = pd.read_sql_query(
    """
    WITH product_sales AS (
        SELECT
            Product_ID,
            Product_Name,
            Category,
            SUM(Sales) AS Total_Sales
        FROM superstore
        GROUP BY
            Product_ID,
            Product_Name,
            Category
    )

    SELECT
        Product_ID,
        Product_Name,
        Category,
        Total_Sales,

        PERCENT_RANK() OVER (
            ORDER BY Total_Sales
        ) AS Sales_Percentile

    FROM product_sales

    ORDER BY Total_Sales DESC;
    """,
    conn
)

display(product_percentile.head(20).round(4))
display(product_percentile.describe().T)

,Product_ID,Product_Name,Category,Total_Sales,Sales_Percentile
0,TEC-MA-10001047,"3D Systems Cube Printer, 2nd Generation, Magenta",Technology,14334.890,1.0000
1,TEC-CO-10004722,Canon imageCLASS 2200 Advanced Copier,Technology,14076.824,0.9994
2,TEC-CO-10001449,Hewlett Packard LaserJet 3310 Copier,Technology,13837.732,0.9989
3,OFF-BI-10001359,GBC DocuBind TL300 Electric Binding System,Office Supplies,12890.258,0.9983
4,OFF-BI-10004995,GBC DocuBind P400 Electric Binding System,Office Supplies,12577.108,0.9978
5,TEC-PH-10001459,Samsung Galaxy Mega 6.3,Technology,12370.708,0.9972
6,OFF-SU-10002881,Martin Yale Chadless Opener Electric Letter Op...,Office Supplies,12268.902,0.9966
7,FUR-CH-10002024,HON 5400 Series Task Chairs for Big and Tall,Furniture,11887.562,0.9961
8,FUR-CH-10001215,Global Troy Executive Leather Low-Back Tilter,Furniture,10217.894,0.9955
9,OFF-BI-10003527,Fellowes PB500 Electric Punch Plastic Comb Bin...,Office Supplies,9756.524,0.9949


,count,mean,std,min,25%,50%,75%,max
Total_Sales,1783.0,878.185263,1430.452031,2.61,211.520000,428.44,873.721,14334.89
Sales_Percentile,1783.0,0.499998,0.288920,0.00,0.249719,0.50,0.750,1.00


## 11. Customer Quartile Segmentation

Business Question:

> Can customers be divided into four Sales-based performance
> groups?

NTILE(4) divides customers into four approximately equal groups
based on their Sales.

```text
Quartile 1 → Highest Sales
Quartile 2 → High-Medium Sales
Quartile 3 → Low-Medium Sales
Quartile 4 → Lowest Sales

In [111]:
### 💻 Code Cell

# Divide customers into four Sales-based quartiles

customer_quartiles = pd.read_sql_query(
    """
    WITH customer_sales AS (
        SELECT
            Customer_ID,
            Customer_Name,
            SUM(Sales) AS Total_Sales,
            SUM(Profit) AS Total_Profit
        FROM superstore
        GROUP BY
            Customer_ID,
            Customer_Name
    )

    SELECT
        Customer_ID,
        Customer_Name,
        Total_Sales,
        Total_Profit,

        NTILE(4) OVER (
            ORDER BY Total_Sales DESC
        ) AS Sales_Quartile

    FROM customer_sales

    ORDER BY Sales_Quartile, Total_Sales DESC;
    """,
    conn
)

display(customer_quartiles.head(30).round(2))
display(customer_quartiles.describe().T)

,Customer_ID,Customer_Name,Total_Sales,Total_Profit,Sales_Quartile
0,CJ-12010,Caroline Jumper,11596.97,858.74,1
1,KF-16285,Karen Ferguson,10941.27,1577.61,1
2,SV-20365,Seth Vernon,10751.15,1610.00,1
3,HW-14935,Helen Wasserman,10074.93,2146.96,1
4,EH-13765,Edward Hooks,9542.99,1164.63,1
5,BS-11365,Bill Shonely,9199.78,2405.36,1
6,PK-19075,Pete Kriz,8812.05,2024.95,1
7,JL-15835,John Lee,8765.33,839.96,1
8,AB-10060,Adam Bellavance,8167.08,2045.87,1
9,JW-15220,Jane Waco,7933.55,2084.51,1


,count,mean,std,min,25%,50%,75%,max
Total_Sales,773.0,2025.620082,1778.099150,5.3040,748.7420,1584.7840,2717.4900,11596.9740
Total_Profit,773.0,226.729762,752.374279,-6652.7235,15.4452,115.7168,317.5336,8764.9483
Sales_Quartile,773.0,2.498060,1.119335,1.0000,1.0000,2.0000,3.0000,4.0000


## 12. High Sales but Low Profit Products

Business Question:

> Which products generate high Sales but relatively weak Profit?

These products can represent potential pricing, discount,
cost, or margin optimization opportunities.

The analysis compares each product against the average product
Sales and Profit benchmarks.

In [112]:
# Identify high-Sales but low-Profit products

high_sales_low_profit = pd.read_sql_query(
    """
    WITH product_summary AS (
        SELECT
            Product_ID,
            Product_Name,
            Category,
            SUM(Sales) AS Total_Sales,
            SUM(Profit) AS Total_Profit
        FROM superstore
        GROUP BY
            Product_ID,
            Product_Name,
            Category
    ),

    product_benchmarks AS (
        SELECT
            AVG(Total_Sales) AS Avg_Product_Sales,
            AVG(Total_Profit) AS Avg_Product_Profit
        FROM product_summary
    )

    SELECT
        p.Product_ID,
        p.Product_Name,
        p.Category,
        p.Total_Sales,
        p.Total_Profit,

        CASE
            WHEN p.Total_Sales > b.Avg_Product_Sales
             AND p.Total_Profit < b.Avg_Product_Profit
            THEN 'High Sales / Low Profit'
            ELSE 'Other'
        END AS Performance_Status

    FROM product_summary AS p

    CROSS JOIN product_benchmarks AS b

    WHERE
        p.Total_Sales > b.Avg_Product_Sales
        AND p.Total_Profit < b.Avg_Product_Profit

    ORDER BY p.Total_Sales DESC;
    """,
    conn
)

display(high_sales_low_profit.round(2))
display(high_sales_low_profit.describe().T)

,Product_ID,Product_Name,Category,Total_Sales,Total_Profit,Performance_Status
0,OFF-SU-10002881,Martin Yale Chadless Opener Electric Letter Op...,Office Supplies,12268.90,-1232.56,High Sales / Low Profit
1,FUR-CH-10002024,HON 5400 Series Task Chairs for Big and Tall,Furniture,11887.56,70.10,High Sales / Low Profit
2,OFF-BI-10003527,Fellowes PB500 Electric Punch Plastic Comb Bin...,Office Supplies,9756.52,-508.40,High Sales / Low Profit
3,TEC-MA-10000418,Cubify CubeX 3D Printer Double Head Print,Technology,9323.97,-6239.98,High Sales / Low Profit
4,OFF-SU-10000151,High Speed Automatic Electric Letter Opener,Office Supplies,8842.66,-589.51,High Sales / Low Profit
...,...,...,...,...,...,...
186,FUR-FU-10001935,3M Hangers With Command Adhesive,Furniture,913.68,17.02,High Sales / Low Profit
187,OFF-BI-10002012,Wilson Jones Easy Flow II Sheet Lifters,Office Supplies,906.04,2.09,High Sales / Low Profit
188,TEC-AC-10001998,Logitech LS21 Speaker System - PC Multimedia -...,Technology,904.91,55.97,High Sales / Low Profit
189,TEC-PH-10002564,OtterBox Defender Series Case - Samsung Galaxy S4,Technology,904.82,77.97,High Sales / Low Profit


,count,mean,std,min,25%,50%,75%,max
Total_Sales,191.0,2144.304291,1856.596913,885.3480,1180.014,1404.40,2216.2690,12268.9020
Total_Profit,191.0,-190.579361,614.752930,-6239.9792,-179.784,-19.47,36.9054,91.5288


## 13. High Sales but Low Profit Customers

Business Question:

> Which customers generate high revenue but relatively weak
> profitability?

These customers may require deeper analysis of discounts,
product mix, order behavior, or service costs.

In [113]:
# Identify high-Sales but low-Profit customers

high_sales_low_profit_customers = pd.read_sql_query(
    """
    WITH customer_summary AS (
        SELECT
            Customer_ID,
            Customer_Name,
            Segment,
            SUM(Sales) AS Total_Sales,
            SUM(Profit) AS Total_Profit
        FROM superstore
        GROUP BY
            Customer_ID,
            Customer_Name,
            Segment
    ),

    customer_benchmarks AS (
        SELECT
            AVG(Total_Sales) AS Avg_Customer_Sales,
            AVG(Total_Profit) AS Avg_Customer_Profit
        FROM customer_summary
    )

    SELECT
        c.Customer_ID,
        c.Customer_Name,
        c.Segment,
        c.Total_Sales,
        c.Total_Profit,

        CASE
            WHEN c.Total_Sales > b.Avg_Customer_Sales
             AND c.Total_Profit < b.Avg_Customer_Profit
            THEN 'High Sales / Low Profit'
            ELSE 'Other'
        END AS Performance_Status

    FROM customer_summary AS c

    CROSS JOIN customer_benchmarks AS b

    WHERE
        c.Total_Sales > b.Avg_Customer_Sales
        AND c.Total_Profit < b.Avg_Customer_Profit

    ORDER BY c.Total_Sales DESC;
    """,
    conn
)

display(high_sales_low_profit_customers.round(2))
display(high_sales_low_profit_customers.describe().T)

,Customer_ID,Customer_Name,Segment,Total_Sales,Total_Profit,Performance_Status
0,MH-18115,Mick Hernandez,Home Office,7813.09,170.97,High Sales / Low Profit
1,TP-21415,Tom Prescott,Consumer,7456.41,-1151.91,High Sales / Low Profit
2,PO-18850,Patrick O'Brill,Consumer,7453.98,-25.15,High Sales / Low Profit
3,EP-13915,Emily Phan,Consumer,7230.63,-391.24,High Sales / Low Profit
4,JA-15970,Joseph Airdo,Consumer,6262.71,-916.36,High Sales / Low Profit
...,...,...,...,...,...,...
115,NB-18655,Nona Balk,Corporate,2060.90,-96.81,High Sales / Low Profit
116,DH-13075,Dave Hallsten,Corporate,2056.67,87.38,High Sales / Low Profit
117,SA-20830,Sue Ann Reed,Consumer,2051.45,160.74,High Sales / Low Profit
118,EB-13750,Edward Becker,Corporate,2046.70,-257.70,High Sales / Low Profit


,count,mean,std,min,25%,50%,75%,max
Total_Sales,120.0,3153.619176,1209.138294,2028.1120,2301.93475,2752.69060,3536.214400,7813.0930
Total_Profit,120.0,-258.131704,878.953745,-6652.7235,-269.88595,17.19935,144.354725,225.5701


# 🔵 Stage 9 — Time Intelligence & Period-over-Period Analysis

This stage focuses on time-based SQL business analytics.

The objective is to understand how Sales and Profit change across
years, quarters, and months and identify growth, decline, trends,
seasonality, and high-performing periods.

---

## 🎯 Business Questions

- Which year generated the highest Sales?
- Which year generated the highest Profit?
- How did Sales change year-over-year?
- How did Profit change year-over-year?
- Which quarter performs best?
- Which quarter generates the highest Profit?
- Which months generate the highest Sales?
- Which months generate the highest Profit?
- How does Sales change month-over-month?
- Which periods show Sales growth?
- Which periods show Sales decline?
- What is the cumulative Sales over time?
- What is the cumulative Profit over time?
- Which periods have the highest Profit Margin?
- Which periods have the weakest performance?
- Which months contribute the most to annual Sales?

---

## SQL Concepts

```text
GROUP BY
LAG()
LEAD()
SUM() OVER()
AVG() OVER()
RANK()
DENSE_RANK()
PARTITION BY
ORDER BY
CASE WHEN

## 1. Yearly Sales & Profit Performance

Business Question:

> Which years generated the highest Sales and Profit?

The transaction data is aggregated by Order Year to establish the
overall annual business performance baseline.

In [114]:
# Calculate yearly Sales, Profit, Quantity and Orders

yearly_performance = pd.read_sql_query(
    """
    SELECT
        Order_Year,
        SUM(Sales) AS Total_Sales,
        SUM(Profit) AS Total_Profit,
        SUM(Quantity) AS Total_Quantity,
        COUNT(DISTINCT Order_ID) AS Total_Orders,
        COUNT(DISTINCT Customer_ID) AS Total_Customers
    FROM superstore
    GROUP BY Order_Year
    ORDER BY Order_Year;
    """,
    conn
)

yearly_performance["Profit_Margin_%"] = (
    yearly_performance["Total_Profit"]
    / yearly_performance["Total_Sales"]
    * 100
)

display(yearly_performance.round(2))
display(yearly_performance.describe().T)

,Order_Year,Total_Sales,Total_Profit,Total_Quantity,Total_Orders,Total_Customers,Profit_Margin_%
0,NaN,944552.25,105137.52,13198,1791,695,11.13
1,2019.0,245863.79,44553.36,4106,531,372,18.12
2,2020.0,375388.28,25571.22,5013,681,465,6.81


,count,mean,std,min,25%,50%,75%,max
Order_Year,2.0,2019.500000,0.707107,2019.000000,2019.250000,2019.500000,2019.750000,2020.000000
Total_Sales,3.0,521934.774400,371683.055767,245863.792900,310626.038600,375388.284300,659970.265150,944552.246000
Total_Profit,3.0,58420.701967,41556.306112,25571.218100,35062.290800,44553.363500,74845.443900,105137.524300
Total_Quantity,3.0,7439.000000,5008.015875,4106.000000,4559.500000,5013.000000,9105.500000,13198.000000
Total_Orders,3.0,1001.000000,688.258672,531.000000,606.000000,681.000000,1236.000000,1791.000000
Total_Customers,3.0,510.666667,166.271866,372.000000,418.500000,465.000000,580.000000,695.000000
Profit_Margin_%,3.0,12.021344,5.706945,6.811938,8.971438,11.130938,14.626047,18.121157


## 2. Year-over-Year Sales Growth

Business Question:

> How much did Sales increase or decrease compared with the
> previous year?

LAG() retrieves the previous year's Sales and allows the
year-over-year growth rate to be calculated.

In [115]:
# Calculate Year-over-Year Sales Growth

yoy_sales = pd.read_sql_query(
    """
    WITH yearly_sales AS (
        SELECT
            Order_Year,
            SUM(Sales) AS Total_Sales
        FROM superstore
        GROUP BY Order_Year
    ),

    yearly_comparison AS (
        SELECT
            Order_Year,
            Total_Sales,

            LAG(Total_Sales) OVER (
                ORDER BY Order_Year
            ) AS Previous_Year_Sales

        FROM yearly_sales
    )

    SELECT
        Order_Year,
        Total_Sales,
        Previous_Year_Sales,

        Total_Sales - Previous_Year_Sales
            AS Sales_Change,

        CASE
            WHEN Previous_Year_Sales IS NULL
                THEN NULL
            WHEN Previous_Year_Sales = 0
                THEN NULL
            ELSE
                (Total_Sales - Previous_Year_Sales)
                * 100.0
                / Previous_Year_Sales
        END AS Sales_Growth_Percent

    FROM yearly_comparison
    ORDER BY Order_Year;
    """,
    conn
)

display(yoy_sales.round(2))
display(yoy_sales.describe().T)

,Order_Year,Total_Sales,Previous_Year_Sales,Sales_Change,Sales_Growth_Percent
0,NaN,944552.25,NaN,NaN,NaN
1,2019.0,245863.79,944552.25,-698688.45,-73.97
2,2020.0,375388.28,245863.79,129524.49,52.68


,count,mean,std,min,25%,50%,75%,max
Order_Year,2.0,2019.500000,0.707107,2019.000000,2019.250000,2019.500000,2019.750000,2020.000000
Total_Sales,3.0,521934.774400,371683.055767,245863.792900,310626.038600,375388.284300,659970.265150,944552.246000
Previous_Year_Sales,2.0,595208.019450,494047.343124,245863.792900,420535.906175,595208.019450,769880.132725,944552.246000
Sales_Change,2.0,-284581.980850,585634.989322,-698688.453100,-491635.216975,-284581.980850,-77528.744725,129524.491400
Sales_Growth_Percent,2.0,-10.644466,89.556301,-73.970334,-42.307400,-10.644466,21.018467,52.681401


## 3. Year-over-Year Profit Growth

Business Question:

> How did Profit change from one year to the next?

Sales growth does not always mean profitability growth, so Profit
must be analyzed independently.

In [116]:
# Calculate Year-over-Year Profit Growth

yoy_profit = pd.read_sql_query(
    """
    WITH yearly_profit AS (
        SELECT
            Order_Year,
            SUM(Profit) AS Total_Profit
        FROM superstore
        GROUP BY Order_Year
    ),

    profit_comparison AS (
        SELECT
            Order_Year,
            Total_Profit,

            LAG(Total_Profit) OVER (
                ORDER BY Order_Year
            ) AS Previous_Year_Profit

        FROM yearly_profit
    )

    SELECT
        Order_Year,
        Total_Profit,
        Previous_Year_Profit,

        Total_Profit - Previous_Year_Profit
            AS Profit_Change,

        CASE
            WHEN Previous_Year_Profit IS NULL
                THEN NULL
            WHEN Previous_Year_Profit = 0
                THEN NULL
            ELSE
                (Total_Profit - Previous_Year_Profit)
                * 100.0
                / ABS(Previous_Year_Profit)
        END AS Profit_Growth_Percent

    FROM profit_comparison
    ORDER BY Order_Year;
    """,
    conn
)

display(yoy_profit.round(2))
display(yoy_profit.describe().T)

,Order_Year,Total_Profit,Previous_Year_Profit,Profit_Change,Profit_Growth_Percent
0,NaN,105137.52,NaN,NaN,NaN
1,2019.0,44553.36,105137.52,-60584.16,-57.62
2,2020.0,25571.22,44553.36,-18982.15,-42.61


,count,mean,std,min,25%,50%,75%,max
Order_Year,2.0,2019.500000,0.707107,2019.000000,2019.25000,2019.500000,2019.750000,2020.000000
Total_Profit,3.0,58420.701967,41556.306112,25571.218100,35062.29080,44553.363500,74845.443900,105137.524300
Previous_Year_Profit,2.0,74845.443900,42839.470934,44553.363500,59699.40370,74845.443900,89991.484100,105137.524300
Profit_Change,2.0,-39783.153100,29417.067200,-60584.160800,-50183.65695,-39783.153100,-29382.649250,-18982.145400
Profit_Growth_Percent,2.0,-50.114571,10.619551,-57.623728,-53.86915,-50.114571,-46.359993,-42.605415


## 4. Yearly Business Ranking

Business Question:

> Which years are the strongest based on Sales and Profit?

Each year receives separate Sales and Profit rankings.

In [117]:
# Rank years according to Sales and Profit

yearly_ranking = pd.read_sql_query(
    """
    WITH yearly_summary AS (
        SELECT
            Order_Year,
            SUM(Sales) AS Total_Sales,
            SUM(Profit) AS Total_Profit
        FROM superstore
        GROUP BY Order_Year
    )

    SELECT
        Order_Year,
        Total_Sales,
        Total_Profit,

        DENSE_RANK() OVER (
            ORDER BY Total_Sales DESC
        ) AS Sales_Rank,

        DENSE_RANK() OVER (
            ORDER BY Total_Profit DESC
        ) AS Profit_Rank

    FROM yearly_summary

    ORDER BY Order_Year;
    """,
    conn
)

display(yearly_ranking.round(2))
display(yearly_ranking.describe().T)

,Order_Year,Total_Sales,Total_Profit,Sales_Rank,Profit_Rank
0,NaN,944552.25,105137.52,1,1
1,2019.0,245863.79,44553.36,3,2
2,2020.0,375388.28,25571.22,2,3


,count,mean,std,min,25%,50%,75%,max
Order_Year,2.0,2019.500000,0.707107,2019.0000,2019.2500,2019.5000,2019.75000,2020.0000
Total_Sales,3.0,521934.774400,371683.055767,245863.7929,310626.0386,375388.2843,659970.26515,944552.2460
Total_Profit,3.0,58420.701967,41556.306112,25571.2181,35062.2908,44553.3635,74845.44390,105137.5243
Sales_Rank,3.0,2.000000,1.000000,1.0000,1.5000,2.0000,2.50000,3.0000
Profit_Rank,3.0,2.000000,1.000000,1.0000,1.5000,2.0000,2.50000,3.0000


## 5. Quarterly Sales & Profit Performance

Business Question:

> Which quarters generate the strongest Sales and Profit?

Quarter-level aggregation reveals seasonal business performance
within each year.

In [118]:
# Analyze Sales and Profit by Year and Quarter

quarterly_performance = pd.read_sql_query(
    """
    SELECT
        Order_Year,
        Order_Quarter,
        SUM(Sales) AS Total_Sales,
        SUM(Profit) AS Total_Profit,
        SUM(Quantity) AS Total_Quantity,
        COUNT(DISTINCT Order_ID) AS Total_Orders
    FROM superstore
    GROUP BY
        Order_Year,
        Order_Quarter
    ORDER BY
        Order_Year,
        Order_Quarter;
    """,
    conn
)

quarterly_performance["Profit_Margin_%"] = (
    quarterly_performance["Total_Profit"]
    / quarterly_performance["Total_Sales"]
    * 100
)

display(quarterly_performance.round(2))
display(quarterly_performance.describe().T)

,Order_Year,Order_Quarter,Total_Sales,Total_Profit,Total_Quantity,Total_Orders,Profit_Margin_%
0,NaN,NaN,944552.25,105137.52,13198,1791,11.13
1,2019.0,1.0,72068.59,22496.20,1067,130,31.21
2,2019.0,2.0,59691.26,9689.52,1057,135,16.23
3,2019.0,3.0,49517.26,4978.74,860,129,10.05
4,2019.0,4.0,64586.68,7388.91,1122,137,11.44
5,2020.0,1.0,127947.84,12018.64,1659,204,9.39
6,2020.0,2.0,66421.88,1358.15,991,144,2.04
7,2020.0,3.0,96046.04,8753.05,1222,162,9.11
8,2020.0,4.0,84972.52,3441.38,1141,171,4.05


,count,mean,std,min,25%,50%,75%,max
Order_Year,8.0,2019.500000,0.534522,2019.000000,2019.000000,2019.500000,2020.000000,2020.000000
Order_Quarter,8.0,2.500000,1.195229,1.000000,1.750000,2.500000,3.250000,4.000000
Total_Sales,9.0,173978.258133,289907.539382,49517.263600,64586.680500,72068.591000,96046.039500,944552.246000
Total_Profit,9.0,19473.567322,32701.340524,1358.151800,4978.737400,8753.047100,12018.635100,105137.524300
Total_Quantity,9.0,2479.666667,4025.422835,860.000000,1057.000000,1122.000000,1222.000000,13198.000000
Total_Orders,9.0,333.666667,547.043417,129.000000,135.000000,144.000000,171.000000,1791.000000
Profit_Margin_%,9.0,11.630555,8.427585,2.044736,9.113387,10.054549,11.440293,31.214989


## 6. Quarterly Sales Ranking

Business Question:

> Which quarters are the strongest Sales periods across all years?

This ranking compares every year-quarter combination.

In [119]:
# Rank all Year-Quarter combinations by Sales

quarterly_ranking = pd.read_sql_query(
    """
    WITH quarterly_sales AS (
        SELECT
            Order_Year,
            Order_Quarter,
            SUM(Sales) AS Total_Sales,
            SUM(Profit) AS Total_Profit
        FROM superstore
        GROUP BY
            Order_Year,
            Order_Quarter
    )

    SELECT
        Order_Year,
        Order_Quarter,
        Total_Sales,
        Total_Profit,

        DENSE_RANK() OVER (
            ORDER BY Total_Sales DESC
        ) AS Sales_Rank

    FROM quarterly_sales

    ORDER BY Sales_Rank;
    """,
    conn
)

display(quarterly_ranking.round(2))
display(quarterly_ranking.describe().T)

,Order_Year,Order_Quarter,Total_Sales,Total_Profit,Sales_Rank
0,NaN,NaN,944552.25,105137.52,1
1,2020.0,1.0,127947.84,12018.64,2
2,2020.0,3.0,96046.04,8753.05,3
3,2020.0,4.0,84972.52,3441.38,4
4,2019.0,1.0,72068.59,22496.20,5
5,2020.0,2.0,66421.88,1358.15,6
6,2019.0,4.0,64586.68,7388.91,7
7,2019.0,2.0,59691.26,9689.52,8
8,2019.0,3.0,49517.26,4978.74,9


,count,mean,std,min,25%,50%,75%,max
Order_Year,8.0,2019.500000,0.534522,2019.0000,2019.0000,2019.5000,2020.0000,2020.0000
Order_Quarter,8.0,2.500000,1.195229,1.0000,1.7500,2.5000,3.2500,4.0000
Total_Sales,9.0,173978.258133,289907.539382,49517.2636,64586.6805,72068.5910,96046.0395,944552.2460
Total_Profit,9.0,19473.567322,32701.340524,1358.1518,4978.7374,8753.0471,12018.6351,105137.5243
Sales_Rank,9.0,5.000000,2.738613,1.0000,3.0000,5.0000,7.0000,9.0000


## 7. Monthly Sales & Profit Performance

Business Question:

> Which months generate the highest Sales and Profit?

Monthly aggregation helps identify seasonal demand patterns.

In [120]:
# Calculate monthly Sales and Profit

monthly_performance = pd.read_sql_query(
    """
    SELECT
        Order_Year,
        Order_Month,
        Order_Month_Name,
        SUM(Sales) AS Total_Sales,
        SUM(Profit) AS Total_Profit,
        SUM(Quantity) AS Total_Quantity,
        COUNT(DISTINCT Order_ID) AS Total_Orders
    FROM superstore
    GROUP BY
        Order_Year,
        Order_Month,
        Order_Month_Name
    ORDER BY
        Order_Year,
        Order_Month;
    """,
    conn
)

monthly_performance["Profit_Margin_%"] = (
    monthly_performance["Total_Profit"]
    / monthly_performance["Total_Sales"]
    * 100
)

display(monthly_performance.round(2))
display(monthly_performance.describe().T)

,Order_Year,Order_Month,Order_Month_Name,Total_Sales,Total_Profit,Total_Quantity,Total_Orders,Profit_Margin_%
0,NaN,NaN,NaN,944552.25,105137.52,13198,1791,11.13
1,2019.0,1.0,January,26416.48,7085.72,336,47,26.82
2,2019.0,2.0,February,19598.74,13891.17,324,37,70.88
3,2019.0,3.0,March,26053.38,1519.30,407,47,5.83
4,2019.0,4.0,April,22296.39,5031.83,324,46,22.57
5,2019.0,5.0,May,21777.46,3226.08,446,57,14.81
6,2019.0,6.0,June,15617.40,1431.60,287,32,9.17
7,2019.0,7.0,July,16571.28,3784.66,227,38,22.84
8,2019.0,8.0,August,23156.32,-328.34,402,53,-1.42
9,2019.0,9.0,September,9789.66,1522.42,231,38,15.55


,count,mean,std,min,25%,50%,75%,max
Order_Year,24.0,2019.500000,0.510754,2019.000000,2019.000000,2019.500000,2020.00000,2020.000000
Order_Month,24.0,6.500000,3.526299,1.000000,3.750000,6.500000,9.25000,12.000000
Total_Sales,25.0,62632.172928,183926.268903,9789.662000,21472.218500,23892.002000,32039.47950,944552.246000
Total_Profit,25.0,7010.484236,20676.185719,-3331.957800,1519.304300,2436.668400,3784.65640,105137.524300
Total_Quantity,25.0,892.680000,2565.217595,227.000000,327.000000,374.000000,421.00000,13198.000000
Total_Orders,25.0,120.160000,348.262195,32.000000,45.000000,51.000000,57.00000,1791.000000
Profit_Margin_%,25.0,12.342685,14.782047,-14.007022,6.358696,10.319549,14.81386,70.877913


## 8. Month-over-Month Sales Growth

Business Question:

> How does monthly Sales change compared with the previous month?

LAG() compares each month's Sales with the immediately preceding
period.

In [121]:
# Calculate Month-over-Month Sales Growth

mom_sales = pd.read_sql_query(
    """
    WITH monthly_sales AS (
        SELECT
            Order_Year,
            Order_Month,
            Order_Month_Name,
            SUM(Sales) AS Total_Sales
        FROM superstore
        GROUP BY
            Order_Year,
            Order_Month,
            Order_Month_Name
    ),

    monthly_comparison AS (
        SELECT
            Order_Year,
            Order_Month,
            Order_Month_Name,
            Total_Sales,

            LAG(Total_Sales) OVER (
                ORDER BY
                    Order_Year,
                    Order_Month
            ) AS Previous_Month_Sales

        FROM monthly_sales
    )

    SELECT
        Order_Year,
        Order_Month,
        Order_Month_Name,
        Total_Sales,
        Previous_Month_Sales,

        Total_Sales - Previous_Month_Sales
            AS Sales_Change,

        CASE
            WHEN Previous_Month_Sales IS NULL
                THEN NULL
            WHEN Previous_Month_Sales = 0
                THEN NULL
            ELSE
                (Total_Sales - Previous_Month_Sales)
                * 100.0
                / Previous_Month_Sales
        END AS MoM_Growth_Percent

    FROM monthly_comparison

    ORDER BY
        Order_Year,
        Order_Month;
    """,
    conn
)

display(mom_sales.round(2))
display(mom_sales.describe().T)

,Order_Year,Order_Month,Order_Month_Name,Total_Sales,Previous_Month_Sales,Sales_Change,MoM_Growth_Percent
0,NaN,NaN,NaN,944552.25,NaN,NaN,NaN
1,2019.0,1.0,January,26416.48,944552.25,-918135.76,-97.20
2,2019.0,2.0,February,19598.74,26416.48,-6817.75,-25.81
3,2019.0,3.0,March,26053.38,19598.74,6454.64,32.93
4,2019.0,4.0,April,22296.39,26053.38,-3756.98,-14.42
5,2019.0,5.0,May,21777.46,22296.39,-518.94,-2.33
6,2019.0,6.0,June,15617.40,21777.46,-6160.05,-28.29
7,2019.0,7.0,July,16571.28,15617.40,953.87,6.11
8,2019.0,8.0,August,23156.32,16571.28,6585.04,39.74
9,2019.0,9.0,September,9789.66,23156.32,-13366.66,-57.72


,count,mean,std,min,25%,50%,75%,max
Order_Year,24.0,2019.500000,0.510754,2019.00000,2019.000000,2019.500000,2020.000000,2020.00000
Order_Month,24.0,6.500000,3.526299,1.00000,3.750000,6.500000,9.250000,12.00000
Total_Sales,25.0,62632.172928,183926.268903,9789.66200,21472.218500,23892.002000,32039.479500,944552.24600
Previous_Month_Sales,24.0,64246.346717,187701.150446,9789.66200,21223.615875,23844.962000,32170.409625,944552.24600
Sales_Change,24.0,-38360.843500,187574.533637,-918135.76500,-6324.476850,-620.862350,4648.068500,18014.69350
MoM_Growth_Percent,24.0,2.288032,46.824931,-97.20328,-22.675188,-2.291687,16.011691,131.23019


## 9. Running Sales Over Time

Business Question:

> How does cumulative Sales grow throughout the complete
> business timeline?

A window SUM creates a running Sales total ordered by year and
month.

In [122]:
# Calculate cumulative Sales across the complete timeline

running_annual_sales = pd.read_sql_query(
    """
    WITH monthly_sales AS (
        SELECT
            Order_Year,
            Order_Month,
            Order_Month_Name,
            SUM(Sales) AS Total_Sales
        FROM superstore
        GROUP BY
            Order_Year,
            Order_Month,
            Order_Month_Name
    )

    SELECT
        Order_Year,
        Order_Month,
        Order_Month_Name,
        Total_Sales,

        SUM(Total_Sales) OVER (
            ORDER BY
                Order_Year,
                Order_Month
            ROWS BETWEEN UNBOUNDED PRECEDING
            AND CURRENT ROW
        ) AS Cumulative_Sales

    FROM monthly_sales

    ORDER BY
        Order_Year,
        Order_Month;
    """,
    conn
)

display(running_annual_sales.round(2))
display(running_annual_sales.describe().T)

,Order_Year,Order_Month,Order_Month_Name,Total_Sales,Cumulative_Sales
0,NaN,NaN,NaN,944552.25,944552.25
1,2019.0,1.0,January,26416.48,970968.73
2,2019.0,2.0,February,19598.74,990567.46
3,2019.0,3.0,March,26053.38,1016620.84
4,2019.0,4.0,April,22296.39,1038917.23
5,2019.0,5.0,May,21777.46,1060694.69
6,2019.0,6.0,June,15617.40,1076312.09
7,2019.0,7.0,July,16571.28,1092883.37
8,2019.0,8.0,August,23156.32,1116039.70
9,2019.0,9.0,September,9789.66,1125829.36


,count,mean,std,min,25%,50%,75%,max
Order_Year,24.0,2.019500e+03,0.510754,2019.000,2.019000e+03,2.019500e+03,2.020000e+03,2.020000e+03
Order_Month,24.0,6.500000e+00,3.526299,1.000,3.750000e+00,6.500000e+00,9.250000e+00,1.200000e+01
Total_Sales,25.0,6.263217e+04,183926.268903,9789.662,2.147222e+04,2.389200e+04,3.203948e+04,9.445522e+05
Cumulative_Sales,25.0,1.232928e+06,192920.964077,944552.246,1.076312e+06,1.190416e+06,1.384786e+06,1.565804e+06


## 10. Seasonal Monthly Performance

Business Question:

> Which calendar months consistently generate the highest Sales
> across different years?

Instead of treating each Year-Month separately, this analysis
groups the same calendar month across all available years.

This helps identify recurring seasonal patterns.

In [123]:
# Analyze recurring Sales patterns by calendar month

seasonal_monthly_performance = pd.read_sql_query(
    """
    SELECT
        Order_Month,
        Order_Month_Name,
        SUM(Sales) AS Total_Sales,
        SUM(Profit) AS Total_Profit,
        AVG(Sales) AS Average_Monthly_Transaction_Sales,
        COUNT(DISTINCT Order_ID) AS Total_Orders
    FROM superstore
    GROUP BY
        Order_Month,
        Order_Month_Name
    ORDER BY
        Total_Sales DESC;
    """,
    conn
)

seasonal_monthly_performance["Profit_Margin_%"] = (
    seasonal_monthly_performance["Total_Profit"]
    / seasonal_monthly_performance["Total_Sales"]
    * 100
)

display(seasonal_monthly_performance.round(2))
display(seasonal_monthly_performance.describe().T)

,Order_Month,Order_Month_Name,Total_Sales,Total_Profit,Average_Monthly_Transaction_Sales,Total_Orders,Profit_Margin_%
0,NaN,NaN,944552.25,105137.52,270.57,1791,11.13
1,2.0,February,67569.19,16941.47,272.46,113,25.07
2,3.0,March,66543.85,6908.50,277.27,124,10.38
3,1.0,January,65903.39,10664.87,324.65,98,16.18
4,8.0,August,54473.01,5222.02,265.72,100,9.59
5,11.0,November,53041.01,4913.98,261.29,103,9.26
6,10.0,October,51153.98,2789.84,265.05,108,5.45
7,7.0,July,48610.76,4550.67,294.61,93,9.36
8,4.0,April,46084.16,1699.88,235.12,102,3.69
9,5.0,May,45679.62,5983.14,221.75,97,13.10


,count,mean,std,min,25%,50%,75%,max
Order_Month,12.0,6.500000,3.605551,1.000000,3.750000,6.500000,9.250000,12.000000
Total_Sales,13.0,120446.486400,247812.651757,34349.360800,45679.615800,51153.976000,65903.393000,944552.246000
Total_Profit,13.0,13481.700454,27831.208791,1699.875900,3364.655100,4913.982500,6908.496700,105137.524300
Average_Monthly_Transaction_Sales,13.0,257.911825,32.288329,217.401018,224.575349,265.046508,272.456415,324.647256
Total_Orders,13.0,231.076923,468.812589,80.000000,97.000000,100.000000,108.000000,1791.000000
Profit_Margin_%,13.0,10.709874,5.335963,3.688634,9.264497,9.586431,11.130938,25.072773


## 11. Best and Worst Sales Periods

Business Question:

> Which Year-Month combinations generated the highest and lowest
> Sales?

The complete timeline is ranked to identify the strongest and
weakest business periods.

In [124]:
# Rank Year-Month combinations by Sales

sales_period_ranking = pd.read_sql_query(
    """
    WITH monthly_sales AS (
        SELECT
            Order_Year,
            Order_Month,
            Order_Month_Name,
            SUM(Sales) AS Total_Sales,
            SUM(Profit) AS Total_Profit
        FROM superstore
        GROUP BY
            Order_Year,
            Order_Month,
            Order_Month_Name
    )

    SELECT
        Order_Year,
        Order_Month,
        Order_Month_Name,
        Total_Sales,
        Total_Profit,

        DENSE_RANK() OVER (
            ORDER BY Total_Sales DESC
        ) AS Best_Sales_Rank,

        DENSE_RANK() OVER (
            ORDER BY Total_Sales ASC
        ) AS Worst_Sales_Rank

    FROM monthly_sales

    ORDER BY Total_Sales DESC;
    """,
    conn
)

display(sales_period_ranking.round(2))
display(sales_period_ranking.describe().T)

,Order_Year,Order_Month,Order_Month_Name,Total_Sales,Total_Profit,Best_Sales_Rank,Worst_Sales_Rank
0,NaN,NaN,NaN,944552.25,105137.52,1,25
1,2020.0,2.0,February,47970.46,3050.30,2,24
2,2020.0,3.0,March,40490.47,5389.19,3,23
3,2020.0,1.0,January,39486.91,3579.15,4,22
4,2020.0,9.0,September,32689.87,2436.67,5,21
5,2020.0,11.0,November,32563.20,2134.53,6,20
6,2020.0,7.0,July,32039.48,766.02,7,19
7,2020.0,8.0,August,31316.69,5550.36,8,18
8,2020.0,10.0,October,28517.32,367.52,9,17
9,2019.0,1.0,January,26416.48,7085.72,10,16


,count,mean,std,min,25%,50%,75%,max
Order_Year,24.0,2019.500000,0.510754,2019.0000,2019.0000,2019.5000,2020.0000,2020.0000
Order_Month,24.0,6.500000,3.526299,1.0000,3.7500,6.5000,9.2500,12.0000
Total_Sales,25.0,62632.172928,183926.268903,9789.6620,21472.2185,23892.0020,32039.4795,944552.2460
Total_Profit,25.0,7010.484236,20676.185719,-3331.9578,1519.3043,2436.6684,3784.6564,105137.5243
Best_Sales_Rank,25.0,13.000000,7.359801,1.0000,7.0000,13.0000,19.0000,25.0000
Worst_Sales_Rank,25.0,13.000000,7.359801,1.0000,7.0000,13.0000,19.0000,25.0000


## 12. Monthly Profitability

Business Question:

> Which periods have the strongest and weakest Profit Margin?

Sales alone cannot determine business health. Profit Margin is
calculated from aggregated monthly Sales and Profit.

In [125]:
# Calculate monthly Profit Margin

monthly_profitability = pd.read_sql_query(
    """
    WITH monthly_summary AS (
        SELECT
            Order_Year,
            Order_Month,
            Order_Month_Name,
            SUM(Sales) AS Total_Sales,
            SUM(Profit) AS Total_Profit
        FROM superstore
        GROUP BY
            Order_Year,
            Order_Month,
            Order_Month_Name
    )

    SELECT
        Order_Year,
        Order_Month,
        Order_Month_Name,
        Total_Sales,
        Total_Profit,

        CASE
            WHEN Total_Sales = 0 THEN 0
            ELSE
                Total_Profit * 100.0 / Total_Sales
        END AS Profit_Margin_Percent

    FROM monthly_summary

    ORDER BY Profit_Margin_Percent DESC;
    """,
    conn
)

display(monthly_profitability.round(2))
display(monthly_profitability.describe().T)

,Order_Year,Order_Month,Order_Month_Name,Total_Sales,Total_Profit,Profit_Margin_Percent
0,2019.0,2.0,February,19598.74,13891.17,70.88
1,2019.0,1.0,January,26416.48,7085.72,26.82
2,2019.0,7.0,July,16571.28,3784.66,22.84
3,2019.0,4.0,April,22296.39,5031.83,22.57
4,2020.0,8.0,August,31316.69,5550.36,17.72
5,2019.0,9.0,September,9789.66,1522.42,15.55
6,2019.0,5.0,May,21777.46,3226.08,14.81
7,2019.0,11.0,November,20477.81,2779.45,13.57
8,2020.0,3.0,March,40490.47,5389.19,13.31
9,2020.0,5.0,May,23902.16,2757.06,11.53


,count,mean,std,min,25%,50%,75%,max
Order_Year,24.0,2019.500000,0.510754,2019.000000,2019.000000,2019.500000,2020.00000,2020.000000
Order_Month,24.0,6.500000,3.526299,1.000000,3.750000,6.500000,9.25000,12.000000
Total_Sales,25.0,62632.172928,183926.268903,9789.662000,21472.218500,23892.002000,32039.47950,944552.246000
Total_Profit,25.0,7010.484236,20676.185719,-3331.957800,1519.304300,2436.668400,3784.65640,105137.524300
Profit_Margin_Percent,25.0,12.342685,14.782047,-14.007022,6.358696,10.319549,14.81386,70.877913


# 🔵 Stage 10 — Multi-Dimensional Business Analysis

This stage combines multiple business dimensions to understand
relationships between Sales, Profit, Customers, Products,
Geography, Segments, Shipping, Payments, and Returns.

Instead of analyzing each dimension independently, this stage
answers cross-dimensional business questions.

---

## 🎯 Business Questions

### Customer Analysis

- Which customer segments generate the highest Sales by Region?
- Which customers perform best within each Segment?
- Which regions contain the highest-value customers?

### Product Analysis

- Which Categories perform best in each Region?
- Which Sub-Categories generate the highest Profit within each Category?
- Which products dominate their respective Regions?

### Geography Analysis

- Which Category is strongest in each Region?
- Which States perform best within each Region?
- Which Cities generate the highest Sales within each State?

### Operations

- Which Shipping Modes perform best within each Region?
- Which Payment Modes are most common within each Segment?
- Which Shipping Modes generate the highest Profit?

### Returns

- Which Categories have the highest number of Returns?
- Which Regions have the highest Return activity?
- Which Segments are most affected by Returns?

---

## 🧠 SQL Concepts

```text
GROUP BY Multiple Dimensions
CTE
CASE WHEN
Conditional Aggregation
Window Functions
RANK()
DENSE_RANK()
ROW_NUMBER()
SUM() OVER()
COUNT()
COUNT(DISTINCT)


---

# 1️⃣ Segment × Region Performance

Business Question:

> Which customer segments generate the highest Sales and Profit
> within each Region?

This analysis identifies the strongest customer segment for every
geographic region.

In [126]:
# Analyze customer Segment performance within each Region

segment_region_performance = pd.read_sql_query(
    """
    SELECT
        Region,
        Segment,
        SUM(Sales) AS Total_Sales,
        SUM(Profit) AS Total_Profit,
        SUM(Quantity) AS Total_Quantity,
        COUNT(DISTINCT Order_ID) AS Total_Orders,
        COUNT(DISTINCT Customer_ID) AS Total_Customers
    FROM superstore
    GROUP BY
        Region,
        Segment
    ORDER BY
        Region,
        Total_Sales DESC;
    """,
    conn
)

display(segment_region_performance.round(2))
display(segment_region_performance.describe().T)

,Region,Segment,Total_Sales,Total_Profit,Total_Quantity,Total_Orders,Total_Customers
0,Central,Consumer,162433.73,8220.55,2539,358,237
1,Central,Corporate,110915.15,13333.66,1693,219,144
2,Central,Home Office,67658.65,5895.80,1007,134,90
3,East,Consumer,226701.34,19728.85,3130,425,264
4,East,Corporate,141351.17,16460.89,1961,269,162
5,East,Home Office,82182.15,17210.68,1160,150,97
6,South,Consumer,135067.93,17989.65,1858,268,197
7,South,Corporate,75241.88,5616.75,995,127,99
8,South,Home Office,41811.27,2945.32,676,92,61
9,West,Consumer,228799.12,35399.54,3672,477,284


,count,mean,std,min,25%,50%,75%,max
Total_Sales,12.0,130483.693600,60830.018617,41811.2655,80447.084625,123237.46750,167384.028325,228799.1225
Total_Profit,12.0,14605.175492,9028.240844,2945.3181,7639.360200,14897.27625,18424.452850,35399.5351
Total_Quantity,12.0,1859.750000,909.160664,676.0000,1121.750000,1775.50000,2296.750000,3672.0000
Total_Orders,12.0,250.250000,122.796302,92.0000,146.000000,243.50000,314.500000,477.0000
Total_Customers,12.0,158.500000,73.708887,61.0000,98.500000,153.00000,207.000000,284.0000


## 2. Top Segment Within Each Region

Business Question:

> Which customer segment is the top Sales contributor in each
> Region?

A window ranking is applied separately within every Region.

In [127]:
# Rank customer Segments independently within each Region

top_segment_region = pd.read_sql_query(
    """
    WITH segment_region AS (
        SELECT
            Region,
            Segment,
            SUM(Sales) AS Total_Sales,
            SUM(Profit) AS Total_Profit
        FROM superstore
        GROUP BY
            Region,
            Segment
    ),

    ranked_segments AS (
        SELECT
            Region,
            Segment,
            Total_Sales,
            Total_Profit,

            DENSE_RANK() OVER (
                PARTITION BY Region
                ORDER BY Total_Sales DESC
            ) AS Segment_Rank

        FROM segment_region
    )

    SELECT *
    FROM ranked_segments
    WHERE Segment_Rank = 1
    ORDER BY Total_Sales DESC;
    """,
    conn
)

display(top_segment_region.round(2))
display(top_segment_region.describe().T)

,Region,Segment,Total_Sales,Total_Profit,Segment_Rank
0,West,Consumer,228799.12,35399.54,1
1,East,Consumer,226701.34,19728.85,1
2,Central,Consumer,162433.73,8220.55,1
3,South,Consumer,135067.93,17989.65,1


,count,mean,std,min,25%,50%,75%,max
Total_Sales,4.0,188250.532275,46966.475228,135067.933,155592.279700,194567.5368,227225.789375,228799.1225
Total_Profit,4.0,20334.646875,11248.233654,8220.546,15547.375875,18859.2532,23646.524200,35399.5351
Segment_Rank,4.0,1.000000,0.000000,1.000,1.000000,1.0000,1.000000,1.0000


## 3. Category × Region Performance

Business Question:

> Which Categories perform best within each Region?

This identifies regional product-category strengths.

In [128]:
# Analyze Category performance across Regions

category_region_performance = pd.read_sql_query(
    """
    SELECT
        Region,
        Category,
        SUM(Sales) AS Total_Sales,
        SUM(Profit) AS Total_Profit,
        SUM(Quantity) AS Total_Quantity,
        COUNT(DISTINCT Order_ID) AS Total_Orders
    FROM superstore
    GROUP BY
        Region,
        Category
    ORDER BY
        Region,
        Total_Sales DESC;
    """,
    conn
)

category_region_performance["Profit_Margin_%"] = (
    category_region_performance["Total_Profit"]
    / category_region_performance["Total_Sales"]
    * 100
)

display(category_region_performance.round(2))
display(category_region_performance.describe().T)

,Region,Category,Total_Sales,Total_Profit,Total_Quantity,Total_Orders,Profit_Margin_%
0,Central,Office Supplies,150154.35,6477.63,3277,529,4.31
1,Central,Furniture,105505.45,-1534.79,1057,238,-1.45
2,Central,Technology,85347.72,22507.17,905,210,26.37
3,East,Office Supplies,189066.82,23248.61,3774,635,12.30
4,East,Technology,144095.87,26530.31,1187,263,18.41
5,East,Furniture,117071.97,3621.50,1290,292,3.09
6,South,Office Supplies,99652.14,11104.98,2190,364,11.14
7,South,Technology,79484.07,12884.72,613,150,16.21
8,South,Furniture,72984.87,2562.02,726,162,3.51
9,West,Office Supplies,204834.37,33966.03,4384,691,16.58


,count,mean,std,min,25%,50%,75%,max
Total_Sales,12.0,130483.693600,43538.140042,72984.8670,96076.034500,130583.922500,157374.845750,204834.373000
Total_Profit,12.0,14605.175492,11836.880992,-1534.7880,4923.782225,11994.848000,24069.035300,33966.030100
Total_Quantity,12.0,1859.750000,1267.481693,613.0000,1019.000000,1323.000000,2461.750000,4384.000000
Total_Orders,12.0,347.833333,178.752156,150.0000,231.000000,292.000000,405.250000,691.000000
Profit_Margin_%,12.0,10.963849,8.372553,-1.4547,3.491688,11.720123,16.849609,26.371139


## 4. Top Category Within Each Region

Business Question:

> Which Category is the strongest Sales contributor in each
> Region?

The ranking restarts for every Region.

In [129]:
# Identify the highest-selling Category within every Region

top_category_region = pd.read_sql_query(
    """
    WITH category_region AS (
        SELECT
            Region,
            Category,
            SUM(Sales) AS Total_Sales,
            SUM(Profit) AS Total_Profit
        FROM superstore
        GROUP BY
            Region,
            Category
    ),

    ranked_categories AS (
        SELECT
            Region,
            Category,
            Total_Sales,
            Total_Profit,

            DENSE_RANK() OVER (
                PARTITION BY Region
                ORDER BY Total_Sales DESC
            ) AS Category_Rank

        FROM category_region
    )

    SELECT *
    FROM ranked_categories
    WHERE Category_Rank = 1
    ORDER BY Region;
    """,
    conn
)

display(top_category_region.round(2))
display(top_category_region.describe().T)

,Region,Category,Total_Sales,Total_Profit,Category_Rank
0,Central,Office Supplies,150154.35,6477.63,1
1,East,Office Supplies,189066.82,23248.61,1
2,South,Office Supplies,99652.14,11104.98,1
3,West,Office Supplies,204834.37,33966.03,1


,count,mean,std,min,25%,50%,75%,max
Total_Sales,4.0,160926.921750,46869.909754,99652.1400,137528.79975,169610.58700,193008.709000,204834.3730
Total_Profit,4.0,18699.311525,12393.711531,6477.6297,9948.13935,17176.79315,25927.965325,33966.0301
Category_Rank,4.0,1.000000,0.000000,1.0000,1.00000,1.00000,1.000000,1.0000


# 🔵 Stage 10 — Multi-Dimensional Business Analysis

This stage combines multiple business dimensions to understand
relationships between Sales, Profit, Customers, Products,
Geography, Segments, Shipping, Payments, and Returns.

Instead of analyzing each dimension independently, this stage
answers cross-dimensional business questions.

---

## 🎯 Business Questions

### Customer Analysis

- Which customer segments generate the highest Sales by Region?
- Which customers perform best within each Segment?
- Which regions contain the highest-value customers?

### Product Analysis

- Which Categories perform best in each Region?
- Which Sub-Categories generate the highest Profit within each Category?
- Which products dominate their respective Regions?

### Geography Analysis

- Which Category is strongest in each Region?
- Which States perform best within each Region?
- Which Cities generate the highest Sales within each State?

### Operations

- Which Shipping Modes perform best within each Region?
- Which Payment Modes are most common within each Segment?
- Which Shipping Modes generate the highest Profit?

### Returns

- Which Categories have the highest number of Returns?
- Which Regions have the highest Return activity?
- Which Segments are most affected by Returns?

---

## 🧠 SQL Concepts

```text
GROUP BY Multiple Dimensions
CTE
CASE WHEN
Conditional Aggregation
Window Functions
RANK()
DENSE_RANK()
ROW_NUMBER()
SUM() OVER()
COUNT()
COUNT(DISTINCT)


---

# 1️⃣ Segment × Region Performance

Business Question:

> Which customer segments generate the highest Sales and Profit
> within each Region?

This analysis identifies the strongest customer segment for every
geographic region.

In [130]:
# Analyze customer Segment performance within each Region

segment_region_performance = pd.read_sql_query(
    """
    SELECT
        Region,
        Segment,
        SUM(Sales) AS Total_Sales,
        SUM(Profit) AS Total_Profit,
        SUM(Quantity) AS Total_Quantity,
        COUNT(DISTINCT Order_ID) AS Total_Orders,
        COUNT(DISTINCT Customer_ID) AS Total_Customers
    FROM superstore
    GROUP BY
        Region,
        Segment
    ORDER BY
        Region,
        Total_Sales DESC;
    """,
    conn
)

display(segment_region_performance.round(2))
display(segment_region_performance.describe().T)

,Region,Segment,Total_Sales,Total_Profit,Total_Quantity,Total_Orders,Total_Customers
0,Central,Consumer,162433.73,8220.55,2539,358,237
1,Central,Corporate,110915.15,13333.66,1693,219,144
2,Central,Home Office,67658.65,5895.80,1007,134,90
3,East,Consumer,226701.34,19728.85,3130,425,264
4,East,Corporate,141351.17,16460.89,1961,269,162
5,East,Home Office,82182.15,17210.68,1160,150,97
6,South,Consumer,135067.93,17989.65,1858,268,197
7,South,Corporate,75241.88,5616.75,995,127,99
8,South,Home Office,41811.27,2945.32,676,92,61
9,West,Consumer,228799.12,35399.54,3672,477,284


,count,mean,std,min,25%,50%,75%,max
Total_Sales,12.0,130483.693600,60830.018617,41811.2655,80447.084625,123237.46750,167384.028325,228799.1225
Total_Profit,12.0,14605.175492,9028.240844,2945.3181,7639.360200,14897.27625,18424.452850,35399.5351
Total_Quantity,12.0,1859.750000,909.160664,676.0000,1121.750000,1775.50000,2296.750000,3672.0000
Total_Orders,12.0,250.250000,122.796302,92.0000,146.000000,243.50000,314.500000,477.0000
Total_Customers,12.0,158.500000,73.708887,61.0000,98.500000,153.00000,207.000000,284.0000


## 2. Top Segment Within Each Region

Business Question:

> Which customer segment is the top Sales contributor in each
> Region?

A window ranking is applied separately within every Region.

In [131]:
# Rank customer Segments independently within each Region

top_segment_region = pd.read_sql_query(
    """
    WITH segment_region AS (
        SELECT
            Region,
            Segment,
            SUM(Sales) AS Total_Sales,
            SUM(Profit) AS Total_Profit
        FROM superstore
        GROUP BY
            Region,
            Segment
    ),

    ranked_segments AS (
        SELECT
            Region,
            Segment,
            Total_Sales,
            Total_Profit,

            DENSE_RANK() OVER (
                PARTITION BY Region
                ORDER BY Total_Sales DESC
            ) AS Segment_Rank

        FROM segment_region
    )

    SELECT *
    FROM ranked_segments
    WHERE Segment_Rank = 1
    ORDER BY Total_Sales DESC;
    """,
    conn
)

display(top_segment_region.round(2))
display(top_segment_region.describe().T)

,Region,Segment,Total_Sales,Total_Profit,Segment_Rank
0,West,Consumer,228799.12,35399.54,1
1,East,Consumer,226701.34,19728.85,1
2,Central,Consumer,162433.73,8220.55,1
3,South,Consumer,135067.93,17989.65,1


,count,mean,std,min,25%,50%,75%,max
Total_Sales,4.0,188250.532275,46966.475228,135067.933,155592.279700,194567.5368,227225.789375,228799.1225
Total_Profit,4.0,20334.646875,11248.233654,8220.546,15547.375875,18859.2532,23646.524200,35399.5351
Segment_Rank,4.0,1.000000,0.000000,1.000,1.000000,1.0000,1.000000,1.0000


## 3. Category × Region Performance

Business Question:

> Which Categories perform best within each Region?

This identifies regional product-category strengths.

In [132]:
# Analyze Category performance across Regions

category_region_performance = pd.read_sql_query(
    """
    SELECT
        Region,
        Category,
        SUM(Sales) AS Total_Sales,
        SUM(Profit) AS Total_Profit,
        SUM(Quantity) AS Total_Quantity,
        COUNT(DISTINCT Order_ID) AS Total_Orders
    FROM superstore
    GROUP BY
        Region,
        Category
    ORDER BY
        Region,
        Total_Sales DESC;
    """,
    conn
)

category_region_performance["Profit_Margin_%"] = (
    category_region_performance["Total_Profit"]
    / category_region_performance["Total_Sales"]
    * 100
)

display(category_region_performance.round(2))
display(category_region_performance.describe().T)

,Region,Category,Total_Sales,Total_Profit,Total_Quantity,Total_Orders,Profit_Margin_%
0,Central,Office Supplies,150154.35,6477.63,3277,529,4.31
1,Central,Furniture,105505.45,-1534.79,1057,238,-1.45
2,Central,Technology,85347.72,22507.17,905,210,26.37
3,East,Office Supplies,189066.82,23248.61,3774,635,12.30
4,East,Technology,144095.87,26530.31,1187,263,18.41
5,East,Furniture,117071.97,3621.50,1290,292,3.09
6,South,Office Supplies,99652.14,11104.98,2190,364,11.14
7,South,Technology,79484.07,12884.72,613,150,16.21
8,South,Furniture,72984.87,2562.02,726,162,3.51
9,West,Office Supplies,204834.37,33966.03,4384,691,16.58


,count,mean,std,min,25%,50%,75%,max
Total_Sales,12.0,130483.693600,43538.140042,72984.8670,96076.034500,130583.922500,157374.845750,204834.373000
Total_Profit,12.0,14605.175492,11836.880992,-1534.7880,4923.782225,11994.848000,24069.035300,33966.030100
Total_Quantity,12.0,1859.750000,1267.481693,613.0000,1019.000000,1323.000000,2461.750000,4384.000000
Total_Orders,12.0,347.833333,178.752156,150.0000,231.000000,292.000000,405.250000,691.000000
Profit_Margin_%,12.0,10.963849,8.372553,-1.4547,3.491688,11.720123,16.849609,26.371139


## 4. Top Category Within Each Region

Business Question:

> Which Category is the strongest Sales contributor in each
> Region?

The ranking restarts for every Region.

In [133]:
# Identify the highest-selling Category within every Region

top_category_region = pd.read_sql_query(
    """
    WITH category_region AS (
        SELECT
            Region,
            Category,
            SUM(Sales) AS Total_Sales,
            SUM(Profit) AS Total_Profit
        FROM superstore
        GROUP BY
            Region,
            Category
    ),

    ranked_categories AS (
        SELECT
            Region,
            Category,
            Total_Sales,
            Total_Profit,

            DENSE_RANK() OVER (
                PARTITION BY Region
                ORDER BY Total_Sales DESC
            ) AS Category_Rank

        FROM category_region
    )

    SELECT *
    FROM ranked_categories
    WHERE Category_Rank = 1
    ORDER BY Region;
    """,
    conn
)

display(top_category_region.round(2))
display(top_category_region.describe().T)

,Region,Category,Total_Sales,Total_Profit,Category_Rank
0,Central,Office Supplies,150154.35,6477.63,1
1,East,Office Supplies,189066.82,23248.61,1
2,South,Office Supplies,99652.14,11104.98,1
3,West,Office Supplies,204834.37,33966.03,1


,count,mean,std,min,25%,50%,75%,max
Total_Sales,4.0,160926.921750,46869.909754,99652.1400,137528.79975,169610.58700,193008.709000,204834.3730
Total_Profit,4.0,18699.311525,12393.711531,6477.6297,9948.13935,17176.79315,25927.965325,33966.0301
Category_Rank,4.0,1.000000,0.000000,1.0000,1.00000,1.00000,1.000000,1.0000


## 5. Sub-Category × Category Performance

Business Question:

> Which Sub-Categories generate the highest Sales and Profit
> inside each Category?

This provides a deeper product hierarchy analysis.

In [134]:
# Analyze Sub-Category performance within each Category

subcategory_category_performance = pd.read_sql_query(
    """
    SELECT
        Category,
        "Sub-Category" AS Sub_Category,
        SUM(Sales) AS Total_Sales,
        SUM(Profit) AS Total_Profit,
        SUM(Quantity) AS Total_Quantity,
        COUNT(DISTINCT Order_ID) AS Total_Orders
    FROM superstore
    GROUP BY
        Category,
        "Sub-Category"
    ORDER BY
        Category,
        Total_Sales DESC;
    """,
    conn
)

subcategory_category_performance["Profit_Margin_%"] = (
    subcategory_category_performance["Total_Profit"]
    / subcategory_category_performance["Total_Sales"]
    * 100
)

display(subcategory_category_performance.round(2))
display(subcategory_category_performance.describe().T)

,Category,Sub_Category,Total_Sales,Total_Profit,Total_Quantity,Total_Orders,Profit_Margin_%
0,Furniture,Chairs,181946.00,13406.70,1288,331,7.37
1,Furniture,Tables,119293.74,-11091.64,736,186,-9.30
2,Furniture,Furnishings,92691.22,8034.43,2133,527,8.67
3,Furniture,Bookcases,57577.69,-342.89,474,130,-0.60
4,Office Supplies,Binders,174978.39,17885.38,3670,785,10.22
5,Office Supplies,Storage,150341.32,13607.09,1830,459,9.05
6,Office Supplies,Paper,99453.61,21112.38,3074,719,21.23
7,Office Supplies,Appliances,80305.25,13166.61,1050,271,16.40
8,Office Supplies,Art,50762.98,3635.93,1779,422,7.16
9,Office Supplies,Supplies,36720.99,-1654.28,408,117,-4.50


,count,mean,std,min,25%,50%,75%,max
Total_Sales,17.0,92106.136659,58637.229584,15205.238000,50762.976000,91987.561000,122301.086000,196563.54600
Total_Profit,17.0,10309.535641,12969.304754,-11091.636500,598.417500,8034.432800,17885.375900,42774.58280
Total_Quantity,17.0,1312.764706,1011.405429,142.000000,474.000000,1050.000000,1830.000000,3670.00000
Total_Orders,17.0,318.529412,225.954674,38.000000,130.000000,271.000000,459.000000,785.00000
Profit_Margin_%,17.0,12.335169,17.654817,-9.297752,3.935601,9.050797,16.395703,71.60628


## 6. Top Sub-Category Within Each Category

Business Question:

> Which Sub-Category is the strongest Sales contributor inside
> each Category?

This identifies the leading product group within every major
Category.

In [135]:
# Rank Sub-Categories within their respective Categories

top_subcategory_category = pd.read_sql_query(
    """
    WITH subcategory_sales AS (
        SELECT
            Category,
            "Sub-Category" AS Sub_Category,
            SUM(Sales) AS Total_Sales,
            SUM(Profit) AS Total_Profit
        FROM superstore
        GROUP BY
            Category,
            "Sub-Category"
    ),

    ranked_subcategories AS (
        SELECT
            Category,
            Sub_Category,
            Total_Sales,
            Total_Profit,

            DENSE_RANK() OVER (
                PARTITION BY Category
                ORDER BY Total_Sales DESC
            ) AS Sub_Category_Rank

        FROM subcategory_sales
    )

    SELECT *
    FROM ranked_subcategories
    WHERE Sub_Category_Rank = 1
    ORDER BY Category;
    """,
    conn
)

display(top_subcategory_category.round(2))
display(top_subcategory_category.describe().T)

,Category,Sub_Category,Total_Sales,Total_Profit,Sub_Category_Rank
0,Furniture,Chairs,181946.00,13406.70,1
1,Office Supplies,Binders,174978.39,17885.38,1
2,Technology,Phones,196563.55,22308.92,1


,count,mean,std,min,25%,50%,75%,max
Total_Sales,3.0,184495.978,11016.194369,174978.3900,178462.19400,181945.9980,189254.7720,196563.5460
Total_Profit,3.0,17866.999,4451.135802,13406.7032,15646.03955,17885.3759,20097.1469,22308.9179
Sub_Category_Rank,3.0,1.000,0.000000,1.0000,1.00000,1.0000,1.0000,1.0000


## 7. Shipping Mode × Region Performance

Business Question:

> Which Shipping Mode handles the most orders within each Region?

The analysis combines operational and geographic dimensions.

In [136]:
# Analyze Shipping Mode performance within each Region

shipping_region_performance = pd.read_sql_query(
    """
    SELECT
        Region,
        Ship_Mode,
        COUNT(DISTINCT Order_ID) AS Total_Orders,
        SUM(Sales) AS Total_Sales,
        SUM(Profit) AS Total_Profit,
        AVG(Shipping_Days) AS Average_Shipping_Days
    FROM superstore
    GROUP BY
        Region,
        Ship_Mode
    ORDER BY
        Region,
        Total_Orders DESC;
    """,
    conn
)

display(shipping_region_performance.round(2))
display(shipping_region_performance.describe().T)

,Region,Ship_Mode,Total_Orders,Total_Sales,Total_Profit,Average_Shipping_Days
0,Central,Standard Class,430,208878.44,20635.42,150.57
1,Central,Second Class,132,72591.59,5298.33,89.66
2,Central,First Class,111,40554.57,-330.67,66.12
3,Central,Same Day,38,18982.93,1846.92,0.00
4,East,Standard Class,501,279331.88,36641.12,143.41
5,East,Second Class,158,71400.87,4566.32,105.04
6,East,First Class,142,74451.26,9991.61,63.06
7,East,Same Day,43,25050.66,2201.38,0.00
8,South,Standard Class,282,130687.86,10957.76,144.74
9,South,Second Class,100,68674.31,13651.92,90.63


,count,mean,std,min,25%,50%,75%,max
Total_Orders,16.0,187.687500,167.944324,25.0000,74.250000,137.000000,204.000000,560.000000
Total_Sales,16.0,97862.770200,88072.325851,13731.0940,38819.314750,71996.227500,109052.942000,293502.860000
Total_Profit,16.0,10953.881619,10987.685835,-2278.1976,3715.524975,8515.164500,14206.109375,36641.119100
Average_Shipping_Days,16.0,75.310720,54.026903,0.0000,42.128521,74.908171,114.635288,150.566502


## 8. Payment Mode × Segment Analysis

Business Question:

> Which Payment Modes are most commonly used by each customer
> Segment?

This helps understand payment behavior across customer groups.

In [137]:
# Analyze Payment Mode usage across customer Segments

payment_segment_performance = pd.read_sql_query(
    """
    SELECT
        Segment,
        Payment_Mode,
        COUNT(*) AS Total_Transactions,
        COUNT(DISTINCT Order_ID) AS Total_Orders,
        SUM(Sales) AS Total_Sales,
        SUM(Profit) AS Total_Profit
    FROM superstore
    GROUP BY
        Segment,
        Payment_Mode
    ORDER BY
        Segment,
        Total_Transactions DESC;
    """,
    conn
)

display(payment_segment_performance.round(2))
display(payment_segment_performance.describe().T)

,Segment,Payment_Mode,Total_Transactions,Total_Orders,Total_Sales,Total_Profit
0,Consumer,COD,1241,852,325891.33,44921.45
1,Consumer,Online,1113,810,268323.80,27963.01
2,Consumer,Cards,643,505,158787.00,8454.12
3,Corporate,COD,726,528,210992.60,23985.42
4,Corporate,Online,650,463,182957.06,16177.15
5,Corporate,Cards,398,299,115793.47,17643.23
6,Home Office,COD,486,337,130533.82,13185.55
7,Home Office,Online,401,289,102712.60,9907.36
8,Home Office,Cards,243,181,69812.65,13024.81


,count,mean,std,min,25%,50%,75%,max
Total_Transactions,9.0,655.666667,333.023272,243.0000,401.0000,643.0000,726.0000,1241.0000
Total_Orders,9.0,473.777778,231.636989,181.0000,299.0000,463.0000,528.0000,852.0000
Total_Sales,9.0,173978.258133,82811.741069,69812.6480,115793.4676,158786.9956,210992.6036,325891.3302
Total_Profit,9.0,19473.567322,11447.188907,8454.1204,13024.8053,16177.1474,23985.4199,44921.4526


## 9. Returns × Category Analysis

Business Question:

> Which Categories have the highest number of returned
> transactions?

The analysis compares returned and non-returned activity by
Category.

In [138]:
# Analyze returned and non-returned transactions by Category

return_category_performance = pd.read_sql_query(
    """
    SELECT
        Category,

        COUNT(*) AS Total_Transactions,

        SUM(
            CASE
                WHEN Return_Flag = 1 THEN 1
                ELSE 0
            END
        ) AS Returned_Transactions,

        SUM(
            CASE
                WHEN Return_Flag = 0 THEN 1
                ELSE 0
            END
        ) AS Non_Returned_Transactions,

        SUM(Sales) AS Total_Sales,
        SUM(Profit) AS Total_Profit

    FROM superstore

    GROUP BY Category

    ORDER BY Returned_Transactions DESC;
    """,
    conn
)

return_category_performance["Return_Rate_%"] = (
    return_category_performance["Returned_Transactions"]
    / return_category_performance["Total_Transactions"]
    * 100
)

display(return_category_performance.round(2))
display(return_category_performance.describe().T)

,Category,Total_Transactions,Returned_Transactions,Non_Returned_Transactions,Total_Sales,Total_Profit,Return_Rate_%
0,Office Supplies,3569,165,3404,643707.69,74797.25,4.62
1,Furniture,1249,70,1179,451508.65,10006.61,5.60
2,Technology,1083,52,1031,470587.99,90458.25,4.80


,count,mean,std,min,25%,50%,75%,max
Total_Transactions,3.0,1967.000000,1389.853230,1083.000000,1166.000000,1249.000000,2409.00000,3569.000000
Returned_Transactions,3.0,95.666667,60.715182,52.000000,61.000000,70.000000,117.50000,165.000000
Non_Returned_Transactions,3.0,1871.333333,1329.389459,1031.000000,1105.000000,1179.000000,2291.50000,3404.000000
Total_Sales,3.0,521934.774400,105889.031733,451508.645200,461048.318100,470587.991000,557147.83900,643707.687000
Total_Profit,3.0,58420.701967,42652.782892,10006.611200,42401.928650,74797.246100,82627.74735,90458.248600
Return_Rate_%,3.0,5.009702,0.522757,4.623144,4.712311,4.801477,5.20298,5.604484


## 10. Region × Category × Segment Analysis

Business Question:

> Which combinations of Region, Category, and Segment generate
> the strongest Sales and Profit?

This is a three-dimensional business analysis that combines
geography, product category, and customer segmentation.

It provides a deeper view of where specific customer groups are
buying specific product categories.

In [139]:
# Analyze business performance across Region, Category and Segment

region_category_segment = pd.read_sql_query(
    """
    SELECT
        Region,
        Category,
        Segment,
        SUM(Sales) AS Total_Sales,
        SUM(Profit) AS Total_Profit,
        SUM(Quantity) AS Total_Quantity,
        COUNT(DISTINCT Order_ID) AS Total_Orders,
        COUNT(DISTINCT Customer_ID) AS Total_Customers
    FROM superstore
    GROUP BY
        Region,
        Category,
        Segment
    ORDER BY
        Total_Sales DESC;
    """,
    conn
)

region_category_segment["Profit_Margin_%"] = (
    region_category_segment["Total_Profit"]
    / region_category_segment["Total_Sales"]
    * 100
)

display(region_category_segment.head(30).round(2))
display(region_category_segment.describe().T)

,Region,Category,Segment,Total_Sales,Total_Profit,Total_Quantity,Total_Orders,Total_Customers,Profit_Margin_%
0,East,Office Supplies,Consumer,97287.10,9412.52,1909,320,223,9.67
1,West,Office Supplies,Consumer,93974.67,14961.33,2241,351,239,15.92
2,West,Office Supplies,Corporate,74631.17,13086.46,1272,208,141,17.53
3,Central,Office Supplies,Consumer,73908.09,5169.40,1583,259,192,6.99
4,West,Furniture,Consumer,69343.08,2905.27,802,175,146,4.19
5,East,Furniture,Consumer,66391.82,2630.44,654,151,126,3.96
6,West,Technology,Consumer,65481.37,17532.93,629,131,110,26.78
7,East,Technology,Consumer,63022.43,7685.90,567,126,110,12.20
8,East,Office Supplies,Corporate,58584.29,8193.64,1167,199,139,13.99
9,West,Furniture,Corporate,54084.91,1342.13,480,104,88,2.48


,count,mean,std,min,25%,50%,75%,max
Total_Sales,36.0,43494.564533,21965.382395,9927.898500,29496.045250,42019.135000,55209.758125,97287.100000
Total_Profit,36.0,4868.391831,4844.552216,-3175.272200,1442.634200,4358.194150,7755.899800,17532.929500
Total_Quantity,36.0,619.916667,502.669922,116.000000,263.000000,461.500000,724.000000,2241.000000
Total_Orders,36.0,115.944444,77.393962,28.000000,60.750000,98.000000,136.750000,351.000000
Total_Customers,36.0,90.611111,52.652108,25.000000,51.250000,81.500000,112.750000,239.000000
Profit_Margin_%,36.0,10.744937,12.147580,-7.203685,3.890518,9.510056,15.384777,54.762828


## 11. Top Customer Within Each Region

Business Question:

> Who is the highest-value customer in each Region?

Customers are aggregated first and then ranked independently
inside each Region.

In [140]:
# Identify the highest-Sales customer within each Region

top_customer_region = pd.read_sql_query(
    """
    WITH customer_region AS (
        SELECT
            Region,
            Customer_ID,
            Customer_Name,
            Segment,
            SUM(Sales) AS Total_Sales,
            SUM(Profit) AS Total_Profit
        FROM superstore
        GROUP BY
            Region,
            Customer_ID,
            Customer_Name,
            Segment
    ),

    ranked_customers AS (
        SELECT
            Region,
            Customer_ID,
            Customer_Name,
            Segment,
            Total_Sales,
            Total_Profit,

            DENSE_RANK() OVER (
                PARTITION BY Region
                ORDER BY Total_Sales DESC
            ) AS Customer_Rank

        FROM customer_region
    )

    SELECT *
    FROM ranked_customers
    WHERE Customer_Rank = 1
    ORDER BY Region;
    """,
    conn
)

display(top_customer_region.round(2))
display(top_customer_region.describe().T)

,Region,Customer_ID,Customer_Name,Segment,Total_Sales,Total_Profit,Customer_Rank
0,Central,HW-14935,Helen Wasserman,Corporate,6613.43,1361.68,1
1,East,SV-20365,Seth Vernon,Consumer,9537.30,1545.44,1
2,South,PO-18850,Patrick O'Brill,Consumer,5213.82,366.16,1
3,West,JW-15220,Jane Waco,Corporate,7645.53,2073.28,1


,count,mean,std,min,25%,50%,75%,max
Total_Sales,4.0,7252.517000,1820.202678,5213.8160,6263.5235,7129.47800,8118.471500,9537.2960
Total_Profit,4.0,1336.639875,713.839356,366.1592,1112.7980,1453.55875,1677.400625,2073.2828
Customer_Rank,4.0,1.000000,0.000000,1.0000,1.0000,1.00000,1.000000,1.0000


## 12. Top Product Within Each Region

Business Question:

> Which product generates the highest Sales in each Region?

This identifies regional product leaders.

In [141]:
# Identify the highest-selling product within each Region

top_product_region = pd.read_sql_query(
    """
    WITH product_region AS (
        SELECT
            Region,
            Product_ID,
            Product_Name,
            Category,
            SUM(Sales) AS Total_Sales,
            SUM(Profit) AS Total_Profit
        FROM superstore
        GROUP BY
            Region,
            Product_ID,
            Product_Name,
            Category
    ),

    ranked_products AS (
        SELECT
            Region,
            Product_ID,
            Product_Name,
            Category,
            Total_Sales,
            Total_Profit,

            DENSE_RANK() OVER (
                PARTITION BY Region
                ORDER BY Total_Sales DESC
            ) AS Product_Rank

        FROM product_region
    )

    SELECT *
    FROM ranked_products
    WHERE Product_Rank = 1
    ORDER BY Region;
    """,
    conn
)

display(top_product_region.round(2))
display(top_product_region.describe().T)

,Region,Product_ID,Product_Name,Category,Total_Sales,Total_Profit,Product_Rank
0,Central,OFF-BI-10004995,GBC DocuBind P400 Electric Binding System,Office Supplies,6544.75,653.28,1
1,East,TEC-MA-10001047,"3D Systems Cube Printer, 2nd Generation, Magenta",Technology,14334.89,3717.97,1
2,South,TEC-PH-10001459,Samsung Galaxy Mega 6.3,Technology,7363.83,1091.97,1
3,West,TEC-MA-10000984,Okidata MB760 Printer,Technology,7834.40,881.37,1


,count,mean,std,min,25%,50%,75%,max
Total_Sales,4.0,9019.46700,3583.455823,6544.7520,7159.0575,7599.113,9459.52250,14334.8900
Total_Profit,4.0,1586.14765,1432.462058,653.2752,824.3463,986.672,1748.47335,3717.9714
Product_Rank,4.0,1.00000,0.000000,1.0000,1.0000,1.000,1.00000,1.0000


# 🔵 Stage 11 — Final Business KPIs & Executive SQL Insights

This stage combines the SQL techniques developed throughout the
project to create executive-level business KPIs.

The objective is to convert transaction-level data into a concise
set of strategic business metrics that can support management
decision-making.

---

## 🎯 Executive Business Questions

### Revenue

- What is the total Sales?
- What is the average Order Value?
- How many Orders were placed?
- How many Customers were served?
- How many Products were sold?

### Profitability

- What is the total Profit?
- What is the overall Profit Margin?
- What percentage of transactions are profitable?
- What percentage of transactions generate losses?

### Customers

- What is the average Sales per Customer?
- What is the average number of Orders per Customer?
- Who are the highest-value customers?
- How concentrated is customer revenue?

### Products

- Which Category generates the highest Sales?
- Which Category generates the highest Profit?
- Which products generate losses?
- How concentrated is product revenue?

### Operations

- What is the average Shipping Time?
- Which Shipping Mode is most frequently used?
- What is the Return Rate?
- Which Category has the highest Return Rate?

### Growth

- What is the latest year's Sales?
- What is the latest year's Profit?
- What is the Year-over-Year Sales Growth?
- What is the Year-over-Year Profit Growth?

---

## 📊 Executive KPI Layer

```text
Total Sales
Total Profit
Profit Margin
Total Orders
Total Customers
Total Products
Average Order Value
Average Sales / Customer
Average Orders / Customer
Return Rate
Average Shipping Days
Sales Growth %
Profit Growth %


---

# 1️⃣ Overall Business KPI

Business Question:

> What is the overall financial and operational performance of
> the business?

This query creates the primary executive-level business KPIs
from the original transaction-level columns.

In [142]:
# Calculate overall executive business KPIs

overall_kpis = pd.read_sql_query(
    """
    SELECT
        COUNT(*) AS Total_Transactions,
        COUNT(DISTINCT Order_ID) AS Total_Orders,
        COUNT(DISTINCT Customer_ID) AS Total_Customers,
        COUNT(DISTINCT Product_ID) AS Total_Products,

        SUM(Sales) AS Total_Sales,
        SUM(Profit) AS Total_Profit,
        SUM(Quantity) AS Total_Quantity,

        AVG(Sales) AS Average_Transaction_Sales,
        AVG(Profit) AS Average_Transaction_Profit,

        CASE
            WHEN SUM(Sales) = 0 THEN 0
            ELSE SUM(Profit) * 100.0 / SUM(Sales)
        END AS Overall_Profit_Margin

    FROM superstore;
    """,
    conn
)

display(overall_kpis.round(2))

,Total_Transactions,Total_Orders,Total_Customers,Total_Products,Total_Sales,Total_Profit,Total_Quantity,Average_Transaction_Sales,Average_Transaction_Profit,Overall_Profit_Margin
0,5901,3003,773,1755,1565804.32,175262.11,22317,265.35,29.7,11.19


## 2. Average Order Value

Business Question:

> How much revenue does the business generate per Order on
> average?

Average Order Value is calculated using total Sales divided by
the number of distinct Orders.

In [143]:
# Calculate Average Order Value

aov_analysis = pd.read_sql_query(
    """
    SELECT
        SUM(Sales) AS Total_Sales,
        COUNT(DISTINCT Order_ID) AS Total_Orders,

        CASE
            WHEN COUNT(DISTINCT Order_ID) = 0 THEN 0
            ELSE
                SUM(Sales) * 1.0
                / COUNT(DISTINCT Order_ID)
        END AS Average_Order_Value

    FROM superstore;
    """,
    conn
)

display(aov_analysis.round(2))

,Total_Sales,Total_Orders,Average_Order_Value
0,1565804.32,3003,521.41


## 3. Customer Value KPIs

Business Question:

> What is the average revenue and order activity generated by
> each customer?

This provides a high-level view of customer value and purchasing
frequency.

In [144]:
# Calculate customer-level business KPIs

customer_kpis = pd.read_sql_query(
    """
    WITH customer_summary AS (
        SELECT
            Customer_ID,
            SUM(Sales) AS Customer_Sales,
            COUNT(DISTINCT Order_ID) AS Customer_Orders,
            SUM(Profit) AS Customer_Profit
        FROM superstore
        GROUP BY Customer_ID
    )

    SELECT
        COUNT(*) AS Total_Customers,

        SUM(Customer_Sales) AS Total_Customer_Sales,

        AVG(Customer_Sales) AS Average_Customer_Sales,

        AVG(Customer_Orders) AS Average_Orders_Per_Customer,

        AVG(Customer_Profit) AS Average_Customer_Profit

    FROM customer_summary;
    """,
    conn
)

display(customer_kpis.round(2))

,Total_Customers,Total_Customer_Sales,Average_Customer_Sales,Average_Orders_Per_Customer,Average_Customer_Profit
0,773,1565804.32,2025.62,3.88,226.73


## 4. Return Rate

Business Question:

> What percentage of transactions are associated with returns?

Return Rate is calculated using returned transaction count divided
by total transaction count.

In [145]:
# Calculate overall Return Rate

return_kpi = pd.read_sql_query(
    """
    SELECT
        COUNT(*) AS Total_Transactions,

        SUM(
            CASE
                WHEN Return_Flag = 1 THEN 1
                ELSE 0
            END
        ) AS Returned_Transactions,

        CASE
            WHEN COUNT(*) = 0 THEN 0
            ELSE
                SUM(
                    CASE
                        WHEN Return_Flag = 1 THEN 1
                        ELSE 0
                    END
                ) * 100.0
                / COUNT(*)
        END AS Return_Rate_Percent

    FROM superstore;
    """,
    conn
)

display(return_kpi.round(2))

,Total_Transactions,Returned_Transactions,Return_Rate_Percent
0,5901,287,4.86


## 5. Shipping Efficiency KPI

Business Question:

> What is the average shipping duration and how does shipping
> performance vary by Shipping Mode?

The overall operational benchmark is calculated first.

In [146]:
# Calculate overall shipping efficiency

shipping_kpi = pd.read_sql_query(
    """
    SELECT
        COUNT(*) AS Total_Shipments,
        AVG(Shipping_Days) AS Average_Shipping_Days,
        MIN(Shipping_Days) AS Minimum_Shipping_Days,
        MAX(Shipping_Days) AS Maximum_Shipping_Days
    FROM superstore
    WHERE Shipping_Days IS NOT NULL;
    """,
    conn
)

display(shipping_kpi.round(2))

,Total_Shipments,Average_Shipping_Days,Minimum_Shipping_Days,Maximum_Shipping_Days
0,1685,102.96,0.0,214.0


## 6. Profitability Health

Business Question:

> What proportion of transactions are profitable versus
> loss-making?

This provides an operational view of transaction-level
profitability.

In [147]:
# Calculate profitability health indicators

profitability_kpi = pd.read_sql_query(
    """
    SELECT
        COUNT(*) AS Total_Transactions,

        SUM(
            CASE
                WHEN Profit > 0 THEN 1
                ELSE 0
            END
        ) AS Profitable_Transactions,

        SUM(
            CASE
                WHEN Profit < 0 THEN 1
                ELSE 0
            END
        ) AS Loss_Transactions,

        SUM(
            CASE
                WHEN Profit = 0 THEN 1
                ELSE 0
            END
        ) AS Break_Even_Transactions,

        SUM(
            CASE
                WHEN Profit > 0 THEN Profit
                ELSE 0
            END
        ) AS Positive_Profit,

        SUM(
            CASE
                WHEN Profit < 0 THEN Profit
                ELSE 0
            END
        ) AS Total_Loss

    FROM superstore;
    """,
    conn
)

display(profitability_kpi.round(2))

,Total_Transactions,Profitable_Transactions,Loss_Transactions,Break_Even_Transactions,Positive_Profit,Total_Loss
0,5901,4764,1098,39,266971.83,-91709.73


## 7. Category Leader KPIs

Business Question:

> Which Category is the strongest Sales and Profit contributor?

Sales and Profit rankings are calculated independently because
the category with the highest revenue may not be the category
with the highest profitability.

In [148]:
# Rank Categories by Sales and Profit

category_leaders = pd.read_sql_query(
    """
    WITH category_summary AS (
        SELECT
            Category,
            SUM(Sales) AS Total_Sales,
            SUM(Profit) AS Total_Profit
        FROM superstore
        GROUP BY Category
    ),

    ranked_categories AS (
        SELECT
            Category,
            Total_Sales,
            Total_Profit,

            DENSE_RANK() OVER (
                ORDER BY Total_Sales DESC
            ) AS Sales_Rank,

            DENSE_RANK() OVER (
                ORDER BY Total_Profit DESC
            ) AS Profit_Rank

        FROM category_summary
    )

    SELECT *
    FROM ranked_categories
    ORDER BY Sales_Rank;
    """,
    conn
)

display(category_leaders.round(2))

,Category,Total_Sales,Total_Profit,Sales_Rank,Profit_Rank
0,Office Supplies,643707.69,74797.25,1,2
1,Technology,470587.99,90458.25,2,1
2,Furniture,451508.65,10006.61,3,3


## 8. Regional Executive Performance

Business Question:

> Which Regions generate strong Sales, Profit, and customer
> coverage?

The query combines financial and customer metrics at the regional
level.

In [149]:
# Calculate regional executive KPIs

regional_kpis = pd.read_sql_query(
    """
    SELECT
        Region,

        SUM(Sales) AS Total_Sales,
        SUM(Profit) AS Total_Profit,
        COUNT(DISTINCT Order_ID) AS Total_Orders,
        COUNT(DISTINCT Customer_ID) AS Total_Customers,

        CASE
            WHEN SUM(Sales) = 0 THEN 0
            ELSE
                SUM(Profit) * 100.0 / SUM(Sales)
        END AS Profit_Margin_Percent,

        CASE
            WHEN COUNT(DISTINCT Order_ID) = 0 THEN 0
            ELSE
                SUM(Sales) * 1.0
                / COUNT(DISTINCT Order_ID)
        END AS Average_Order_Value

    FROM superstore

    GROUP BY Region

    ORDER BY Total_Sales DESC;
    """,
    conn
)

display(regional_kpis.round(2))
display(regional_kpis.describe().T)

,Region,Total_Sales,Total_Profit,Total_Orders,Total_Customers,Profit_Margin_Percent,Average_Order_Value
0,West,522441.05,67859.96,961,551,12.99,543.64
1,East,450234.67,53400.42,844,523,11.86,533.45
2,Central,341007.52,27450.01,711,471,8.05,479.62
3,South,252121.08,26551.72,487,357,10.53,517.70


,count,mean,std,min,25%,50%,75%,max
Total_Sales,4.0,391451.080800,119123.582455,252121.081000,318785.913400,395621.095100,468286.262500,522441.052000
Total_Profit,4.0,43815.526475,20296.751213,26551.716300,27225.434400,40425.215700,57015.307775,67859.958200
Total_Orders,4.0,750.750000,203.342691,487.000000,655.000000,777.500000,873.250000,961.000000
Total_Customers,4.0,475.500000,85.671855,357.000000,442.500000,497.000000,530.000000,551.000000
Profit_Margin_Percent,4.0,10.857652,2.124443,8.049678,9.910921,11.195956,12.142686,12.989017
Average_Order_Value,4.0,518.603932,28.096736,479.616771,508.181012,525.577910,536.000830,543.643134


## 9. Latest Year Growth KPI

Business Question:

> How is the most recent available year performing compared with
> the previous year?

The latest two available years are compared using LAG().

In [150]:
# Calculate latest Year-over-Year business growth

latest_year_growth = pd.read_sql_query(
    """
    WITH yearly_summary AS (
        SELECT
            Order_Year,
            SUM(Sales) AS Total_Sales,
            SUM(Profit) AS Total_Profit
        FROM superstore
        GROUP BY Order_Year
    ),

    yearly_comparison AS (
        SELECT
            Order_Year,
            Total_Sales,
            Total_Profit,

            LAG(Total_Sales) OVER (
                ORDER BY Order_Year
            ) AS Previous_Year_Sales,

            LAG(Total_Profit) OVER (
                ORDER BY Order_Year
            ) AS Previous_Year_Profit

        FROM yearly_summary
    )

    SELECT
        Order_Year,
        Total_Sales,
        Total_Profit,
        Previous_Year_Sales,
        Previous_Year_Profit,

        CASE
            WHEN Previous_Year_Sales IS NULL
              OR Previous_Year_Sales = 0
                THEN NULL
            ELSE
                (Total_Sales - Previous_Year_Sales)
                * 100.0
                / Previous_Year_Sales
        END AS Sales_Growth_Percent,

        CASE
            WHEN Previous_Year_Profit IS NULL
              OR Previous_Year_Profit = 0
                THEN NULL
            ELSE
                (Total_Profit - Previous_Year_Profit)
                * 100.0
                / ABS(Previous_Year_Profit)
        END AS Profit_Growth_Percent

    FROM yearly_comparison

    ORDER BY Order_Year DESC
    LIMIT 1;
    """,
    conn
)

display(latest_year_growth.round(2))

,Order_Year,Total_Sales,Total_Profit,Previous_Year_Sales,Previous_Year_Profit,Sales_Growth_Percent,Profit_Growth_Percent
0,2020.0,375388.28,25571.22,245863.79,44553.36,52.68,-42.61


## 10. Executive KPI Scorecard

Business Question:

> Can the major business KPIs be combined into one executive
> scorecard?

This query combines financial, customer, operational, and
transaction metrics into a single result.

This output can later be connected directly to Plotly or a
dashboard layer.

In [151]:
# Create a consolidated executive KPI scorecard

executive_scorecard = pd.read_sql_query(
    """
    SELECT
        COUNT(*) AS Total_Transactions,

        COUNT(DISTINCT Order_ID) AS Total_Orders,

        COUNT(DISTINCT Customer_ID) AS Total_Customers,

        COUNT(DISTINCT Product_ID) AS Total_Products,

        SUM(Sales) AS Total_Sales,

        SUM(Profit) AS Total_Profit,

        SUM(Quantity) AS Total_Quantity,

        CASE
            WHEN SUM(Sales) = 0 THEN 0
            ELSE
                SUM(Profit) * 100.0 / SUM(Sales)
        END AS Profit_Margin_Percent,

        CASE
            WHEN COUNT(DISTINCT Order_ID) = 0 THEN 0
            ELSE
                SUM(Sales) * 1.0
                / COUNT(DISTINCT Order_ID)
        END AS Average_Order_Value,

        CASE
            WHEN COUNT(DISTINCT Customer_ID) = 0 THEN 0
            ELSE
                SUM(Sales) * 1.0
                / COUNT(DISTINCT Customer_ID)
        END AS Sales_Per_Customer,

        AVG(Shipping_Days) AS Average_Shipping_Days,

        SUM(
            CASE
                WHEN Return_Flag = 1 THEN 1
                ELSE 0
            END
        ) * 100.0 / COUNT(*) AS Return_Rate_Percent,

        SUM(
            CASE
                WHEN Profit > 0 THEN 1
                ELSE 0
            END
        ) * 100.0 / COUNT(*) AS Profitable_Transaction_Percent,

        SUM(
            CASE
                WHEN Profit < 0 THEN 1
                ELSE 0
            END
        ) * 100.0 / COUNT(*) AS Loss_Transaction_Percent

    FROM superstore;
    """,
    conn
)

display(executive_scorecard.round(2))

,Total_Transactions,Total_Orders,Total_Customers,Total_Products,Total_Sales,Total_Profit,Total_Quantity,Profit_Margin_Percent,Average_Order_Value,Sales_Per_Customer,Average_Shipping_Days,Return_Rate_Percent,Profitable_Transaction_Percent,Loss_Transaction_Percent
0,5901,3003,773,1755,1565804.32,175262.11,22317,11.19,521.41,2025.62,102.96,4.86,80.73,18.61


# 🏆 SQL Business Analytics — Conclusion

The SQL Business Analytics phase successfully transformed the
SuperStore transaction dataset into a structured and
decision-ready business intelligence layer.

Starting from transaction-level data, the analysis progressively
moved from basic SQL queries to advanced analytical techniques.

---

## 📊 What Was Achieved

The analysis covered the complete business ecosystem:

- Sales Performance
- Profitability Analysis
- Customer Analysis
- Product Analysis
- Category & Sub-Category Analysis
- Customer Segment Analysis
- Regional Analysis
- State & City Analysis
- Shipping Operations
- Payment Behavior
- Returns Analysis
- Yearly Performance
- Quarterly Performance
- Monthly Performance
- Growth Analysis
- Contribution Analysis
- Ranking Analysis
- Multi-Dimensional Analysis
- Executive KPI Analysis

---

## 🧠 SQL Techniques Applied

The project progressed from fundamental SQL operations to
advanced business analytics.

```text
SELECT
WHERE
DISTINCT
ORDER BY
LIMIT
GROUP BY
HAVING
SUM()
AVG()
COUNT()
COUNT(DISTINCT)
MIN()
MAX()
CASE WHEN
Subqueries
CTEs
Window Functions
ROW_NUMBER()
RANK()
DENSE_RANK()
LAG()
LEAD()
PERCENT_RANK()
NTILE()
SUM() OVER()
PARTITION BY
Conditional Aggregation
Cumulative Analysis
Pareto Analysis
Time Intelligence
Multi-Dimensional Aggregation

<div align="center">
📊 SuperStore Sales & Profitability Analysis

From Raw Data → Business Intelligence → Actionable Insights

S Mohammed Kaif

Data Science | Machine Learning | Data Analytics

</div> 